# Phase 1A VLM 6-DOF Spatial Benchmark

This notebook implements the first reproducible experiment stage described in the research proposal.

It evaluates:

- D1: six RGB-D representations (R1-R6)
- D2: 5x5 raw-depth position correction with the recorded camera intrinsics
- D3: four output interfaces
- full 3D closing and approach directions, with geometric validity checks

Important boundaries:

- The default mode is offline and performs zero API calls.
- Online calls use the existing REST path with [`gemini-robotics-er-1.6-preview`](https://ai.google.dev/gemini-api/docs/models/gemini-robotics-er-1.6-preview); no Google SDK is added.
- The existing align notebooks are not used as experimental evidence and are not modified.
- Sensor-consistency metrics are not grasp ground truth.
- Position and orientation accuracy metrics remain null until reference_grasps.jsonl is supplied.
- Generated artifacts are written under output_vg/vlm_6dof_phase1a, which is git-ignored.

The expected first-run HTTP caps are 10 calls for smoke, 40 for pilot, and 364 for full. Retries also consume this hard cap.


In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import math
import os
import random
import statistics
import sys
import tempfile
import time
import urllib.error
import urllib.request
from dataclasses import asdict, dataclass, field, replace
from datetime import datetime, timezone
from io import BytesIO, StringIO
from pathlib import Path
from typing import Any, Callable, Sequence

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from scipy.stats import wilcoxon

SEED = 20260713
random.seed(SEED)
np.random.seed(SEED)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "vg_pipeline").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the vla-grasp-server repository root")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# sys.executable can be a symlink to a base interpreter. sys.prefix is reliable.
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), (
    f"Select the vla-grasp-server kernel. sys.prefix={sys.prefix!r}"
)

from vg_pipeline.align import deproject_pixel
from vg_pipeline.io import load_observation_npy, new_run_id
from vg_pipeline.prompting import build_grounding_prompt
from vg_pipeline.roi import parse_vlm_result

MODEL_NAME = "gemini-robotics-er-1.6-preview"
API_VERSION = "v1beta"
DEPTH_MAX_M = 2.5
NUM_CANDIDATES = 3
CACHE_SCHEMA_VERSION = "phase1a-cache-v1"
EXPERIMENT_SCHEMA_VERSION = "phase1a-experiment-v1"
REPRESENTATION_VERSION = "phase1a-representations-v1"
PROMPT_VERSION = "phase1a-prompt-v1"
ANCHOR_PROMPT_VERSION = "phase1a-anchor-v1"
REPRESENTATION_IDS = (
    "r1_calibrated_jet",
    "r2_linear_gray",
    "r3_local_depth_text",
    "r4_pointcloud_views",
    "r5_coordinate_grid",
    "r6_surface_normals",
)
INTERFACE_IDS = ("direct_xyz", "pixel_depth", "depth_band", "multi_pixel")

RUN_ONLINE = os.environ.get("PHASE1A_RUN_ONLINE", "0") == "1"
PROFILE = os.environ.get("PHASE1A_PROFILE", "smoke")
MAX_NEW_CALLS = int(os.environ.get("PHASE1A_MAX_NEW_CALLS", "0"))
D1_FOR_D3 = os.environ.get("PHASE1A_D1_FOR_D3", "auto")
RUN_ID_OVERRIDE = os.environ.get("PHASE1A_RUN_ID")
RUN_LEGACY_DIAGNOSTIC = os.environ.get("PHASE1A_LEGACY_DIAGNOSTIC", "0") == "1"
RENDER_3D = os.environ.get("PHASE1A_RENDER_3D", "0") == "1"

OUTPUT_ROOT = ROOT / "output_vg" / "vlm_6dof_phase1a"
CACHE_ROOT = OUTPUT_ROOT / "cache"
REFERENCE_GRASPS_PATH = ROOT / "reference_grasps.jsonl"

@dataclass(frozen=True)
class SceneSpec:
    scene_id: str
    task_id: str
    task_spec: str
    category: str
    clutter: bool
    task_override: str | None = None
    anchor_override: dict[str, Any] | None = None

    @property
    def effective_task_spec(self) -> str:
        return self.task_override or self.task_spec

def manifest_record(spec: SceneSpec) -> dict[str, Any]:
    record = asdict(spec)
    record.update(
        {
            "capture": spec.scene_id,
            "target_text": spec.effective_task_spec,
            "object_category": spec.category,
        }
    )
    return record

MANIFEST: tuple[SceneSpec, ...] = (
    SceneSpec("20260417_115700", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
    SceneSpec("20260417_115805", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
    SceneSpec("20260417_115822", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
    SceneSpec("20260417_115838", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
    SceneSpec("20260417_115854", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
    SceneSpec("20260417_115919", "mustard", "grasp the yellow mustard bottle", "mustard_bottle", False),
    SceneSpec("20260417_120002", "cube", "grasp the Rubik's cube", "rubiks_cube", True),
    SceneSpec("20260417_120019", "mustard", "grasp the yellow mustard bottle", "mustard_bottle", True),
    SceneSpec("20260417_120040", "coffee", "grasp the blue Maxwell House coffee can", "coffee_can", False),
    SceneSpec("20260417_120131", "coffee", "grasp the blue Maxwell House coffee can", "coffee_can", True),
    SceneSpec("20260417_120153", "thermos", "grasp the white thermos", "thermos", False),
    SceneSpec("20260417_120218", "thermos", "grasp the white thermos", "thermos", False),
    SceneSpec("20260417_120244", "cube", "grasp the Rubik's cube", "rubiks_cube", False),
)

PROFILE_CONFIGS = {
    "smoke": {
        "scene_ids": ("20260417_115700",),
        "repeats": 1,
        "expected_http_cap": 10,
    },
    "pilot": {
        "scene_ids": (
            "20260417_115700",
            "20260417_115919",
            "20260417_120040",
            "20260417_120218",
        ),
        "repeats": 1,
        "expected_http_cap": 40,
    },
    "full": {
        "scene_ids": tuple(spec.scene_id for spec in MANIFEST),
        "repeats": 3,
        "expected_http_cap": 364,
    },
}
assert PROFILE in PROFILE_CONFIGS, f"Unknown profile: {PROFILE}"
assert len(MANIFEST) == 13
assert len({(s.scene_id, s.task_id) for s in MANIFEST}) == 13

@dataclass
class SceneData:
    spec: SceneSpec
    capture_dir: Path
    rgb_bgr: np.ndarray
    rgb: np.ndarray
    depth: np.ndarray
    K: np.ndarray
    color_preview: Image.Image
    legacy_depth_preview: Image.Image | None

def validate_K(K: np.ndarray, width: int, height: int) -> None:
    K = np.asarray(K, dtype=np.float64)
    assert K.shape == (3, 3)
    assert np.all(np.isfinite(K))
    assert K[0, 0] > 0 and K[1, 1] > 0
    assert np.allclose(K[2], [0, 0, 1], atol=1e-8)
    assert 0 <= K[0, 2] < width and 0 <= K[1, 2] < height

def load_scene(spec: SceneSpec) -> SceneData:
    capture_dir = ROOT / "rgbd_data" / "captures" / spec.scene_id
    frame = load_observation_npy(capture_dir / "camera_data.npy")
    rgb_bgr = np.asarray(frame["rgb"], dtype=np.uint8)
    rgb = np.ascontiguousarray(rgb_bgr[..., ::-1])
    depth = np.asarray(frame["depth"], dtype=np.float32)
    K = np.asarray(frame["K"], dtype=np.float64)
    color_preview = Image.open(capture_dir / "color_preview.jpg").convert("RGB")
    legacy_path = capture_dir / "depth_preview.jpg"
    legacy_depth_preview = Image.open(legacy_path).convert("RGB") if legacy_path.exists() else None
    assert rgb.ndim == 3 and rgb.shape[2] == 3
    assert depth.shape == rgb.shape[:2]
    assert color_preview.size == (rgb.shape[1], rgb.shape[0])
    validate_K(K, rgb.shape[1], rgb.shape[0])
    return SceneData(
        spec=spec,
        capture_dir=capture_dir,
        rgb_bgr=rgb_bgr,
        rgb=rgb,
        depth=depth,
        K=K,
        color_preview=color_preview,
        legacy_depth_preview=legacy_depth_preview,
    )

def audit_scene(data: SceneData) -> dict[str, Any]:
    valid = np.isfinite(data.depth) & (data.depth > 0)
    values = data.depth[valid].astype(np.float64)
    preview = np.asarray(data.color_preview, dtype=np.int16)
    correct_mae = float(np.mean(np.abs(preview - data.rgb.astype(np.int16))))
    wrong_mae = float(np.mean(np.abs(preview - data.rgb_bgr.astype(np.int16))))
    return {
        "scene_id": data.spec.scene_id,
        "task_id": data.spec.task_id,
        "category": data.spec.category,
        "clutter": data.spec.clutter,
        "height": int(data.depth.shape[0]),
        "width": int(data.depth.shape[1]),
        "valid_depth_fraction": float(valid.mean()),
        "overflow_fraction_valid": float(np.mean(values > DEPTH_MAX_M)),
        "depth_median_m": float(np.median(values)),
        "depth_p99_m": float(np.quantile(values, 0.99)),
        "bgr_to_rgb_preview_mae": correct_mae,
        "unflipped_preview_mae": wrong_mae,
    }

AUDIT_ROWS = [audit_scene(load_scene(spec)) for spec in MANIFEST]
assert all(row["height"] == 720 and row["width"] == 1280 for row in AUDIT_ROWS)
assert all(row["bgr_to_rgb_preview_mae"] < 2.0 for row in AUDIT_ROWS)
assert all(
    row["bgr_to_rgb_preview_mae"] < row["unflipped_preview_mae"]
    for row in AUDIT_ROWS
)

print(f"Project root: {ROOT}")
print(f"Python prefix: {sys.prefix}")
print(f"Profile: {PROFILE}; online={RUN_ONLINE}; max_new_calls={MAX_NEW_CALLS}")
print("Scene audit:")
for row in AUDIT_ROWS:
    print(
        f"  {row['scene_id']} {row['task_id']:<8} clutter={str(row['clutter']):<5} "
        f"valid={row['valid_depth_fraction']:.3f} "
        f"overflow={row['overflow_fraction_valid']:.3f} "
        f"median={row['depth_median_m']:.3f}m "
        f"RGB-MAE={row['bgr_to_rgb_preview_mae']:.2f}"
    )


## D1 — six input representations

All methods share the same `direct_xyz` task prompt. R1 is the formal calibrated baseline; the historical `depth_preview.jpg` is optional diagnostics only. R3 consumes one cached RGB-only localization per scene-task and fails structurally when localization is unavailable.

In [ ]:
@dataclass(frozen=True)
class DepthStats:
    pixel_uv: tuple[int, int]
    window_area: int
    valid_count: int
    valid_fraction: float
    median_m: float | None
    mad_m: float | None

def sample_depth_stats(depth: np.ndarray, pixel_uv: Sequence[int], radius: int = 2) -> DepthStats:
    if len(pixel_uv) != 2:
        raise ValueError("pixel_uv must contain [u, v]")
    u, v = int(pixel_uv[0]), int(pixel_uv[1])
    height, width = depth.shape
    if not (0 <= u < width and 0 <= v < height):
        return DepthStats((u, v), 0, 0, 0.0, None, None)
    u0, u1 = max(0, u - radius), min(width, u + radius + 1)
    v0, v1 = max(0, v - radius), min(height, v + radius + 1)
    window = np.asarray(depth[v0:v1, u0:u1], dtype=np.float64)
    valid = np.isfinite(window) & (window > 0)
    values = window[valid]
    area = int(window.size)
    if values.size == 0:
        return DepthStats((u, v), area, 0, 0.0, None, None)
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    return DepthStats(
        (u, v),
        area,
        int(values.size),
        float(values.size / area),
        median,
        mad,
    )

def pil_png_bytes(image: Image.Image) -> bytes:
    buffer = BytesIO()
    image.save(buffer, format="PNG", optimize=False)
    return buffer.getvalue()

def render_calibrated_jet(depth: np.ndarray) -> Image.Image:
    valid = np.isfinite(depth) & (depth > 0)
    normalized = np.clip(np.nan_to_num(depth, nan=0.0) / DEPTH_MAX_M, 0.0, 1.0)
    rgba = matplotlib.colormaps["jet"](normalized, bytes=True)
    rgb = rgba[..., :3]
    rgb[~valid] = 0
    return Image.fromarray(rgb.astype(np.uint8), "RGB")

def render_linear_gray(depth: np.ndarray) -> Image.Image:
    valid = np.isfinite(depth) & (depth > 0)
    values = np.clip(np.nan_to_num(depth, nan=0.0), 0.0, DEPTH_MAX_M)
    gray = np.rint(values / DEPTH_MAX_M * 255.0).astype(np.uint8)
    gray[~valid] = 0
    return Image.fromarray(gray, "L").convert("RGB")

def render_depth_scale(width: int = 512, height: int = 96) -> Image.Image:
    gradient = np.linspace(0.0, 1.0, width, dtype=np.float32)
    strip = np.repeat(gradient[None, :], height - 28, axis=0)
    rgb = matplotlib.colormaps["jet"](strip, bytes=True)[..., :3]
    image = Image.new("RGB", (width, height), "white")
    image.paste(Image.fromarray(rgb.astype(np.uint8), "RGB"), (0, 0))
    draw = ImageDraw.Draw(image)
    for fraction, label in ((0.0, "0.0 m"), (0.5, "1.25 m"), (1.0, "2.5 m")):
        x = int(round(fraction * (width - 1)))
        draw.line([(x, height - 28), (x, height - 21)], fill="black", width=2)
        anchor = "la" if fraction == 0 else ("ra" if fraction == 1 else "ma")
        draw.text((x, height - 18), label, fill="black", anchor=anchor)
    return image

def organized_pointcloud(depth: np.ndarray, K: np.ndarray) -> np.ndarray:
    height, width = depth.shape
    vv, uu = np.indices((height, width), dtype=np.float32)
    z = depth.astype(np.float32)
    x = (uu - float(K[0, 2])) * z / float(K[0, 0])
    y = (vv - float(K[1, 2])) * z / float(K[1, 1])
    return np.stack((x, y, z), axis=-1)

def scene_seed(scene_id: str, representation_id: str) -> int:
    digest = hashlib.sha256(
        f"{SEED}:{scene_id}:{representation_id}".encode("utf-8")
    ).digest()
    return int.from_bytes(digest[:8], "big") & 0xFFFFFFFF

def render_pointcloud_views(
    data: SceneData,
    max_points: int = 30_000,
    canvas_size: tuple[int, int] = (960, 720),
) -> tuple[Image.Image, Image.Image, Image.Image]:
    points = organized_pointcloud(data.depth, data.K)
    valid = (
        np.isfinite(points).all(axis=-1)
        & (data.depth > 0)
        & (data.depth <= DEPTH_MAX_M)
    )
    xyz = points[valid].astype(np.float64)
    colors = data.rgb[valid]
    if len(xyz) == 0:
        raise ValueError("No valid point-cloud points")
    rng = np.random.default_rng(scene_seed(data.spec.scene_id, "r4_pointcloud_views"))
    if len(xyz) > max_points:
        indices = np.sort(rng.choice(len(xyz), max_points, replace=False))
        xyz = xyz[indices]
        colors = colors[indices]

    # Fixed camera-frustum bounds make scenes directly comparable.
    projections = (
        ("Front view: X-Y", xyz[:, 0], xyz[:, 1], (-1.8, 1.8), (0.5, -1.1), "X camera (m)", "Y camera (m, down +)"),
        ("Top view: X-Z", xyz[:, 0], xyz[:, 2], (-1.8, 1.8), (0.0, 2.5), "X camera (m)", "Z camera (m, forward +)"),
        ("Side view: Z-Y", xyz[:, 2], xyz[:, 1], (0.0, 2.5), (0.5, -1.1), "Z camera (m, forward +)", "Y camera (m, down +)"),
    )
    images: list[Image.Image] = []
    for title, horizontal, vertical, xlim, ylim, xlabel, ylabel in projections:
        fig, ax = plt.subplots(figsize=(canvas_size[0] / 120, canvas_size[1] / 120), dpi=120)
        ax.scatter(horizontal, vertical, c=colors / 255.0, s=0.7, linewidths=0, rasterized=True)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=120)
        plt.close(fig)
        buffer.seek(0)
        images.append(Image.open(buffer).convert("RGB").copy())
    return tuple(images)  # type: ignore[return-value]


In [ ]:
def coordinate_grid_points(width: int, height: int) -> list[tuple[int, int]]:
    return [
        (u, v)
        for v in range(50, height, 100)
        for u in range(50, width, 100)
    ]

def render_coordinate_grid(data: SceneData) -> Image.Image:
    image = Image.fromarray(data.rgb, "RGB").copy()
    draw = ImageDraw.Draw(image)
    for u, v in coordinate_grid_points(image.width, image.height):
        z = float(data.depth[v, u])
        z_text = f"{z:.3f}" if np.isfinite(z) and z > 0 else "invalid"
        label = f"({u},{v},{z_text})"
        draw.ellipse((u - 3, v - 3, u + 3, v + 3), fill=(255, 255, 0), outline=(0, 0, 0))
        box = draw.textbbox((0, 0), label, stroke_width=2)
        text_w = box[2] - box[0]
        x = min(max(0, u + 5), image.width - text_w - 2)
        y = min(max(0, v - 14), image.height - 14)
        draw.text((x, y), label, fill=(255, 255, 0), stroke_width=2, stroke_fill=(0, 0, 0))
    return image

def render_normal_map(depth: np.ndarray, K: np.ndarray, radius: int = 2) -> Image.Image:
    points = organized_pointcloud(depth, K).astype(np.float64)
    height, width = depth.shape
    normals = np.zeros((height, width, 3), dtype=np.float64)
    if height <= 2 * radius or width <= 2 * radius:
        return Image.fromarray(np.zeros((height, width, 3), dtype=np.uint8), "RGB")

    center = points[radius:-radius, radius:-radius]
    left = points[radius:-radius, :-2 * radius]
    right = points[radius:-radius, 2 * radius:]
    up = points[:-2 * radius, radius:-radius]
    down = points[2 * radius:, radius:-radius]

    center_z = center[..., 2]
    z_stack = np.stack(
        (left[..., 2], right[..., 2], up[..., 2], down[..., 2]), axis=-1
    )
    finite = (
        np.isfinite(center).all(axis=-1)
        & np.isfinite(left).all(axis=-1)
        & np.isfinite(right).all(axis=-1)
        & np.isfinite(up).all(axis=-1)
        & np.isfinite(down).all(axis=-1)
    )
    positive = (center_z > 0) & np.all(z_stack > 0, axis=-1)
    discontinuity_limit = np.maximum(0.02, 0.02 * center_z)
    continuous = np.all(np.abs(z_stack - center_z[..., None]) <= discontinuity_limit[..., None], axis=-1)

    tangent_u = right - left
    tangent_v = down - up
    raw = np.cross(tangent_u, tangent_v)
    norm = np.linalg.norm(raw, axis=-1)
    valid = finite & positive & continuous & np.isfinite(norm) & (norm > 1e-10)
    unit = np.zeros_like(raw)
    unit[valid] = raw[valid] / norm[valid, None]

    # The sign is chosen consistently toward the camera origin.
    flip = np.sum(unit * center, axis=-1) > 0
    unit[flip] *= -1
    normals[radius:-radius, radius:-radius] = unit

    encoded = np.zeros((height, width, 3), dtype=np.uint8)
    normal_valid = np.linalg.norm(normals, axis=-1) > 0
    encoded[normal_valid] = np.rint(
        np.clip((normals[normal_valid] + 1.0) * 127.5, 0, 255)
    ).astype(np.uint8)
    return Image.fromarray(encoded, "RGB")

@dataclass(frozen=True)
class AnchorSet:
    object_bbox_xyxy: tuple[int, int, int, int]
    object_uv: tuple[int, int]
    grasp_uvs: tuple[tuple[int, int], ...]
    table_uv: tuple[int, int] | None
    source: str

def _yx_to_uv(point_yx: Sequence[int]) -> tuple[int, int]:
    if len(point_yx) != 2:
        raise ValueError("point must contain [y, x]")
    return int(point_yx[1]), int(point_yx[0])

def _validate_uv(uv: Sequence[int], width: int, height: int) -> tuple[int, int]:
    u, v = int(uv[0]), int(uv[1])
    if not (0 <= u < width and 0 <= v < height):
        raise ValueError(f"anchor out of bounds: {(u, v)}")
    return u, v

def _normalized_anchor_numbers(value: Any, length: int, field_name: str) -> tuple[float, ...]:
    if not isinstance(value, list) or len(value) != length:
        raise ValueError(f"{field_name} must contain {length} normalized numbers")
    if not all(
        isinstance(item, (int, float))
        and not isinstance(item, bool)
        and math.isfinite(float(item))
        and 0.0 <= float(item) <= 1000.0
        for item in value
    ):
        raise ValueError(f"{field_name} must stay within normalized [0,1000]")
    return tuple(float(item) for item in value)

def _validate_normalized_anchor_payload(raw_text: str) -> None:
    def reject_constant(value: str) -> None:
        raise ValueError(f"non-standard anchor JSON constant: {value}")

    payload = json.loads(raw_text, parse_constant=reject_constant)
    if not isinstance(payload, dict):
        raise ValueError("anchor response must be one strict JSON object")
    object_box = _normalized_anchor_numbers(payload.get("object_box"), 4, "object_box")
    object_point = _normalized_anchor_numbers(payload.get("object_point"), 2, "object_point")
    y0, x0, y1, x1 = object_box
    if not (y0 < y1 and x0 < x1):
        raise ValueError("normalized object_box must have positive area")
    if not (y0 <= object_point[0] <= y1 and x0 <= object_point[1] <= x1):
        raise ValueError("object_point must lie inside object_box")
    candidates = payload.get("candidates")
    if not isinstance(candidates, list) or not candidates:
        raise ValueError("anchor response must contain grasp candidates")
    for index, candidate in enumerate(candidates):
        if not isinstance(candidate, dict):
            raise ValueError(f"anchor candidate[{index}] must be an object")
        box = _normalized_anchor_numbers(
            candidate.get("grasp_region_box"), 4, f"candidate[{index}].grasp_region_box"
        )
        point = _normalized_anchor_numbers(
            candidate.get("grasp_point"), 2, f"candidate[{index}].grasp_point"
        )
        gy0, gx0, gy1, gx1 = box
        if not (gy0 < gy1 and gx0 < gx1):
            raise ValueError(f"candidate[{index}] grasp box must have positive area")
        if not (gy0 <= point[0] <= gy1 and gx0 <= point[1] <= gx1):
            raise ValueError(f"candidate[{index}] grasp point must lie inside its box")

def derive_anchor_set(data: SceneData, raw_text: str) -> AnchorSet:
    _validate_normalized_anchor_payload(raw_text)
    height, width = data.rgb.shape[:2]
    parsed = parse_vlm_result(
        raw_text,
        canvas_h=height,
        canvas_w=width,
        rgb_h=height,
    )
    bbox_yxyx = parsed.object_box
    target_yx = parsed.object_point
    grasp_yx = [point for point in parsed.grasp_points if point is not None]
    if bbox_yxyx is None or target_yx is None or not grasp_yx:
        raise ValueError("RGB-only localization lacks object box, object point, or grasp point")

    y0, x0, y1, x1 = [int(round(float(x))) for x in bbox_yxyx]
    if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
        raise ValueError(f"invalid target box: {bbox_yxyx}")
    object_uv = _validate_uv(_yx_to_uv(target_yx), width, height)
    grasp_uvs = tuple(
        _validate_uv(_yx_to_uv(point), width, height)
        for point in grasp_yx[:3]
    )

    object_depth = sample_depth_stats(data.depth, object_uv).median_m
    table_uv: tuple[int, int] | None = None
    if object_depth is not None:
        start_v = min(height - 1, y1 + 10)
        stop_v = min(height, y1 + 181)
        center_u = int(round((x0 + x1) / 2))
        search_us = range(max(0, x0), min(width, x1 + 1), max(1, (x1 - x0) // 8))
        candidates: list[tuple[int, int, float]] = []
        for v in range(start_v, stop_v, 10):
            for u in search_us:
                stats = sample_depth_stats(data.depth, (u, v))
                if stats.median_m is not None and stats.median_m >= object_depth + 0.02:
                    candidates.append((u, v, stats.median_m))
            if candidates:
                break
        if candidates:
            u, v, _ = min(candidates, key=lambda row: abs(row[0] - center_u))
            table_uv = (u, v)

    return AnchorSet(
        object_bbox_xyxy=(x0, y0, x1, y1),
        object_uv=object_uv,
        grasp_uvs=grasp_uvs,
        table_uv=table_uv,
        source="rgb_only_vlm",
    )

def anchor_set_from_override(data: SceneData, override: dict[str, Any]) -> AnchorSet:
    width, height = data.rgb.shape[1], data.rgb.shape[0]
    bbox = tuple(int(x) for x in override["object_bbox_xyxy"])
    if len(bbox) != 4:
        raise ValueError("object_bbox_xyxy must have four values")
    x0, y0, x1, y1 = bbox
    if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
        raise ValueError("override object box is invalid")
    object_uv = _validate_uv(override["object_uv"], width, height)
    grasp_uvs = tuple(
        _validate_uv(point, width, height)
        for point in override["grasp_uvs"]
    )
    if not grasp_uvs:
        raise ValueError("override must provide at least one grasp anchor")
    table_value = override.get("table_uv")
    table_uv = None if table_value is None else _validate_uv(table_value, width, height)
    return AnchorSet((x0, y0, x1, y1), object_uv, grasp_uvs, table_uv, "manifest_override")

def render_local_depth_annotations(data: SceneData, anchors: AnchorSet) -> Image.Image:
    image = Image.fromarray(data.rgb, "RGB").copy()
    draw = ImageDraw.Draw(image)
    x0, y0, x1, y1 = anchors.object_bbox_xyxy
    draw.rectangle((x0, y0, x1, y1), outline=(255, 255, 0), width=4)
    labeled = [
        ("object", anchors.object_uv, (255, 255, 0)),
        *[
            (f"grasp-{index + 1}", uv, (0, 255, 0))
            for index, uv in enumerate(anchors.grasp_uvs)
        ],
    ]
    if anchors.table_uv is not None:
        labeled.append(("table", anchors.table_uv, (0, 255, 255)))
    for label, (u, v), color in labeled:
        stats = sample_depth_stats(data.depth, (u, v))
        depth_text = (
            f"{stats.median_m * 1000:.0f} mm"
            if stats.median_m is not None
            else "invalid"
        )
        draw.ellipse((u - 7, v - 7, u + 7, v + 7), fill=color, outline=(0, 0, 0), width=2)
        text = f"{label}: {depth_text}"
        draw.text((u + 10, max(0, v - 10)), text, fill=color, stroke_width=3, stroke_fill=(0, 0, 0))
    return image


In [ ]:
@dataclass
class RepresentationBundle:
    representation_id: str
    images: list[tuple[str, Image.Image]]
    metadata: dict[str, Any] = field(default_factory=dict)

class RepresentationUnavailable(RuntimeError):
    pass

def build_representation(
    data: SceneData,
    representation_id: str,
    anchors: AnchorSet | None = None,
) -> RepresentationBundle:
    rgb_image = Image.fromarray(data.rgb, "RGB")
    if representation_id == "r1_calibrated_jet":
        return RepresentationBundle(
            representation_id,
            [
                ("RGB observation", rgb_image),
                ("Calibrated metric depth: 0.0–2.5 m JET; invalid black", render_calibrated_jet(data.depth)),
                ("Independent metric depth scale", render_depth_scale()),
            ],
        )
    if representation_id == "r2_linear_gray":
        return RepresentationBundle(
            representation_id,
            [
                ("RGB observation", rgb_image),
                ("Linear metric depth: 0.0 m black, 2.5 m white; invalid black", render_linear_gray(data.depth)),
            ],
        )
    if representation_id == "r3_local_depth_text":
        if anchors is None:
            raise RepresentationUnavailable("R3 requires successful cached RGB-only localization")
        return RepresentationBundle(
            representation_id,
            [("RGB with 5x5 median metric-depth annotations", render_local_depth_annotations(data, anchors))],
            {"anchor_source": anchors.source, "anchors": asdict(anchors)},
        )
    if representation_id == "r4_pointcloud_views":
        front, top, side = render_pointcloud_views(data)
        return RepresentationBundle(
            representation_id,
            [
                ("RGB observation", rgb_image),
                ("Front metric point-cloud projection", front),
                ("Top metric point-cloud projection", top),
                ("Side metric point-cloud projection", side),
            ],
        )
    if representation_id == "r5_coordinate_grid":
        return RepresentationBundle(
            representation_id,
            [("RGB with (u,v,Z_m) grid sampled at (50+100n,50+100m)", render_coordinate_grid(data))],
        )
    if representation_id == "r6_surface_normals":
        return RepresentationBundle(
            representation_id,
            [
                ("RGB observation", rgb_image),
                (
                    "Camera-frame surface normals encoded as (n+1)/2; invalid black",
                    render_normal_map(data.depth, data.K),
                ),
            ],
        )
    raise KeyError(representation_id)

def legacy_diagnostic(data: SceneData) -> RepresentationBundle | None:
    if not RUN_LEGACY_DIAGNOSTIC or data.legacy_depth_preview is None:
        return None
    return RepresentationBundle(
        "legacy_depth_preview_diagnostic_only",
        [
            ("RGB observation", Image.fromarray(data.rgb, "RGB")),
            (
                "Historical depth_preview.jpg (diagnostic only; excluded from rank/budget)",
                data.legacy_depth_preview,
            ),
        ],
        {"formal_ranking": False, "default_budget": False},
    )


## Prompts, content-hash cache, and bounded REST runner

The runner is cache-first and offline by default. Every actual HTTP transport attempt—including retries—consumes the hard budget. Cache identity includes the exact prompt, ordered PNG bytes, model/configuration, semantic interface, and repeat index; it deliberately excludes orchestration stage so D3 `direct_xyz` reuses the corresponding D1 result.

In [ ]:
GENERATION_CONFIG = {
    "temperature": 0,
}

COMMON_CANDIDATE_FIELDS = {
    "rank",
    "pixel_uv",
    "closing_direction_3d",
    "approach_direction_3d",
    "estimated_object_width_along_close_m",
    "confidence",
    "reasoning_summary",
}
INTERFACE_EXTRA_FIELDS = {
    "direct_xyz": {"position_3d"},
    "pixel_depth": {"estimated_depth_m"},
    "depth_band": {"depth_band_index"},
    "multi_pixel": set(),
}

def depth_band_description() -> str:
    rows = [
        f"{index}: [{index * 0.25:.2f}, {(index + 1) * 0.25:.2f}) m"
        for index in range(10)
    ]
    rows.append("10: overflow, depth >= 2.50 m")
    return "; ".join(rows)

def interface_schema_text(interface_id: str, K: np.ndarray) -> str:
    pixels = ([620, 350], [640, 380], [660, 410])
    candidates: list[dict[str, Any]] = []
    for rank, pixel in zip((1, 2, 3), pixels, strict=True):
        candidate: dict[str, Any] = {
            "rank": rank,
            "pixel_uv": pixel,
            "closing_direction_3d": [1.0, 0.0, 0.0],
            "approach_direction_3d": [0.0, 0.0, 1.0],
            "estimated_object_width_along_close_m": 0.05,
            "confidence": 0.8,
            "reasoning_summary": f"brief candidate {rank} rationale",
        }
        if interface_id == "direct_xyz":
            example_position = deproject_pixel(pixel[0], pixel[1], 0.8, K)
            candidate["position_3d"] = [round(float(value), 8) for value in example_position]
        elif interface_id == "pixel_depth":
            candidate["estimated_depth_m"] = 0.8
        elif interface_id == "depth_band":
            candidate["depth_band_index"] = 3
        elif interface_id != "multi_pixel":
            raise KeyError(interface_id)
        candidates.append(candidate)
    return json.dumps({"candidates": candidates}, separators=(",", ":"), allow_nan=False)

def build_task_prompt(data: SceneData, interface_id: str) -> str:
    width, height = data.rgb.shape[1], data.rgb.shape[0]
    interface_instruction = {
        "direct_xyz": (
            "For each candidate estimate both pixel_uv and camera-frame position_3d [x,y,z] in meters."
        ),
        "pixel_depth": (
            "For each candidate estimate pixel_uv and estimated_depth_m. Do not output position_3d; "
            "the server will deproject with the supplied intrinsics."
        ),
        "depth_band": (
            "For each candidate estimate pixel_uv and depth_band_index only. Do not output a metric "
            f"depth or position. Band definitions: {depth_band_description()}."
        ),
        "multi_pixel": (
            "Return three ranked pixel candidates. Do not output position_3d, estimated_depth_m, or "
            "depth_band_index; the server will choose using measured 5x5 depth support."
        ),
    }[interface_id]
    return (
        f"PROMPT_VERSION={PROMPT_VERSION}\n"
        "You are planning a parallel-jaw 6-DOF grasp from calibrated camera observations.\n"
        f"Task: {data.spec.effective_task_spec}.\n"
        f"The RGB/depth image coordinate system is width={width}, height={height}; pixel_uv is [u,v] "
        "with u rightward and v downward, in integer pixel coordinates.\n"
        "Camera coordinates are +x right, +y down, +z forward, in meters.\n"
        f"Camera intrinsics K={np.asarray(data.K).round(8).tolist()}.\n"
        "closing_direction_3d is the direction along which the jaws close. "
        "approach_direction_3d points in the gripper approach direction. Predict both full 3D vectors; "
        "do not replace either vector with a default. They should be nonzero and mutually orthogonal.\n"
        "Choose stable, visible main-body contact regions, avoid handles/spouts/edges/thin parts/table "
        "collisions, and keep estimated width within the supported 0.02–0.08 m jaw range.\n"
        f"{interface_instruction}\n"
        f"Return exactly {NUM_CANDIDATES} candidates with unique ranks 1,2,3. Confidence is a JSON "
        "number in [0,1]. All numeric fields must be finite JSON numbers, never strings, NaN, or Infinity.\n"
        "Return only one strict JSON object, without Markdown fences, commentary, or additional keys.\n"
        "The following example defines key structure only; replace every coordinate and value for this scene.\n"
        f"Exact three-candidate schema example: {interface_schema_text(interface_id, data.K)}"
    )

def build_anchor_prompt(data: SceneData) -> str:
    height, width = data.rgb.shape[:2]
    return (
        f"ANCHOR_PROMPT_VERSION={ANCHOR_PROMPT_VERSION}\n"
        + build_grounding_prompt(
            data.spec.effective_task_spec,
            width,
            height,
            num_candidates=3,
        )
    )

@dataclass(frozen=True)
class EncodedImage:
    descriptor: str
    mime_type: str
    png_bytes: bytes

def encode_bundle(bundle: RepresentationBundle) -> list[EncodedImage]:
    return [
        EncodedImage(descriptor, "image/png", pil_png_bytes(image))
        for descriptor, image in bundle.images
    ]

def encode_rgb_only(data: SceneData) -> list[EncodedImage]:
    return [
        EncodedImage(
            "RGB-only observation for target prelocalization",
            "image/png",
            pil_png_bytes(Image.fromarray(data.rgb, "RGB")),
        )
    ]

def request_contents(prompt: str, images: Sequence[EncodedImage]) -> list[dict[str, Any]]:
    parts: list[dict[str, Any]] = [{"text": prompt}]
    for item in images:
        parts.append({"text": f"Image descriptor: {item.descriptor}"})
        parts.append(
            {
                "inlineData": {
                    "mimeType": item.mime_type,
                    "data": base64.b64encode(item.png_bytes).decode("ascii"),
                }
            }
        )
    return [{"role": "user", "parts": parts}]

def canonical_json_bytes(value: Any) -> bytes:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

def cache_descriptor(
    *,
    semantic_interface: str,
    prompt: str,
    images: Sequence[EncodedImage],
    repeat_index: int,
) -> dict[str, Any]:
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "api_version": API_VERSION,
        "model": MODEL_NAME,
        "semantic_interface": semantic_interface,
        "generation_config": GENERATION_CONFIG,
        "prompt": prompt,
        "repeat_index": int(repeat_index),
        "images": [
            {
                "descriptor": image.descriptor,
                "mime_type": image.mime_type,
                "sha256": hashlib.sha256(image.png_bytes).hexdigest(),
            }
            for image in images
        ],
    }

def cache_key(**kwargs: Any) -> str:
    return hashlib.sha256(canonical_json_bytes(cache_descriptor(**kwargs))).hexdigest()


In [ ]:
class BudgetExceeded(RuntimeError):
    pass

@dataclass
class CallBudget:
    max_new_http_calls: int
    new_http_calls: int = 0
    ledger_path: Path | None = None

    def persist(self) -> None:
        if self.ledger_path is None:
            return
        self.ledger_path.parent.mkdir(parents=True, exist_ok=True)
        payload = json.dumps(
            {
                "hard_http_call_limit": self.max_new_http_calls,
                "used_http_transport_attempts": self.new_http_calls,
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            },
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        with tempfile.NamedTemporaryFile(
            "w", encoding="utf-8", dir=self.ledger_path.parent, delete=False
        ) as handle:
            handle.write(payload)
            temporary = Path(handle.name)
        temporary.replace(self.ledger_path)

    def reserve_transport_attempt(self) -> None:
        if self.new_http_calls >= self.max_new_http_calls:
            raise BudgetExceeded(
                f"Hard HTTP-call budget exhausted: {self.new_http_calls}/{self.max_new_http_calls}"
            )
        self.new_http_calls += 1
        # Persist before transport; a crash can consume budget conservatively, never exceed it.
        self.persist()

@dataclass
class CallResult:
    status: str
    cache_key: str
    response_text: str | None
    raw_body: str | None
    from_cache: bool
    original_network_latency_s: float | None
    current_lookup_latency_s: float
    transport_attempts: int
    error: str | None = None

def reject_nonstandard_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON constant: {value}")

def extract_gemini_text(raw_body: str) -> str:
    body = json.loads(raw_body, parse_constant=reject_nonstandard_constant)
    parts = body["candidates"][0]["content"]["parts"]
    if not isinstance(parts, list):
        raise TypeError("Gemini content.parts is not a list")
    text_parts = [part["text"] for part in parts if isinstance(part, dict) and "text" in part]
    result = "\n".join(text_parts).strip()
    if not result:
        raise ValueError("Gemini response contains no text")
    return result

def redact_secret(text: str, secret: str | None) -> str:
    return text if not secret else text.replace(secret, "<redacted>")

def cache_path_for(key: str) -> Path:
    return CACHE_ROOT / key[:2] / f"{key}.json"

def read_cache_record(path: Path) -> dict[str, Any]:
    record = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_nonstandard_constant)
    if not isinstance(record, dict) or record.get("cache_schema_version") != CACHE_SCHEMA_VERSION:
        raise ValueError("cache schema mismatch")
    if not isinstance(record.get("raw_body"), str):
        raise ValueError("cache is missing raw_body")
    return record

def write_cache_record(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )
    with tempfile.NamedTemporaryFile(
        "w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{path.name}.",
        suffix=".tmp",
        delete=False,
    ) as handle:
        handle.write(payload)
        temp_path = Path(handle.name)
    temp_path.replace(path)

Transport = Callable[..., Any]  # called as transport(request, timeout=...)

def run_cached_gemini_request(
    *,
    semantic_interface: str,
    prompt: str,
    images: Sequence[EncodedImage],
    repeat_index: int,
    online: bool,
    budget: CallBudget,
    timeout_s: float = 60.0,
    transport: Transport = urllib.request.urlopen,
    sleeper: Callable[[float], None] = time.sleep,
) -> CallResult:
    started = time.perf_counter()
    descriptor = cache_descriptor(
        semantic_interface=semantic_interface,
        prompt=prompt,
        images=images,
        repeat_index=repeat_index,
    )
    key = hashlib.sha256(canonical_json_bytes(descriptor)).hexdigest()
    cache_path = cache_path_for(key)
    if cache_path.exists():
        try:
            cached = read_cache_record(cache_path)
            raw_body = cached["raw_body"]
            try:
                response_text = extract_gemini_text(raw_body)
                error = None
                status = "ok"
            except Exception as exc:
                response_text = None
                error = f"cached_response_parse_error: {exc}"
                status = "response_error"
            return CallResult(
                status=status,
                cache_key=key,
                response_text=response_text,
                raw_body=raw_body,
                from_cache=True,
                original_network_latency_s=cached.get("network_latency_s"),
                current_lookup_latency_s=time.perf_counter() - started,
                transport_attempts=0,
                error=error,
            )
        except Exception as exc:
            if not online:
                return CallResult(
                    "cache_corrupt",
                    key,
                    None,
                    None,
                    True,
                    None,
                    time.perf_counter() - started,
                    0,
                    f"cache_corrupt: {exc}",
                )

    if not online:
        return CallResult(
            "offline_miss",
            key,
            None,
            None,
            False,
            None,
            time.perf_counter() - started,
            0,
            "offline cache miss; network disabled",
        )

    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        return CallResult(
            "configuration_error",
            key,
            None,
            None,
            False,
            None,
            time.perf_counter() - started,
            0,
            "GEMINI_API_KEY or GOOGLE_API_KEY is required for explicit online mode",
        )

    payload = {
        "contents": request_contents(prompt, images),
        "generationConfig": GENERATION_CONFIG,
    }
    endpoint = (
        f"https://generativelanguage.googleapis.com/{API_VERSION}/models/"
        f"{MODEL_NAME}:generateContent"
    )
    request = urllib.request.Request(
        endpoint,
        data=canonical_json_bytes(payload),
        headers={
            "Content-Type": "application/json",
            "x-goog-api-key": api_key,
        },
        method="POST",
    )

    attempts = 0
    last_error: str | None = None
    network_started = time.perf_counter()
    for retry_index in range(4):
        try:
            budget.reserve_transport_attempt()
        except BudgetExceeded as exc:
            return CallResult(
                "budget_exhausted",
                key,
                None,
                None,
                False,
                None,
                time.perf_counter() - started,
                attempts,
                str(exc),
            )
        attempts += 1
        try:
            with transport(request, timeout=timeout_s) as response:
                raw_body = response.read().decode("utf-8")
            latency = time.perf_counter() - network_started
            record = {
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "cache_key": key,
                "request_descriptor": descriptor,
                "raw_body": raw_body,
                "network_latency_s": latency,
                "transport_attempts": attempts,
                "cached_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            # Any 2xx body is cached, including a safety block or malformed model output.
            try:
                write_cache_record(cache_path, record)
            except Exception as exc:
                try:
                    response_text = extract_gemini_text(raw_body)
                except Exception:
                    response_text = None
                return CallResult(
                    "cache_write_error", key, response_text, raw_body, False, latency,
                    time.perf_counter() - started, attempts,
                    f"cache_write_error: {redact_secret(str(exc), api_key)}",
                )
            try:
                response_text = extract_gemini_text(raw_body)
                return CallResult(
                    "ok",
                    key,
                    response_text,
                    raw_body,
                    False,
                    latency,
                    time.perf_counter() - started,
                    attempts,
                    None,
                )
            except Exception as exc:
                return CallResult(
                    "response_error",
                    key,
                    None,
                    raw_body,
                    False,
                    latency,
                    time.perf_counter() - started,
                    attempts,
                    f"response_parse_error: {redact_secret(str(exc), api_key)}",
                )
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            last_error = redact_secret(f"HTTP {exc.code}: {detail}", api_key)
            retryable = exc.code == 429 or 500 <= exc.code < 600
            if not retryable or retry_index == 3:
                break
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            last_error = redact_secret(f"transport_error: {exc}", api_key)
            if retry_index == 3:
                break
        except Exception as exc:
            last_error = redact_secret(f"unexpected_transport_error: {exc}", api_key)
            break
        if retry_index < 3:
            sleeper(float(2**retry_index))

    return CallResult(
        "api_error",
        key,
        None,
        None,
        False,
        time.perf_counter() - network_started,
        time.perf_counter() - started,
        attempts,
        last_error or "unknown API error",
    )


## Strict parsers, D2 correction, and geometry checks

The parser accepts only one complete JSON object, exactly three ranked candidates, finite numeric values, integer pixels, and interface-specific fields; image bounds are evaluated separately as a sensor-consistency flag. Candidate-level failures remain visible. For `multi_pixel`, `server_selected` is assigned only to a fully compliant three-candidate response; any recoverable selection from a malformed response is isolated as `partial_recovery_selected` diagnostics and excluded from the formal endpoint. A valid orientation is built by Gram–Schmidt with pose columns `closing, lateral, approach`; degenerate inputs do not produce a pose. Every valid pixel is independently corrected from the real 5×5 median depth and calibrated `K`.

In [ ]:
def strict_json_object(raw_text: str) -> dict[str, Any]:
    value = json.loads(raw_text, parse_constant=reject_nonstandard_constant)
    if not isinstance(value, dict):
        raise ValueError("top-level JSON value must be an object")
    return value

def is_plain_number(value: Any) -> bool:
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and math.isfinite(float(value))
    )

def parse_numeric_vector(
    value: Any,
    *,
    length: int,
    field_name: str,
) -> tuple[float, ...]:
    if not isinstance(value, list) or len(value) != length:
        raise ValueError(f"{field_name} must be a {length}-element JSON array")
    if not all(is_plain_number(item) for item in value):
        raise ValueError(f"{field_name} must contain finite JSON numbers")
    return tuple(float(item) for item in value)

@dataclass
class ParsedCandidate:
    raw: Any
    rank: int | None = None
    schema_valid: bool = False
    errors: list[str] = field(default_factory=list)
    pixel_uv: tuple[int, int] | None = None
    position_3d: tuple[float, float, float] | None = None
    estimated_depth_m: float | None = None
    depth_band_index: int | None = None
    closing_direction_3d: tuple[float, float, float] | None = None
    approach_direction_3d: tuple[float, float, float] | None = None
    estimated_object_width_along_close_m: float | None = None
    confidence: float | None = None
    reasoning_summary: str | None = None

@dataclass
class ParsedResponse:
    interface_id: str
    strict_json_valid: bool
    top_level_valid: bool
    candidates: list[ParsedCandidate]
    errors: list[str] = field(default_factory=list)

def parse_candidate_strict(raw: Any, interface_id: str) -> ParsedCandidate:
    parsed = ParsedCandidate(raw=raw)
    if not isinstance(raw, dict):
        parsed.errors.append("candidate must be a JSON object")
        return parsed

    expected = COMMON_CANDIDATE_FIELDS | INTERFACE_EXTRA_FIELDS[interface_id]
    missing = sorted(expected - set(raw))
    unexpected = sorted(set(raw) - expected)
    if missing:
        parsed.errors.append(f"missing fields: {missing}")
    if unexpected:
        parsed.errors.append(f"unexpected fields: {unexpected}")

    rank = raw.get("rank")
    if isinstance(rank, int) and not isinstance(rank, bool):
        parsed.rank = rank
        if not 1 <= rank <= NUM_CANDIDATES:
            parsed.errors.append(f"rank must be an integer in [1,{NUM_CANDIDATES}]")
    else:
        parsed.errors.append("rank must be a JSON integer")

    pixel = raw.get("pixel_uv")
    if (
        isinstance(pixel, list)
        and len(pixel) == 2
        and all(isinstance(item, int) and not isinstance(item, bool) for item in pixel)
    ):
        parsed.pixel_uv = (pixel[0], pixel[1])
    else:
        parsed.errors.append("pixel_uv must contain exactly two JSON integers [u,v]")

    for field_name in ("closing_direction_3d", "approach_direction_3d"):
        try:
            vector = parse_numeric_vector(raw.get(field_name), length=3, field_name=field_name)
            setattr(parsed, field_name, vector)
        except ValueError as exc:
            parsed.errors.append(str(exc))

    width = raw.get("estimated_object_width_along_close_m")
    if is_plain_number(width):
        parsed.estimated_object_width_along_close_m = float(width)
        if float(width) <= 0:
            parsed.errors.append("estimated width must be positive")
    else:
        parsed.errors.append("estimated width must be a finite JSON number")

    confidence = raw.get("confidence")
    if is_plain_number(confidence):
        parsed.confidence = float(confidence)
        if not 0 <= float(confidence) <= 1:
            parsed.errors.append("confidence must be in [0,1]")
    else:
        parsed.errors.append("confidence must be a finite JSON number")

    reasoning = raw.get("reasoning_summary")
    if isinstance(reasoning, str) and reasoning.strip():
        parsed.reasoning_summary = reasoning.strip()
    else:
        parsed.errors.append("reasoning_summary must be a nonempty string")

    if interface_id == "direct_xyz":
        try:
            position = parse_numeric_vector(
                raw.get("position_3d"),
                length=3,
                field_name="position_3d",
            )
            parsed.position_3d = position  # type: ignore[assignment]
            if position[2] <= 0:
                parsed.errors.append("position_3d z must be positive")
        except ValueError as exc:
            parsed.errors.append(str(exc))
    elif interface_id == "pixel_depth":
        depth_value = raw.get("estimated_depth_m")
        if is_plain_number(depth_value) and float(depth_value) > 0:
            parsed.estimated_depth_m = float(depth_value)
        else:
            parsed.errors.append("estimated_depth_m must be a positive finite number")
    elif interface_id == "depth_band":
        band = raw.get("depth_band_index")
        if isinstance(band, int) and not isinstance(band, bool) and 0 <= band <= 10:
            parsed.depth_band_index = band
        else:
            parsed.errors.append("depth_band_index must be an integer in [0,10]")

    parsed.schema_valid = not parsed.errors
    return parsed

def parse_response_strict(raw_text: str, interface_id: str) -> ParsedResponse:
    if interface_id not in INTERFACE_IDS:
        raise KeyError(interface_id)
    try:
        body = strict_json_object(raw_text)
    except Exception as exc:
        return ParsedResponse(interface_id, False, False, [], [f"strict_json_error: {exc}"])

    top_errors: list[str] = []
    if set(body) != {"candidates"}:
        top_errors.append("top-level object must contain only candidates")
    raw_candidates = body.get("candidates")
    if not isinstance(raw_candidates, list):
        return ParsedResponse(
            interface_id,
            True,
            False,
            [],
            top_errors + ["candidates must be a JSON array"],
        )
    candidates = [parse_candidate_strict(item, interface_id) for item in raw_candidates]
    if len(candidates) != NUM_CANDIDATES:
        top_errors.append(f"expected exactly {NUM_CANDIDATES} candidates")
    ranks = [candidate.rank for candidate in candidates]
    if len(candidates) == NUM_CANDIDATES and set(ranks) != {1, 2, 3}:
        top_errors.append("candidate ranks must be unique and equal to {1,2,3}")
    return ParsedResponse(
        interface_id,
        True,
        not top_errors,
        candidates,
        top_errors,
    )


In [ ]:
@dataclass(frozen=True)
class OrientationResult:
    valid: bool
    closing_raw_norm: float | None
    approach_raw_norm: float | None
    raw_abs_cosine: float | None
    rotation_3x3: np.ndarray | None
    error: str | None

def build_orientation(
    closing_direction: Sequence[float] | None,
    approach_direction: Sequence[float] | None,
    tolerance: float = 1e-8,
) -> OrientationResult:
    if closing_direction is None or approach_direction is None:
        return OrientationResult(False, None, None, None, None, "orientation vectors missing")
    closing_raw = np.asarray(closing_direction, dtype=np.float64)
    approach_raw = np.asarray(approach_direction, dtype=np.float64)
    if closing_raw.shape != (3,) or approach_raw.shape != (3,):
        return OrientationResult(False, None, None, None, None, "orientation vectors must be length 3")
    if not np.all(np.isfinite(closing_raw)) or not np.all(np.isfinite(approach_raw)):
        return OrientationResult(False, None, None, None, None, "orientation vectors are non-finite")
    closing_norm = float(np.linalg.norm(closing_raw))
    approach_norm = float(np.linalg.norm(approach_raw))
    if closing_norm <= tolerance or approach_norm <= tolerance:
        return OrientationResult(
            False,
            closing_norm,
            approach_norm,
            None,
            None,
            "orientation vector has near-zero norm",
        )
    closing = closing_raw / closing_norm
    approach_unit_raw = approach_raw / approach_norm
    raw_abs_cosine = float(abs(np.dot(closing, approach_unit_raw)))
    approach_orthogonal = approach_unit_raw - np.dot(approach_unit_raw, closing) * closing
    orthogonal_norm = float(np.linalg.norm(approach_orthogonal))
    if orthogonal_norm <= tolerance:
        return OrientationResult(
            False,
            closing_norm,
            approach_norm,
            raw_abs_cosine,
            None,
            "closing and approach vectors are parallel or degenerate",
        )
    approach = approach_orthogonal / orthogonal_norm
    lateral = np.cross(approach, closing)
    lateral_norm = float(np.linalg.norm(lateral))
    if lateral_norm <= tolerance:
        return OrientationResult(
            False,
            closing_norm,
            approach_norm,
            raw_abs_cosine,
            None,
            "lateral axis is degenerate",
        )
    lateral /= lateral_norm
    rotation = np.column_stack((closing, lateral, approach))
    if not np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-7):
        return OrientationResult(False, closing_norm, approach_norm, raw_abs_cosine, None, "rotation is not orthonormal")
    if not np.isclose(np.linalg.det(rotation), 1.0, atol=1e-7):
        return OrientationResult(False, closing_norm, approach_norm, raw_abs_cosine, None, "rotation is not right-handed")
    return OrientationResult(True, closing_norm, approach_norm, raw_abs_cosine, rotation, None)

def pose_from_rotation_position(rotation: np.ndarray, position: Sequence[float]) -> np.ndarray:
    pose = np.eye(4, dtype=np.float64)
    pose[:3, :3] = np.asarray(rotation, dtype=np.float64)
    pose[:3, 3] = np.asarray(position, dtype=np.float64)
    return pose

def project_position(position: Sequence[float], K: np.ndarray) -> tuple[float, float]:
    x, y, z = [float(value) for value in position]
    if not math.isfinite(z) or z <= 0:
        raise ValueError("position z must be positive")
    u = float(K[0, 0]) * x / z + float(K[0, 2])
    v = float(K[1, 1]) * y / z + float(K[1, 2])
    return u, v

def depth_band_bounds(index: int) -> tuple[float, float | None]:
    if not 0 <= index <= 10:
        raise ValueError(index)
    if index == 10:
        return DEPTH_MAX_M, None
    return index * 0.25, (index + 1) * 0.25

def depth_band_metrics(index: int, measured_depth_m: float) -> tuple[bool, float]:
    lower, upper = depth_band_bounds(index)
    if upper is None:
        return measured_depth_m >= lower, max(0.0, lower - measured_depth_m)
    hit = lower <= measured_depth_m < upper
    distance = 0.0 if hit else min(abs(measured_depth_m - lower), abs(measured_depth_m - upper))
    return hit, float(distance)


In [ ]:
def evaluate_parsed_response(
    *,
    data: SceneData,
    parsed: ParsedResponse,
    stage: str,
    representation_id: str,
    interface_id: str,
    repeat_index: int,
    call_result: CallResult,
) -> list[dict[str, Any]]:
    height, width = data.depth.shape
    request_compliant = (
        parsed.strict_json_valid
        and parsed.top_level_valid
        and len(parsed.candidates) == NUM_CANDIDATES
        and all(candidate.schema_valid for candidate in parsed.candidates)
    )
    records: list[dict[str, Any]] = []
    for candidate_index, candidate in enumerate(parsed.candidates):
        errors = list(parsed.errors) + list(candidate.errors)
        pixel_in_bounds = False
        depth_stats = DepthStats((-1, -1), 0, 0, 0.0, None, None)
        if candidate.pixel_uv is not None:
            u, v = candidate.pixel_uv
            pixel_in_bounds = 0 <= u < width and 0 <= v < height
            if pixel_in_bounds:
                depth_stats = sample_depth_stats(data.depth, candidate.pixel_uv)
            else:
                errors.append(f"pixel out of bounds for {width}x{height}")
        sensor_position: np.ndarray | None = None
        if candidate.pixel_uv is not None and depth_stats.median_m is not None:
            sensor_position = deproject_pixel(
                candidate.pixel_uv[0],
                candidate.pixel_uv[1],
                depth_stats.median_m,
                data.K,
            )

        orientation = build_orientation(
            candidate.closing_direction_3d,
            candidate.approach_direction_3d,
        )
        if orientation.error:
            errors.append(orientation.error)

        predicted_position: np.ndarray | None = None
        if interface_id == "direct_xyz" and candidate.position_3d is not None:
            predicted_position = np.asarray(candidate.position_3d, dtype=np.float64)
        elif (
            interface_id == "pixel_depth"
            and candidate.pixel_uv is not None
            and candidate.estimated_depth_m is not None
        ):
            predicted_position = deproject_pixel(
                candidate.pixel_uv[0],
                candidate.pixel_uv[1],
                candidate.estimated_depth_m,
                data.K,
            )

        predicted_z_m: float | None = None
        if predicted_position is not None:
            predicted_z_m = float(predicted_position[2])

        sensor_z_abs_difference_m: float | None = None
        sensor_position_difference_m: float | None = None
        if sensor_position is not None and predicted_position is not None:
            sensor_z_abs_difference_m = float(
                abs(predicted_position[2] - sensor_position[2])
            )
            sensor_position_difference_m = float(
                np.linalg.norm(predicted_position - sensor_position)
            )

        vlm_backprojection_arithmetic_error_m: float | None = None
        projection_consistency_error_px: float | None = None
        if (
            interface_id == "direct_xyz"
            and candidate.position_3d is not None
            and candidate.pixel_uv is not None
            and candidate.position_3d[2] > 0
        ):
            arithmetic_position = deproject_pixel(
                candidate.pixel_uv[0],
                candidate.pixel_uv[1],
                candidate.position_3d[2],
                data.K,
            )
            vlm_backprojection_arithmetic_error_m = float(
                np.linalg.norm(np.asarray(candidate.position_3d) - arithmetic_position)
            )
            projected_uv = project_position(candidate.position_3d, data.K)
            projection_consistency_error_px = float(
                np.linalg.norm(np.asarray(projected_uv) - np.asarray(candidate.pixel_uv))
            )

        band_hit: bool | None = None
        band_distance_m: float | None = None
        if (
            interface_id == "depth_band"
            and candidate.depth_band_index is not None
            and depth_stats.median_m is not None
        ):
            band_hit, band_distance_m = depth_band_metrics(
                candidate.depth_band_index,
                depth_stats.median_m,
            )

        width_feasible = (
            candidate.estimated_object_width_along_close_m is not None
            and 0.02 <= candidate.estimated_object_width_along_close_m <= 0.08
        )
        corrected_pose: np.ndarray | None = None
        if sensor_position is not None and orientation.valid and orientation.rotation_3x3 is not None:
            corrected_pose = pose_from_rotation_position(
                orientation.rotation_3x3,
                sensor_position,
            )

        validation_flags = {
            "strict_json_valid": parsed.strict_json_valid,
            "top_level_valid": parsed.top_level_valid,
            "candidate_schema_valid": candidate.schema_valid,
            "request_parse_compliant": request_compliant,
            "pixel_in_bounds": pixel_in_bounds,
            "sensor_depth_valid": depth_stats.median_m is not None,
            "width_feasible_0p02_0p08_m": width_feasible,
            "orientation_valid": orientation.valid,
            "corrected_pose_valid": corrected_pose is not None,
        }
        record = {
            "scene_id": data.spec.scene_id,
            "task_id": data.spec.task_id,
            "category": data.spec.category,
            "clutter": data.spec.clutter,
            "stage": stage,
            "representation_id": representation_id,
            "interface_id": interface_id,
            "repeat_index": int(repeat_index),
            "candidate_index": candidate_index,
            "rank": candidate.rank,
            "call_status": call_result.status,
            "cache_key": call_result.cache_key,
            "cache_hit": call_result.from_cache,
            "network_latency_s": call_result.original_network_latency_s,
            "cache_lookup_latency_s": call_result.current_lookup_latency_s,
            "pixel_uv": list(candidate.pixel_uv) if candidate.pixel_uv is not None else None,
            "position_3d": list(candidate.position_3d) if candidate.position_3d is not None else None,
            "estimated_depth_m": candidate.estimated_depth_m,
            "depth_band_index": candidate.depth_band_index,
            "closing_direction_3d": (
                list(candidate.closing_direction_3d)
                if candidate.closing_direction_3d is not None
                else None
            ),
            "approach_direction_3d": (
                list(candidate.approach_direction_3d)
                if candidate.approach_direction_3d is not None
                else None
            ),
            "estimated_object_width_along_close_m": candidate.estimated_object_width_along_close_m,
            "confidence": candidate.confidence,
            "reasoning_summary": candidate.reasoning_summary,
            "closing_raw_norm": orientation.closing_raw_norm,
            "approach_raw_norm": orientation.approach_raw_norm,
            "raw_orientation_abs_cosine": orientation.raw_abs_cosine,
            "orthonormal_rotation_3x3": (
                orientation.rotation_3x3.tolist()
                if orientation.rotation_3x3 is not None
                else None
            ),
            "predicted_position_3d": (
                predicted_position.tolist() if predicted_position is not None else None
            ),
            "sensor_corrected_position_3d": (
                sensor_position.tolist() if sensor_position is not None else None
            ),
            "sensor_corrected_pose_4x4": (
                corrected_pose.tolist() if corrected_pose is not None else None
            ),
            "depth_window_area": depth_stats.window_area,
            "valid_depth_count": depth_stats.valid_count,
            "valid_depth_fraction": depth_stats.valid_fraction,
            "sensor_depth_median_m": depth_stats.median_m,
            "sensor_depth_mad_m": depth_stats.mad_m,
            "sensor_z_abs_difference_m": sensor_z_abs_difference_m,
            "sensor_position_difference_m": sensor_position_difference_m,
            "vlm_backprojection_arithmetic_error_m": vlm_backprojection_arithmetic_error_m,
            "projection_consistency_error_px": projection_consistency_error_px,
            "depth_band_hit": band_hit,
            "depth_band_distance_m": band_distance_m,
            "width_feasible": width_feasible,
            "server_selected": False,
            "partial_recovery_selected": False,
            "validation_flags": validation_flags,
            "errors": errors,
            "reference_available": False,
            "epsilon_pos_m": None,
            "xy_error_m": None,
            "raw_epsilon_pos_m": None,
            "raw_xy_error_m": None,
            "corrected_epsilon_pos_m": None,
            "corrected_xy_error_m": None,
            "orientation_error_rad": None,
            "good_pixel_hit": None,
            "good_pixel_distance_px": None,
        }
        records.append(record)

    if interface_id == "multi_pixel":
        eligible = [
            record
            for record in records
            if record["validation_flags"]["candidate_schema_valid"]
            and record["validation_flags"]["pixel_in_bounds"]
            and record["valid_depth_count"] > 0
        ]
        if eligible:
            selected = min(
                eligible,
                key=lambda record: (
                    -record["valid_depth_count"],
                    (
                        record["sensor_depth_mad_m"]
                        if record["sensor_depth_mad_m"] is not None
                        else math.inf
                    ),
                    record["rank"],
                ),
            )
            selection_field = (
                "server_selected" if request_compliant else "partial_recovery_selected"
            )
            selected[selection_field] = True
    return records

def evaluate_call(
    *,
    data: SceneData,
    stage: str,
    representation_id: str,
    interface_id: str,
    repeat_index: int,
    call_result: CallResult,
) -> tuple[ParsedResponse, list[dict[str, Any]]]:
    if call_result.response_text is None:
        parsed = ParsedResponse(
            interface_id,
            False,
            False,
            [],
            [call_result.error or call_result.status],
        )
        return parsed, []
    parsed = parse_response_strict(call_result.response_text, interface_id)
    records = evaluate_parsed_response(
        data=data,
        parsed=parsed,
        stage=stage,
        representation_id=representation_id,
        interface_id=interface_id,
        repeat_index=repeat_index,
        call_result=call_result,
    )
    return parsed, records


In [ ]:
@dataclass(frozen=True)
class ReferenceEntry:
    scene_id: str
    task_id: str
    poses_4x4: tuple[np.ndarray, ...]
    symmetry_rotations_3x3: tuple[np.ndarray, ...]
    good_pixels_uv: frozenset[tuple[int, int]] | None

def validate_rotation(rotation: np.ndarray, *, name: str) -> None:
    if rotation.shape != (3, 3) or not np.all(np.isfinite(rotation)):
        raise ValueError(f"{name} must be a finite 3x3 matrix")
    if not np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-5):
        raise ValueError(f"{name} is not orthonormal")
    if not np.isclose(np.linalg.det(rotation), 1.0, atol=1e-5):
        raise ValueError(f"{name} must have determinant +1")

def load_reference_grasps(
    path: Path = REFERENCE_GRASPS_PATH,
    *,
    contents: bytes | None = None,
) -> dict[tuple[str, str], ReferenceEntry]:
    if contents is None:
        if not path.exists():
            return {}
        contents = path.read_bytes()
    try:
        text = contents.decode("utf-8")
    except UnicodeDecodeError as exc:
        raise ValueError(f"{path}: reference file must be UTF-8") from exc
    entries: dict[tuple[str, str], ReferenceEntry] = {}
    with StringIO(text) as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            raw = json.loads(line, parse_constant=reject_nonstandard_constant)
            if not isinstance(raw, dict):
                raise ValueError(f"{path}:{line_number}: row must be an object")
            scene_id = raw.get("scene_id")
            task_id = raw.get("task_id")
            if not isinstance(scene_id, str) or not isinstance(task_id, str):
                raise ValueError(f"{path}:{line_number}: scene_id/task_id must be strings")
            key = (scene_id, task_id)
            if key in entries:
                raise ValueError(f"{path}:{line_number}: duplicate key {key}")

            raw_grasps = raw.get("grasps")
            if not isinstance(raw_grasps, list) or not raw_grasps:
                raise ValueError(f"{path}:{line_number}: grasps must be a nonempty list")
            poses: list[np.ndarray] = []
            for index, grasp in enumerate(raw_grasps):
                if not isinstance(grasp, dict) or "pose_4x4" not in grasp:
                    raise ValueError(f"{path}:{line_number}: grasp[{index}] lacks pose_4x4")
                pose = np.asarray(grasp["pose_4x4"], dtype=np.float64)
                if pose.shape != (4, 4) or not np.all(np.isfinite(pose)):
                    raise ValueError(f"{path}:{line_number}: grasp[{index}] pose is not finite 4x4")
                if not np.allclose(pose[3], [0, 0, 0, 1], atol=1e-7):
                    raise ValueError(f"{path}:{line_number}: grasp[{index}] is not homogeneous")
                validate_rotation(pose[:3, :3], name=f"grasp[{index}] rotation")
                poses.append(pose)

            symmetries: list[np.ndarray] = [np.eye(3, dtype=np.float64)]
            raw_symmetries = raw.get("symmetry_rotations_3x3", [])
            if not isinstance(raw_symmetries, list):
                raise ValueError(f"{path}:{line_number}: symmetry_rotations_3x3 must be a list")
            for index, raw_rotation in enumerate(raw_symmetries):
                rotation = np.asarray(raw_rotation, dtype=np.float64)
                validate_rotation(rotation, name=f"symmetry[{index}]")
                if not any(np.allclose(rotation, existing, atol=1e-7) for existing in symmetries):
                    symmetries.append(rotation)

            raw_pixels = raw.get("good_pixels_uv")
            pixels: set[tuple[int, int]] | None = None
            if raw_pixels is not None:
                if not isinstance(raw_pixels, list):
                    raise ValueError(f"{path}:{line_number}: good_pixels_uv must be a list")
                pixels = set()
                for pixel in raw_pixels:
                    if (
                        not isinstance(pixel, list)
                        or len(pixel) != 2
                        or not all(isinstance(value, int) and not isinstance(value, bool) for value in pixel)
                    ):
                        raise ValueError(f"{path}:{line_number}: invalid good pixel {pixel!r}")
                    pixels.add((pixel[0], pixel[1]))
            entries[key] = ReferenceEntry(
                scene_id,
                task_id,
                tuple(poses),
                tuple(symmetries),
                None if pixels is None else frozenset(pixels),
            )
    return entries

def rotation_geodesic_rad(left: np.ndarray, right: np.ndarray) -> float:
    relative = left.T @ right
    cosine = float(np.clip((np.trace(relative) - 1.0) / 2.0, -1.0, 1.0))
    return float(math.acos(cosine))

def apply_reference_metrics(
    records: list[dict[str, Any]],
    references: dict[tuple[str, str], ReferenceEntry],
) -> None:
    for record in records:
        reference = references.get((record["scene_id"], record["task_id"]))
        if reference is None:
            continue
        record["reference_available"] = True
        pixel = record.get("pixel_uv")
        if pixel is not None and reference.good_pixels_uv:
            pixel_tuple = tuple(pixel)
            distances = [
                float(np.linalg.norm(np.asarray(pixel_tuple) - np.asarray(good_pixel)))
                for good_pixel in reference.good_pixels_uv
            ]
            record["good_pixel_hit"] = pixel_tuple in reference.good_pixels_uv
            record["good_pixel_distance_px"] = min(distances)

        raw_position_value = record.get("predicted_position_3d")
        if raw_position_value is not None:
            raw_position = np.asarray(raw_position_value, dtype=np.float64)
            raw_errors = [
                float(np.linalg.norm(raw_position - reference_pose[:3, 3]))
                for reference_pose in reference.poses_4x4
            ]
            raw_reference_index = int(np.argmin(raw_errors))
            raw_reference_pose = reference.poses_4x4[raw_reference_index]
            record["raw_epsilon_pos_m"] = raw_errors[raw_reference_index]
            record["raw_xy_error_m"] = float(
                np.linalg.norm(raw_position[:2] - raw_reference_pose[:2, 3])
            )
            record["epsilon_pos_m"] = record["raw_epsilon_pos_m"]
            record["xy_error_m"] = record["raw_xy_error_m"]

        corrected_position_value = record.get("sensor_corrected_position_3d")
        if corrected_position_value is not None:
            corrected_position = np.asarray(corrected_position_value, dtype=np.float64)
            corrected_errors = [
                float(np.linalg.norm(corrected_position - reference_pose[:3, 3]))
                for reference_pose in reference.poses_4x4
            ]
            corrected_reference_index = int(np.argmin(corrected_errors))
            corrected_reference_pose = reference.poses_4x4[corrected_reference_index]
            record["corrected_epsilon_pos_m"] = corrected_errors[corrected_reference_index]
            record["corrected_xy_error_m"] = float(
                np.linalg.norm(corrected_position[:2] - corrected_reference_pose[:2, 3])
            )

        rotation_value = record.get("orthonormal_rotation_3x3")
        if rotation_value is not None:
            rotation = np.asarray(rotation_value, dtype=np.float64)
            record["orientation_error_rad"] = min(
                rotation_geodesic_rad(rotation, reference_pose[:3, :3] @ symmetry)
                for reference_pose in reference.poses_4x4
                for symmetry in reference.symmetry_rotations_3x3
            )

REFERENCES_AT_SETUP = load_reference_grasps()
print(
    f"Reference grasps at setup: {len(REFERENCES_AT_SETUP)} scene-task entries"
    if REFERENCES_AT_SETUP
    else "Reference grasps: unavailable; all true position/orientation/good-pixel metrics remain null"
)


## Batch orchestration, resume, artifacts, and statistics

`smoke`, `pilot`, and `full` have nominal no-retry HTTP caps of 10, 40, and 364. R3 localization is once per scene-task. D1 runs six `direct_xyz` representations; D3 compares four interfaces on the chosen representation while reusing the exact D1 `direct_xyz` result. Failures are logged without terminating the batch, and append-only call/candidate JSONL files provide checkpoint resume.

In [ ]:
def json_safe(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set, frozenset)):
        return [json_safe(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def dump_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(json_safe(value), ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False),
        encoding="utf-8",
    )

def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(json_safe(row), ensure_ascii=False, sort_keys=True, allow_nan=False)
            + "\n"
        )

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            row = json.loads(line, parse_constant=reject_nonstandard_constant)
            if not isinstance(row, dict):
                raise ValueError(f"{path}:{line_number}: JSONL row must be an object")
            rows.append(row)
    return rows

def latest_call_attempts(rows: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    for row in rows:
        latest[str(row["logical_id"])] = row
    return list(latest.values())

def latest_candidate_attempts(rows: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    latest: dict[tuple[str, int], dict[str, Any]] = {}
    for row in rows:
        key = (str(row["logical_id"]), int(row.get("candidate_index", -1)))
        latest[key] = row
    return list(latest.values())

def candidates_for_latest_calls(
    rows: Sequence[dict[str, Any]],
    call_rows: Sequence[dict[str, Any]],
) -> list[dict[str, Any]]:
    current_cache_keys = {
        str(row["logical_id"]): str(row.get("cache_key") or "")
        for row in latest_call_attempts(call_rows)
    }
    matching = [
        row
        for row in rows
        if str(row.get("cache_key") or "")
        == current_cache_keys.get(str(row.get("logical_id")))
    ]
    return latest_candidate_attempts(matching)

def write_csv(path: Path, rows: Sequence[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = sorted({key for row in rows for key in row})
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            cooked: dict[str, Any] = {}
            for key in fieldnames:
                value = json_safe(row.get(key))
                cooked[key] = (
                    json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
                    if isinstance(value, (dict, list))
                    else value
                )
            writer.writerow(cooked)

def profile_specs(profile: str) -> tuple[SceneSpec, ...]:
    selected = set(PROFILE_CONFIGS[profile]["scene_ids"])
    specs = tuple(spec for spec in MANIFEST if spec.scene_id in selected)
    assert len(specs) == len(selected)
    return specs

def logical_request_id(
    stage: str,
    scene_id: str,
    task_id: str,
    representation_id: str,
    interface_id: str,
    repeat_index: int,
) -> str:
    payload = {
        "stage": stage,
        "scene_id": scene_id,
        "task_id": task_id,
        "representation_id": representation_id,
        "interface_id": interface_id,
        "repeat_index": repeat_index,
    }
    return hashlib.sha256(canonical_json_bytes(payload)).hexdigest()[:20]

def save_raw_response(run_dir: Path, logical_id: str, result: CallResult) -> str | None:
    if result.raw_body is None:
        return None
    path = run_dir / "raw_responses" / f"{logical_id}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(result.raw_body, encoding="utf-8")
    return str(path.relative_to(run_dir))

def save_prompt(run_dir: Path, logical_id: str, prompt: str) -> str:
    path = run_dir / "prompts" / f"{logical_id}.txt"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(prompt, encoding="utf-8")
    return str(path.relative_to(run_dir))

def save_representation_inputs(
    run_dir: Path,
    data: SceneData,
    bundle: RepresentationBundle,
) -> list[str]:
    directory = run_dir / "inputs" / data.spec.scene_id / bundle.representation_id
    directory.mkdir(parents=True, exist_ok=True)
    paths: list[str] = []
    for index, (descriptor, image) in enumerate(bundle.images):
        path = directory / f"{index:02d}.png"
        image.save(path, format="PNG")
        paths.append(str(path.relative_to(run_dir)))
    dump_json(
        directory / "metadata.json",
        {
            "scene_id": data.spec.scene_id,
            "task_id": data.spec.task_id,
            "representation_id": bundle.representation_id,
            "image_descriptors": [descriptor for descriptor, _ in bundle.images],
            "metadata": bundle.metadata,
        },
    )
    return paths

def make_call_row(
    *,
    logical_id: str,
    data: SceneData,
    stage: str,
    representation_id: str,
    interface_id: str,
    repeat_index: int,
    result: CallResult,
    parsed: ParsedResponse | None,
    raw_response_path: str | None,
    prompt_path: str | None,
    input_paths: Sequence[str],
    amortized_anchor_latency_s: float = 0.0,
    reused_from_logical_id: str | None = None,
) -> dict[str, Any]:
    parse_compliant = bool(
        parsed is not None
        and parsed.strict_json_valid
        and parsed.top_level_valid
        and len(parsed.candidates) == NUM_CANDIDATES
        and all(candidate.schema_valid for candidate in parsed.candidates)
    )
    return {
        "logical_id": logical_id,
        "scene_id": data.spec.scene_id,
        "task_id": data.spec.task_id,
        "category": data.spec.category,
        "clutter": data.spec.clutter,
        "stage": stage,
        "representation_id": representation_id,
        "interface_id": interface_id,
        "repeat_index": repeat_index,
        "call_status": result.status,
        "parse_compliant": parse_compliant,
        "strict_json_valid": parsed.strict_json_valid if parsed is not None else False,
        "top_level_valid": parsed.top_level_valid if parsed is not None else False,
        "candidate_count": len(parsed.candidates) if parsed is not None else 0,
        "cache_key": result.cache_key,
        "cache_hit": result.from_cache,
        "network_latency_s": result.original_network_latency_s,
        "cache_lookup_latency_s": result.current_lookup_latency_s,
        "amortized_anchor_latency_s": amortized_anchor_latency_s,
        "effective_cold_latency_s": (
            result.original_network_latency_s + amortized_anchor_latency_s
            if result.original_network_latency_s is not None
            else None
        ),
        "transport_attempts": result.transport_attempts,
        "error": result.error,
        "raw_response_path": raw_response_path,
        "prompt_path": prompt_path,
        "input_paths": list(input_paths),
        "reused_from_logical_id": reused_from_logical_id,
    }

def synthetic_call_result(status: str, error: str) -> CallResult:
    return CallResult(status, "", None, None, False, None, 0.0, 0, error)


In [ ]:
SUMMARY_METRICS = (
    "parse_compliant",
    "pixel_out_of_bounds",
    "valid_depth",
    "valid_depth_fraction",
    "sensor_depth_mad_m",
    "sensor_z_abs_difference_m",
    "sensor_position_difference_m",
    "vlm_backprojection_arithmetic_error_m",
    "projection_consistency_error_px",
    "width_feasible",
    "orientation_valid",
    "network_latency_s",
    "effective_cold_latency_s",
    "cache_lookup_latency_s",
    "cache_hit",
    "transport_attempts",
    "depth_band_hit",
    "depth_band_distance_m",
    "epsilon_pos_m",
    "xy_error_m",
    "corrected_epsilon_pos_m",
    "corrected_xy_error_m",
    "orientation_error_rad",
    "good_pixel_hit",
    "good_pixel_distance_px",
)

def finite_float(value: Any) -> float | None:
    if isinstance(value, bool):
        return float(value)
    if is_plain_number(value):
        return float(value)
    return None

def mean_sd(values: Sequence[float]) -> tuple[float | None, float | None]:
    finite = [float(value) for value in values if math.isfinite(float(value))]
    if not finite:
        return None, None
    return (
        float(statistics.mean(finite)),
        float(statistics.stdev(finite)) if len(finite) > 1 else None,
    )

def candidate_for_call(
    candidate_rows: Sequence[dict[str, Any]],
    logical_id: str,
    cache_key_value: str,
    *,
    rank: int = 1,
    server_selected: bool = False,
) -> dict[str, Any] | None:
    matches = [
        row
        for row in candidate_rows
        if row.get("logical_id") == logical_id
        and str(row.get("cache_key") or "") == cache_key_value
        and (
            bool(row.get("server_selected"))
            if server_selected
            else row.get("rank") == rank
        )
    ]
    return matches[0] if matches else None

def metric_value_for_call(
    call: dict[str, Any],
    candidate_rows: Sequence[dict[str, Any]],
    metric: str,
    *,
    rank: int = 1,
    server_selected: bool = False,
) -> float | None:
    if metric == "parse_compliant":
        return float(bool(call.get("parse_compliant")))
    candidate = candidate_for_call(
        candidate_rows,
        call["logical_id"],
        str(call.get("cache_key") or ""),
        rank=rank,
        server_selected=server_selected,
    )
    if metric in {
        "network_latency_s",
        "effective_cold_latency_s",
        "cache_lookup_latency_s",
        "transport_attempts",
    }:
        return finite_float(call.get(metric))
    if metric == "cache_hit":
        return float(bool(call.get("cache_hit")))
    candidate_schema_valid = bool(
        candidate and candidate.get("validation_flags", {}).get("candidate_schema_valid")
    )
    if metric == "pixel_out_of_bounds":
        return float(
            bool(
                candidate
                and candidate.get("pixel_uv") is not None
                and not candidate.get("validation_flags", {}).get("pixel_in_bounds")
            )
        )
    if metric == "valid_depth":
        return float(
            bool(
                candidate_schema_valid
                and candidate
                and candidate.get("validation_flags", {}).get("sensor_depth_valid")
            )
        )
    if metric == "width_feasible":
        return float(bool(candidate_schema_valid and candidate and candidate.get("width_feasible")))
    if metric == "orientation_valid":
        return float(
            bool(
                candidate_schema_valid
                and candidate
                and candidate.get("validation_flags", {}).get("orientation_valid")
            )
        )
    if candidate is None or not candidate_schema_valid:
        return None
    return finite_float(candidate.get(metric))

def method_id(call: dict[str, Any]) -> str:
    return (
        str(call["representation_id"])
        if call["stage"] == "D1"
        else str(call["interface_id"])
    )

def scene_method_means(
    call_rows: Sequence[dict[str, Any]],
    candidate_rows: Sequence[dict[str, Any]],
    *,
    stage: str,
    metric: str,
    rank: int = 1,
    server_selected: bool = False,
) -> dict[tuple[str, str, str], float]:
    repeat_values: dict[tuple[str, str, str], list[float]] = {}
    for call in call_rows:
        if call.get("stage") != stage:
            continue
        method = method_id(call)
        if server_selected and not (stage == "D3" and method == "multi_pixel"):
            continue
        value = metric_value_for_call(
            call,
            candidate_rows,
            metric,
            rank=rank,
            server_selected=server_selected,
        )
        if value is None:
            continue
        key = (str(call["scene_id"]), str(call["task_id"]), method)
        repeat_values.setdefault(key, []).append(value)
    return {
        key: float(statistics.mean(values))
        for key, values in repeat_values.items()
        if values
    }

def summarize_rank1(
    call_rows: Sequence[dict[str, Any]],
    candidate_rows: Sequence[dict[str, Any]],
) -> list[dict[str, Any]]:
    output: list[dict[str, Any]] = []
    for stage in ("D1", "D3"):
        methods = sorted(
            {
                method_id(call)
                for call in call_rows
                if call.get("stage") == stage
            }
        )
        for metric in SUMMARY_METRICS:
            scene_means = scene_method_means(
                call_rows,
                candidate_rows,
                stage=stage,
                metric=metric,
            )
            for method in methods:
                values = [
                    value
                    for (scene_id, task_id, row_method), value in scene_means.items()
                    if row_method == method
                ]
                mean, sd = mean_sd(values)
                output.append(
                    {
                        "stage": stage,
                        "method": method,
                        "analysis_endpoint": "rank_1",
                        "metric": metric,
                        "n_scene_tasks": len(values),
                        "mean": mean,
                        "sd": sd,
                        "mean_plus_minus_sd": (
                            None
                            if mean is None
                            else f"{mean:.6g} ± {sd:.6g}" if sd is not None else f"{mean:.6g} ± NA"
                        ),
                    }
                )
    # Multi-pixel server selection is an operational endpoint, never a rank-1 substitute.
    for metric in SUMMARY_METRICS:
        scene_means = scene_method_means(
            call_rows,
            candidate_rows,
            stage="D3",
            metric=metric,
            server_selected=True,
        )
        values = list(scene_means.values())
        if not values:
            continue
        mean, sd = mean_sd(values)
        output.append(
            {
                "stage": "D3",
                "method": "multi_pixel",
                "analysis_endpoint": "server_selected_operational",
                "metric": metric,
                "n_scene_tasks": len(values),
                "mean": mean,
                "sd": sd,
                "mean_plus_minus_sd": (
                    f"{mean:.6g} ± {sd:.6g}" if mean is not None and sd is not None
                    else f"{mean:.6g} ± NA" if mean is not None else None
                ),
            }
        )
    return output

def summarize_all_candidate_ranks(
    call_rows: Sequence[dict[str, Any]],
    candidate_rows: Sequence[dict[str, Any]],
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    diagnostic_metrics = (
        "valid_depth",
        "sensor_z_abs_difference_m",
        "sensor_position_difference_m",
        "width_feasible",
        "orientation_valid",
    )
    for stage in ("D1", "D3"):
        methods = sorted({method_id(call) for call in call_rows if call.get("stage") == stage})
        for rank in (1, 2, 3):
            for metric in diagnostic_metrics:
                scene_means = scene_method_means(
                    call_rows,
                    candidate_rows,
                    stage=stage,
                    metric=metric,
                    rank=rank,
                )
                for method in methods:
                    values = [
                        value
                        for (_, _, row_method), value in scene_means.items()
                        if row_method == method
                    ]
                    mean, sd = mean_sd(values)
                    rows.append(
                        {
                            "stage": stage,
                            "method": method,
                            "rank": rank,
                            "metric": metric,
                            "n_scene_tasks": len(values),
                            "mean": mean,
                            "sd": sd,
                        }
                    )
    return rows

def paired_wilcoxon_rows(
    call_rows: Sequence[dict[str, Any]],
    candidate_rows: Sequence[dict[str, Any]],
) -> list[dict[str, Any]]:
    comparisons: list[dict[str, Any]] = []
    comparable_metrics = (
        "parse_compliant",
        "pixel_out_of_bounds",
        "valid_depth",
        "valid_depth_fraction",
        "sensor_depth_mad_m",
        "sensor_z_abs_difference_m",
        "sensor_position_difference_m",
        "vlm_backprojection_arithmetic_error_m",
        "projection_consistency_error_px",
        "width_feasible",
        "orientation_valid",
        "network_latency_s",
        "effective_cold_latency_s",
        "cache_lookup_latency_s",
        "epsilon_pos_m",
        "xy_error_m",
        "corrected_epsilon_pos_m",
        "corrected_xy_error_m",
        "orientation_error_rad",
        "good_pixel_hit",
        "good_pixel_distance_px",
    )
    for stage, baseline in (("D1", "r1_calibrated_jet"), ("D3", "direct_xyz")):
        methods = sorted({method_id(call) for call in call_rows if call.get("stage") == stage})
        for metric in comparable_metrics:
            table = scene_method_means(
                call_rows,
                candidate_rows,
                stage=stage,
                metric=metric,
            )
            baseline_values = {
                (scene_id, task_id): value
                for (scene_id, task_id, method), value in table.items()
                if method == baseline
            }
            for method in methods:
                if method == baseline:
                    continue
                method_values = {
                    (scene_id, task_id): value
                    for (scene_id, task_id, row_method), value in table.items()
                    if row_method == method
                }
                shared = sorted(set(baseline_values) & set(method_values))
                x = np.asarray([method_values[key] for key in shared], dtype=np.float64)
                y = np.asarray([baseline_values[key] for key in shared], dtype=np.float64)
                statistic: float | None = None
                p_value: float | None = None
                error: str | None = None
                if len(shared):
                    differences = x - y
                    if np.all(differences == 0.0):
                        statistic, p_value = 0.0, 1.0
                    else:
                        try:
                            result = wilcoxon(x, y, alternative="two-sided")
                            statistic = float(result.statistic)
                            p_value = float(result.pvalue)
                        except ValueError as exc:
                            error = str(exc)
                comparisons.append(
                    {
                        "stage": stage,
                        "baseline": baseline,
                        "method": method,
                        "metric": metric,
                        "n_pairs": len(shared),
                        "statistic": statistic,
                        "p_value": p_value,
                        "error": error,
                    }
                )
    return comparisons

def recommend_d1_method(
    call_rows: Sequence[dict[str, Any]],
    candidate_rows: Sequence[dict[str, Any]],
    override: str = D1_FOR_D3,
) -> tuple[str | None, list[dict[str, Any]]]:
    rows: list[dict[str, Any]] = []
    scene_tables = {
        metric: scene_method_means(
            call_rows,
            candidate_rows,
            stage="D1",
            metric=metric,
        )
        for metric in (
            "parse_compliant",
            "valid_depth",
            "sensor_z_abs_difference_m",
            "sensor_position_difference_m",
            "effective_cold_latency_s",
        )
    }

    def across_scene_values(metric: str, representation_id: str) -> list[float]:
        return [
            value
            for (_, _, method), value in scene_tables[metric].items()
            if method == representation_id
        ]

    for representation_id in REPRESENTATION_IDS:
        parse_values = across_scene_values("parse_compliant", representation_id)
        valid_values = across_scene_values("valid_depth", representation_id)
        z_values = across_scene_values("sensor_z_abs_difference_m", representation_id)
        position_values = across_scene_values("sensor_position_difference_m", representation_id)
        latency_values = across_scene_values("effective_cold_latency_s", representation_id)
        parse_rate = float(statistics.mean(parse_values)) if parse_values else 0.0
        valid_rate = float(statistics.mean(valid_values)) if valid_values else 0.0
        rows.append(
            {
                "representation_id": representation_id,
                "parse_compliance_rate": parse_rate,
                "rank1_valid_depth_rate": valid_rate,
                "rank1_mean_sensor_z_abs_difference_m": (
                    float(statistics.mean(z_values)) if z_values else None
                ),
                "rank1_mean_sensor_position_difference_m": (
                    float(statistics.mean(position_values)) if position_values else None
                ),
                "mean_effective_cold_latency_s": (
                    float(statistics.mean(latency_values)) if latency_values else None
                ),
                "eligible": parse_rate >= 0.90 and valid_rate >= 0.90,
            }
        )
    if override != "auto":
        if override not in REPRESENTATION_IDS:
            raise ValueError(f"PHASE1A_D1_FOR_D3 must be auto or one of {REPRESENTATION_IDS}")
        return override, rows
    eligible = [row for row in rows if row["eligible"]]
    if not eligible:
        return None, rows
    winner = min(
        eligible,
        key=lambda row: (
            (
                finite_float(row["rank1_mean_sensor_z_abs_difference_m"])
                if finite_float(row["rank1_mean_sensor_z_abs_difference_m"]) is not None
                else math.inf
            ),
            (
                finite_float(row["rank1_mean_sensor_position_difference_m"])
                if finite_float(row["rank1_mean_sensor_position_difference_m"]) is not None
                else math.inf
            ),
            (
                finite_float(row["mean_effective_cold_latency_s"])
                if finite_float(row["mean_effective_cold_latency_s"]) is not None
                else math.inf
            ),
            row["representation_id"],
        ),
    )
    return str(winner["representation_id"]), rows


In [ ]:
@dataclass
class AnchorOutcome:
    anchors: AnchorSet | None
    call_result: CallResult | None
    error: str | None
    logical_id: str | None
    original_latency_s: float

def load_saved_anchor(run_dir: Path, data: SceneData) -> tuple[AnchorSet, str] | None:
    path = run_dir / "anchors" / f"{data.spec.scene_id}__{data.spec.task_id}.json"
    if not path.exists():
        return None
    raw = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_nonstandard_constant)
    anchor_raw = raw.get("anchor_set")
    saved_key = raw.get("cache_key")
    if not isinstance(anchor_raw, dict) or not isinstance(saved_key, str):
        return None
    anchors = AnchorSet(
        object_bbox_xyxy=tuple(anchor_raw["object_bbox_xyxy"]),
        object_uv=tuple(anchor_raw["object_uv"]),
        grasp_uvs=tuple(tuple(point) for point in anchor_raw["grasp_uvs"]),
        table_uv=None if anchor_raw.get("table_uv") is None else tuple(anchor_raw["table_uv"]),
        source=str(anchor_raw["source"]),
    )
    return anchors, saved_key

def save_anchor(run_dir: Path, data: SceneData, anchors: AnchorSet, cache_key_value: str) -> None:
    dump_json(
        run_dir / "anchors" / f"{data.spec.scene_id}__{data.spec.task_id}.json",
        {
            "anchor_prompt_version": ANCHOR_PROMPT_VERSION,
            "cache_key": cache_key_value,
            "anchor_set": asdict(anchors),
        },
    )

def get_scene_anchors(
    *,
    data: SceneData,
    run_dir: Path,
    online: bool,
    budget: CallBudget,
    call_rows: list[dict[str, Any]],
    completed_ids: set[str],
) -> AnchorOutcome:
    if data.spec.anchor_override is not None:
        anchors = anchor_set_from_override(data, data.spec.anchor_override)
        override_key = "manifest_override:" + hashlib.sha256(
            canonical_json_bytes(
                {
                    "scene_id": data.spec.scene_id,
                    "task_id": data.spec.task_id,
                    "task": data.spec.effective_task_spec,
                    "anchor_override": data.spec.anchor_override,
                }
            )
        ).hexdigest()
        save_anchor(run_dir, data, anchors, override_key)
        return AnchorOutcome(anchors, None, None, None, 0.0)

    logical_id = logical_request_id(
        "R3_anchor",
        data.spec.scene_id,
        data.spec.task_id,
        "rgb_only",
        "rgb_anchor_localization",
        0,
    )
    prompt = build_anchor_prompt(data)
    images = encode_rgb_only(data)
    expected_anchor_key = cache_key(
        semantic_interface="rgb_anchor_localization",
        prompt=prompt,
        images=images,
        repeat_index=0,
    )
    saved = load_saved_anchor(run_dir, data)
    if saved is not None and saved[1] == expected_anchor_key:
        saved_anchors = saved[0]
        previous = next(
            (row for row in reversed(call_rows) if row.get("logical_id") == logical_id),
            None,
        )
        latency = float(previous.get("network_latency_s") or 0.0) if previous else 0.0
        return AnchorOutcome(saved_anchors, None, None, logical_id, latency)

    if logical_id in completed_ids:
        prior = next(row for row in reversed(call_rows) if row["logical_id"] == logical_id)
        if prior.get("cache_key") != expected_anchor_key:
            completed_ids.discard(logical_id)

    if logical_id in completed_ids:
        previous = next(row for row in reversed(call_rows) if row["logical_id"] == logical_id)
        raw_path = previous.get("raw_response_path")
        if raw_path:
            try:
                raw_body = (run_dir / raw_path).read_text(encoding="utf-8")
                raw_text = extract_gemini_text(raw_body)
                anchors = derive_anchor_set(data, raw_text)
                save_anchor(run_dir, data, anchors, expected_anchor_key)
                return AnchorOutcome(
                    anchors,
                    None,
                    None,
                    logical_id,
                    float(previous.get("network_latency_s") or 0.0),
                )
            except Exception as exc:
                return AnchorOutcome(None, None, f"saved anchor parse failed: {exc}", logical_id, 0.0)
        return AnchorOutcome(
            None,
            None,
            str(previous.get("error") or previous.get("call_status")),
            logical_id,
            float(previous.get("network_latency_s") or 0.0),
        )

    prompt_path = save_prompt(run_dir, logical_id, prompt)
    anchor_input = run_dir / "inputs" / data.spec.scene_id / "rgb_only_anchor.png"
    anchor_input.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(data.rgb, "RGB").save(anchor_input, format="PNG")
    result = run_cached_gemini_request(
        semantic_interface="rgb_anchor_localization",
        prompt=prompt,
        images=images,
        repeat_index=0,
        online=online,
        budget=budget,
    )
    raw_path = save_raw_response(run_dir, logical_id, result)
    anchors: AnchorSet | None = None
    anchor_error = result.error
    if result.response_text is not None:
        try:
            anchors = derive_anchor_set(data, result.response_text)
            save_anchor(run_dir, data, anchors, result.cache_key)
            anchor_error = None
        except Exception as exc:
            anchor_error = f"RGB-only prelocalization parse failed: {exc}"
    row = {
        "logical_id": logical_id,
        "scene_id": data.spec.scene_id,
        "task_id": data.spec.task_id,
        "category": data.spec.category,
        "clutter": data.spec.clutter,
        "stage": "R3_anchor",
        "representation_id": "rgb_only",
        "interface_id": "rgb_anchor_localization",
        "repeat_index": 0,
        "call_status": (
            "anchor_ok"
            if anchors is not None
            else "anchor_parse_failed"
            if result.response_text is not None
            else result.status
        ),
        "parse_compliant": anchors is not None,
        "strict_json_valid": None,
        "top_level_valid": None,
        "candidate_count": None,
        "cache_key": result.cache_key,
        "cache_hit": result.from_cache,
        "network_latency_s": result.original_network_latency_s,
        "cache_lookup_latency_s": result.current_lookup_latency_s,
        "amortized_anchor_latency_s": 0.0,
        "effective_cold_latency_s": result.original_network_latency_s,
        "transport_attempts": result.transport_attempts,
        "error": anchor_error,
        "raw_response_path": raw_path,
        "prompt_path": prompt_path,
        "input_paths": [str(anchor_input.relative_to(run_dir))],
        "reused_from_logical_id": None,
    }
    call_rows.append(row)
    append_jsonl(run_dir / "calls.jsonl", row)
    completed_ids.add(logical_id)
    return AnchorOutcome(
        anchors,
        result,
        anchor_error,
        logical_id,
        float(result.original_network_latency_s or 0.0),
    )

def rehydrate_call_result(run_dir: Path, row: dict[str, Any]) -> CallResult:
    raw_path = row.get("raw_response_path")
    raw_body = (run_dir / raw_path).read_text(encoding="utf-8") if raw_path else None
    response_text: str | None = None
    error = row.get("error")
    if raw_body is not None:
        try:
            response_text = extract_gemini_text(raw_body)
        except Exception as exc:
            error = f"rehydrated_response_error: {exc}"
    return CallResult(
        status=str(row.get("call_status")),
        cache_key=str(row.get("cache_key") or ""),
        response_text=response_text,
        raw_body=raw_body,
        from_cache=bool(row.get("cache_hit")),
        original_network_latency_s=finite_float(row.get("network_latency_s")),
        current_lookup_latency_s=float(row.get("cache_lookup_latency_s") or 0.0),
        transport_attempts=0,
        error=error,
    )


In [ ]:
@dataclass
class RequestState:
    logical_id: str
    result: CallResult
    parsed: ParsedResponse
    records: list[dict[str, Any]]
    call_row: dict[str, Any]

def execute_benchmark_request(
    *,
    data: SceneData,
    bundle: RepresentationBundle,
    stage: str,
    interface_id: str,
    repeat_index: int,
    run_dir: Path,
    online: bool,
    budget: CallBudget,
    call_rows: list[dict[str, Any]],
    candidate_rows: list[dict[str, Any]],
    completed_ids: set[str],
    references: dict[tuple[str, str], ReferenceEntry],
    amortized_anchor_latency_s: float = 0.0,
    reuse_state: RequestState | None = None,
) -> RequestState:
    representation_id = bundle.representation_id
    logical_id = logical_request_id(
        stage,
        data.spec.scene_id,
        data.spec.task_id,
        representation_id,
        interface_id,
        repeat_index,
    )
    prompt = build_task_prompt(data, interface_id)
    images = encode_bundle(bundle)

    expected_cache_key = cache_key(
        semantic_interface=interface_id,
        prompt=prompt,
        images=images,
        repeat_index=repeat_index,
    )
    if logical_id in completed_ids:
        prior = next(row for row in reversed(call_rows) if row["logical_id"] == logical_id)
        if prior.get("cache_key") != expected_cache_key:
            completed_ids.discard(logical_id)
            candidate_rows[:] = [
                row for row in candidate_rows if row.get("logical_id") != logical_id
            ]

    if logical_id in completed_ids:
        previous = next(row for row in reversed(call_rows) if row["logical_id"] == logical_id)
        result = rehydrate_call_result(run_dir, previous)
        parsed = (
            parse_response_strict(result.response_text, interface_id)
            if result.response_text is not None
            else ParsedResponse(interface_id, False, False, [], [result.error or result.status])
        )
        existing_records = [
            row
            for row in candidate_rows
            if row.get("logical_id") == logical_id
            and str(row.get("cache_key") or "") == str(result.cache_key or "")
        ]
        if not existing_records and result.response_text is not None:
            existing_records = evaluate_parsed_response(
                data=data,
                parsed=parsed,
                stage=stage,
                representation_id=representation_id,
                interface_id=interface_id,
                repeat_index=repeat_index,
                call_result=result,
            )
            apply_reference_metrics(existing_records, references)
            for record in existing_records:
                record["logical_id"] = logical_id
                append_jsonl(run_dir / "candidates.jsonl", record)
            candidate_rows.extend(existing_records)
        return RequestState(logical_id, result, parsed, existing_records, previous)

    if reuse_state is None:
        result = run_cached_gemini_request(
            semantic_interface=interface_id,
            prompt=prompt,
            images=images,
            repeat_index=repeat_index,
            online=online,
            budget=budget,
        )
        reused_from = None
        prompt_path = save_prompt(run_dir, logical_id, prompt)
        input_paths = save_representation_inputs(run_dir, data, bundle)
        raw_path = save_raw_response(run_dir, logical_id, result)
    else:
        # D3 direct_xyz consumes the exact in-memory D1 result and performs no cache/API call.
        expected_key = cache_key(
            semantic_interface=interface_id,
            prompt=prompt,
            images=images,
            repeat_index=repeat_index,
        )
        if reuse_state.result.cache_key and reuse_state.result.cache_key != expected_key:
            raise AssertionError("D3 direct_xyz does not exactly match the reusable D1 request")
        result = replace(
            reuse_state.result,
            status="reused_d1_direct",
            current_lookup_latency_s=0.0,
            transport_attempts=0,
        )
        reused_from = reuse_state.logical_id
        prompt_path = reuse_state.call_row.get("prompt_path")
        input_paths = list(reuse_state.call_row.get("input_paths") or [])
        raw_path = reuse_state.call_row.get("raw_response_path")

    parsed, records = evaluate_call(
        data=data,
        stage=stage,
        representation_id=representation_id,
        interface_id=interface_id,
        repeat_index=repeat_index,
        call_result=result,
    )
    apply_reference_metrics(records, references)
    for record in records:
        record["logical_id"] = logical_id

    call_row = make_call_row(
        logical_id=logical_id,
        data=data,
        stage=stage,
        representation_id=representation_id,
        interface_id=interface_id,
        repeat_index=repeat_index,
        result=result,
        parsed=parsed,
        raw_response_path=raw_path,
        prompt_path=prompt_path,
        input_paths=input_paths,
        amortized_anchor_latency_s=amortized_anchor_latency_s,
        reused_from_logical_id=reused_from,
    )
    call_rows.append(call_row)
    append_jsonl(run_dir / "calls.jsonl", call_row)
    for record in records:
        candidate_rows.append(record)
        append_jsonl(run_dir / "candidates.jsonl", record)
    completed_ids.add(logical_id)
    return RequestState(logical_id, result, parsed, records, call_row)

def log_representation_unavailable(
    *,
    data: SceneData,
    representation_id: str,
    repeat_index: int,
    reason: str,
    run_dir: Path,
    call_rows: list[dict[str, Any]],
    completed_ids: set[str],
    amortized_anchor_latency_s: float,
) -> None:
    logical_id = logical_request_id(
        "D1",
        data.spec.scene_id,
        data.spec.task_id,
        representation_id,
        "direct_xyz",
        repeat_index,
    )
    if logical_id in completed_ids:
        return
    result = synthetic_call_result("representation_unavailable", reason)
    call_row = make_call_row(
        logical_id=logical_id,
        data=data,
        stage="D1",
        representation_id=representation_id,
        interface_id="direct_xyz",
        repeat_index=repeat_index,
        result=result,
        parsed=None,
        raw_response_path=None,
        prompt_path=None,
        input_paths=[],
        amortized_anchor_latency_s=amortized_anchor_latency_s,
    )
    call_rows.append(call_row)
    append_jsonl(run_dir / "calls.jsonl", call_row)
    completed_ids.add(logical_id)


In [ ]:
def log_unavailable_request(
    *,
    data: SceneData,
    stage: str,
    representation_id: str,
    interface_id: str,
    repeat_index: int,
    reason: str,
    run_dir: Path,
    call_rows: list[dict[str, Any]],
    completed_ids: set[str],
    amortized_anchor_latency_s: float = 0.0,
) -> None:
    logical_id = logical_request_id(
        stage,
        data.spec.scene_id,
        data.spec.task_id,
        representation_id,
        interface_id,
        repeat_index,
    )
    if logical_id in completed_ids:
        return
    result = synthetic_call_result("representation_unavailable", reason)
    call_row = make_call_row(
        logical_id=logical_id,
        data=data,
        stage=stage,
        representation_id=representation_id,
        interface_id=interface_id,
        repeat_index=repeat_index,
        result=result,
        parsed=None,
        raw_response_path=None,
        prompt_path=None,
        input_paths=[],
        amortized_anchor_latency_s=amortized_anchor_latency_s,
    )
    call_rows.append(call_row)
    append_jsonl(run_dir / "calls.jsonl", call_row)
    completed_ids.add(logical_id)

def save_summary_chart(
    run_dir: Path,
    d1_recommendation_rows: Sequence[dict[str, Any]],
) -> None:
    labels = [str(row["representation_id"]).split("_", 1)[0].upper() for row in d1_recommendation_rows]
    parse_rates = [float(row["parse_compliance_rate"]) for row in d1_recommendation_rows]
    valid_rates = [float(row["rank1_valid_depth_rate"]) for row in d1_recommendation_rows]
    z_errors = [
        float(row["rank1_mean_sensor_z_abs_difference_m"])
        if row["rank1_mean_sensor_z_abs_difference_m"] is not None
        else np.nan
        for row in d1_recommendation_rows
    ]
    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=130)
    axes[0].bar(x - 0.18, parse_rates, 0.36, label="parse compliance")
    axes[0].bar(x + 0.18, valid_rates, 0.36, label="rank-1 valid depth")
    axes[0].axhline(0.90, color="black", linestyle="--", linewidth=1)
    axes[0].set_ylim(0, 1.05)
    axes[0].set_xticks(x, labels)
    axes[0].set_ylabel("rate")
    axes[0].legend()
    axes[0].set_title("D1 eligibility")
    axes[1].bar(x, z_errors)
    axes[1].set_xticks(x, labels)
    axes[1].set_ylabel("absolute difference (m)")
    axes[1].set_title("Rank-1 VLM Z vs sensor median (consistency only)")
    fig.tight_layout()
    chart_path = run_dir / "charts" / "d1_overview.png"
    chart_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(chart_path)
    plt.close(fig)

@dataclass
class Phase1ARunResult:
    run_dir: Path
    selected_d1_for_d3: str | None
    budget: CallBudget
    attempts_before_run: int
    call_rows: list[dict[str, Any]]
    candidate_rows: list[dict[str, Any]]
    summary_rows: list[dict[str, Any]]
    wilcoxon_rows: list[dict[str, Any]]

def run_phase1a(
    *,
    profile: str = PROFILE,
    online: bool = RUN_ONLINE,
    max_new_calls: int = MAX_NEW_CALLS,
    run_id: str | None = RUN_ID_OVERRIDE,
    d1_for_d3: str = D1_FOR_D3,
) -> Phase1ARunResult:
    if profile not in PROFILE_CONFIGS:
        raise ValueError(f"unknown profile: {profile}")
    expected_cap = int(PROFILE_CONFIGS[profile]["expected_http_cap"])
    if online and max_new_calls <= 0:
        raise ValueError("Online mode requires an explicit positive PHASE1A_MAX_NEW_CALLS")
    resolved_run_id = run_id or new_run_id()
    output_root_resolved = OUTPUT_ROOT.resolve()
    run_dir = (OUTPUT_ROOT / resolved_run_id).resolve()
    if run_dir == output_root_resolved or output_root_resolved not in run_dir.parents:
        raise ValueError("PHASE1A_RUN_ID must name a child of the Phase 1A output directory")
    run_dir.mkdir(parents=True, exist_ok=True)
    calls_path = run_dir / "calls.jsonl"
    candidates_path = run_dir / "candidates.jsonl"
    calls_path.touch(exist_ok=True)
    candidates_path.touch(exist_ok=True)
    raw_call_rows = read_jsonl(calls_path)
    raw_candidate_rows = read_jsonl(candidates_path)

    specs = profile_specs(profile)
    repeats = int(PROFILE_CONFIGS[profile]["repeats"])
    reference_bytes = (REFERENCE_GRASPS_PATH.read_bytes() if REFERENCE_GRASPS_PATH.exists() else None)
    reference_sha256 = hashlib.sha256(reference_bytes).hexdigest() if reference_bytes is not None else None
    references = (
        load_reference_grasps(REFERENCE_GRASPS_PATH, contents=reference_bytes)
        if reference_bytes is not None
        else {}
    )
    fingerprint_payload = {
        "experiment_schema_version": EXPERIMENT_SCHEMA_VERSION,
        "representation_version": REPRESENTATION_VERSION,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "prompt_version": PROMPT_VERSION,
        "anchor_prompt_version": ANCHOR_PROMPT_VERSION,
        "model": MODEL_NAME,
        "api_version": API_VERSION,
        "generation_config": GENERATION_CONFIG,
        "seed": SEED,
        "depth_max_m": DEPTH_MAX_M,
        "profile": profile,
        "repeats": repeats,
        "d1_for_d3_override": d1_for_d3,
        "manifest": [manifest_record(spec) for spec in specs],
        "reference_grasps_sha256": reference_sha256,
    }
    run_fingerprint = hashlib.sha256(canonical_json_bytes(fingerprint_payload)).hexdigest()
    config_path = run_dir / "config.json"
    if config_path.exists():
        existing_config = json.loads(
            config_path.read_text(encoding="utf-8"),
            parse_constant=reject_nonstandard_constant,
        )
        if existing_config.get("run_fingerprint") != run_fingerprint:
            raise RuntimeError(
                "Run configuration changed; choose a new PHASE1A_RUN_ID instead of mixing experiments"
            )

    requested_budget_limit = min(max_new_calls, expected_cap) if online else 0
    ledger_path = run_dir / "http_attempt_ledger.json"
    ledger = (
        json.loads(ledger_path.read_text(encoding="utf-8"), parse_constant=reject_nonstandard_constant)
        if ledger_path.exists()
        else {}
    )
    attempts_from_log = sum(int(row.get("transport_attempts") or 0) for row in raw_call_rows)
    attempts_used = max(int(ledger.get("used_http_transport_attempts", 0)), attempts_from_log)
    attempts_before_run = attempts_used
    stored_limit = int(ledger.get("hard_http_call_limit", 0))
    budget_limit = requested_budget_limit if online else stored_limit
    if attempts_used > expected_cap:
        raise RuntimeError("Persisted HTTP attempts exceed the immutable profile cap")
    budget = CallBudget(budget_limit, attempts_used, ledger_path)
    budget.persist()

    call_rows = latest_call_attempts(raw_call_rows)
    candidate_rows = candidates_for_latest_calls(raw_candidate_rows, call_rows)
    terminal_statuses = {
        "ok", "response_error", "cache_write_error", "reused_d1_direct",
        "anchor_ok", "anchor_parse_failed",
    }
    completed_ids = {
        str(row["logical_id"])
        for row in call_rows
        if row.get("call_status") in terminal_statuses
    }

    dump_json(run_dir / "manifest.json", [manifest_record(spec) for spec in specs])
    dump_json(
        run_dir / "config.json",
        {
            "phase": "1A",
            "profile": profile,
            "online_for_this_execution": online,
            "model": MODEL_NAME,
            "api_version": API_VERSION,
            "seed": SEED,
            "depth_max_m": DEPTH_MAX_M,
            "repeats": repeats,
            "expected_no_retry_http_cap": expected_cap,
            "hard_new_http_call_limit": budget_limit,
            "attempts_before_this_execution": attempts_before_run,
            "d1_for_d3_override": d1_for_d3,
            "prompt_version": PROMPT_VERSION,
            "anchor_prompt_version": ANCHOR_PROMPT_VERSION,
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "experiment_schema_version": EXPERIMENT_SCHEMA_VERSION,
            "representation_version": REPRESENTATION_VERSION,
            "run_fingerprint": run_fingerprint,
            "run_fingerprint_payload": fingerprint_payload,
            "reference_grasps_path": str(REFERENCE_GRASPS_PATH),
            "reference_available_scene_tasks": len(references),
            "legacy_depth_preview_diagnostic": RUN_LEGACY_DIAGNOSTIC,
            "render_3d_requested": RENDER_3D,
            "api_key_persisted": False,
        },
    )

    scene_data = {
        (spec.scene_id, spec.task_id): load_scene(spec)
        for spec in specs
    }
    if RUN_LEGACY_DIAGNOSTIC:
        for data in scene_data.values():
            diagnostic_bundle = legacy_diagnostic(data)
            if diagnostic_bundle is not None:
                save_representation_inputs(run_dir, data, diagnostic_bundle)
    anchors: dict[tuple[str, str], AnchorOutcome] = {}
    for key, data in scene_data.items():
        anchors[key] = get_scene_anchors(
            data=data,
            run_dir=run_dir,
            online=online,
            budget=budget,
            call_rows=call_rows,
            completed_ids=completed_ids,
        )

    d1_states: dict[tuple[str, str, str, int], RequestState] = {}
    for key, data in scene_data.items():
        anchor_outcome = anchors[key]
        for representation_id in REPRESENTATION_IDS:
            try:
                bundle = build_representation(
                    data,
                    representation_id,
                    anchors=anchor_outcome.anchors if representation_id == "r3_local_depth_text" else None,
                )
            except Exception as exc:
                reason = f"{type(exc).__name__}: {exc}"
                for repeat_index in range(repeats):
                    log_unavailable_request(
                        data=data,
                        stage="D1",
                        representation_id=representation_id,
                        interface_id="direct_xyz",
                        repeat_index=repeat_index,
                        reason=reason,
                        run_dir=run_dir,
                        call_rows=call_rows,
                        completed_ids=completed_ids,
                        amortized_anchor_latency_s=(
                            anchor_outcome.original_latency_s / repeats
                            if representation_id == "r3_local_depth_text"
                            else 0.0
                        ),
                    )
                continue
            for repeat_index in range(repeats):
                state = execute_benchmark_request(
                    data=data,
                    bundle=bundle,
                    stage="D1",
                    interface_id="direct_xyz",
                    repeat_index=repeat_index,
                    run_dir=run_dir,
                    online=online,
                    budget=budget,
                    call_rows=call_rows,
                    candidate_rows=candidate_rows,
                    completed_ids=completed_ids,
                    references=references,
                    amortized_anchor_latency_s=(
                        anchor_outcome.original_latency_s / repeats
                        if representation_id == "r3_local_depth_text"
                        else 0.0
                    ),
                )
                d1_states[
                    (data.spec.scene_id, data.spec.task_id, representation_id, repeat_index)
                ] = state

    call_rows[:] = latest_call_attempts(call_rows)
    candidate_rows[:] = candidates_for_latest_calls(candidate_rows, call_rows)
    selected_d1, recommendation_rows = recommend_d1_method(
        call_rows,
        candidate_rows,
        override=d1_for_d3,
    )
    selection_lock_path = run_dir / "d3_input_selection.json"
    if selection_lock_path.exists():
        locked_selection = json.loads(
            selection_lock_path.read_text(encoding="utf-8"),
            parse_constant=reject_nonstandard_constant,
        ).get("selected_d1_for_d3")
        if locked_selection != selected_d1:
            raise RuntimeError(
                "Automatic D1 selection changed during resume; choose a new run ID"
            )
    elif selected_d1 is not None:
        dump_json(
            selection_lock_path,
            {
                "run_fingerprint": run_fingerprint,
                "selected_d1_for_d3": selected_d1,
            },
        )
    dump_json(
        run_dir / "d1_recommendation.json",
        {
            "selected_d1_for_d3": selected_d1,
            "minimum_parse_compliance_rate": 0.90,
            "minimum_rank1_valid_depth_rate": 0.90,
            "tie_break_order": [
                "rank1_mean_sensor_z_abs_difference_m",
                "rank1_mean_sensor_position_difference_m",
                "mean_effective_cold_latency_s",
            ],
            "rows": recommendation_rows,
        },
    )
    write_csv(run_dir / "d1_recommendation.csv", recommendation_rows)

    if selected_d1 is not None:
        for key, data in scene_data.items():
            anchor_outcome = anchors[key]
            try:
                selected_bundle = build_representation(
                    data,
                    selected_d1,
                    anchors=anchor_outcome.anchors if selected_d1 == "r3_local_depth_text" else None,
                )
            except Exception as exc:
                for interface_id in INTERFACE_IDS:
                    for repeat_index in range(repeats):
                        log_unavailable_request(
                            data=data,
                            stage="D3",
                            representation_id=selected_d1,
                            interface_id=interface_id,
                            repeat_index=repeat_index,
                            reason=f"{type(exc).__name__}: {exc}",
                            run_dir=run_dir,
                            call_rows=call_rows,
                            completed_ids=completed_ids,
                            amortized_anchor_latency_s=(
                                anchor_outcome.original_latency_s / repeats
                                if selected_d1 == "r3_local_depth_text"
                                else 0.0
                            ),
                        )
                continue
            for repeat_index in range(repeats):
                d1_state = d1_states.get(
                    (data.spec.scene_id, data.spec.task_id, selected_d1, repeat_index)
                )
                if d1_state is None:
                    log_unavailable_request(
                        data=data,
                        stage="D3",
                        representation_id=selected_d1,
                        interface_id="direct_xyz",
                        repeat_index=repeat_index,
                        reason="matching D1 direct result unavailable for reuse",
                        run_dir=run_dir,
                        call_rows=call_rows,
                        completed_ids=completed_ids,
                    )
                else:
                    execute_benchmark_request(
                        data=data,
                        bundle=selected_bundle,
                        stage="D3",
                        interface_id="direct_xyz",
                        repeat_index=repeat_index,
                        run_dir=run_dir,
                        online=online,
                        budget=budget,
                        call_rows=call_rows,
                        candidate_rows=candidate_rows,
                        completed_ids=completed_ids,
                        references=references,
                        amortized_anchor_latency_s=(
                            anchor_outcome.original_latency_s / repeats
                            if selected_d1 == "r3_local_depth_text"
                            else 0.0
                        ),
                        reuse_state=d1_state,
                    )
                for interface_id in ("pixel_depth", "depth_band", "multi_pixel"):
                    execute_benchmark_request(
                        data=data,
                        bundle=selected_bundle,
                        stage="D3",
                        interface_id=interface_id,
                        repeat_index=repeat_index,
                        run_dir=run_dir,
                        online=online,
                        budget=budget,
                        call_rows=call_rows,
                        candidate_rows=candidate_rows,
                        completed_ids=completed_ids,
                        references=references,
                        amortized_anchor_latency_s=(
                            anchor_outcome.original_latency_s / repeats
                            if selected_d1 == "r3_local_depth_text"
                            else 0.0
                        ),
                    )

    call_rows[:] = latest_call_attempts(call_rows)
    candidate_rows[:] = candidates_for_latest_calls(candidate_rows, call_rows)
    summary_rows = summarize_rank1(call_rows, candidate_rows)
    candidate_summary_rows = summarize_all_candidate_ranks(call_rows, candidate_rows)
    wilcoxon_rows = paired_wilcoxon_rows(call_rows, candidate_rows)
    write_csv(run_dir / "calls.csv", call_rows)
    write_csv(run_dir / "candidates.csv", candidate_rows)
    write_csv(run_dir / "summary.csv", summary_rows)
    write_csv(run_dir / "summary_rank1.csv", summary_rows)
    write_csv(run_dir / "summary_all_candidates.csv", candidate_summary_rows)
    write_csv(run_dir / "paired_wilcoxon.csv", wilcoxon_rows)
    save_summary_chart(run_dir, recommendation_rows)
    status_record = {
        "online_for_this_execution": online,
        "selected_d1_for_d3": selected_d1,
        "new_http_transport_attempts_this_execution": (
            budget.new_http_calls - attempts_before_run
        ),
        "cumulative_http_transport_attempts": budget.new_http_calls,
        "hard_new_http_call_limit": budget.max_new_http_calls,
        "call_rows": len(call_rows),
        "candidate_rows": len(candidate_rows),
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    dump_json(run_dir / "run_status.json", status_record)
    append_jsonl(run_dir / "execution_history.jsonl", status_record)
    return Phase1ARunResult(
        run_dir,
        selected_d1,
        budget,
        attempts_before_run,
        call_rows,
        candidate_rows,
        summary_rows,
        wilcoxon_rows,
    )


## Phase 1B — interactive 3D grasp inspection

The point cloud is explanatory only: it is not fed back to Gemini, used for collision checking, or treated as ground truth. Formal grippers use the stored `sensor_corrected_pose_4x4` without rebuilding the VLM orientation. The default view combines a dense 0.25 m grasp neighborhood with a translucent deterministic whole-scene sample.

In [ ]:
import plotly
import plotly.graph_objects as go

from vg_pipeline.grasp_results import FINGER_LENGTH_METERS, GRIPPER_DEPTH_METERS

PHASE1B_VIZ_SCHEMA_VERSION = "phase1b-viz-v1"
PHASE1B_FULL_MAX_POINTS = 10_000
PHASE1B_LOCAL_MAX_POINTS = 15_000
PHASE1B_LOCAL_RADIUS_M = 0.25
PHASE1B_AXIS_LENGTH_M = 0.05
PHASE1B_WIDTH_MIN_M = 0.02
PHASE1B_WIDTH_MAX_M = 0.08

@dataclass
class Phase1BSelection:
    records: list[dict[str, Any]]
    excluded: list[dict[str, Any]]
    call_statuses: list[dict[str, Any]]
    source_stage: str
    fallback_reason: str | None

@dataclass
class Phase1AVisualizationResult:
    figure: Any | None
    html_path: Path | None
    manifest_path: Path
    status: str
    entry: dict[str, Any]

def _phase1b_valid_pose(value: Any) -> np.ndarray | None:
    try:
        pose = np.asarray(value, dtype=np.float64)
    except (TypeError, ValueError):
        return None
    if pose.shape != (4, 4) or not np.all(np.isfinite(pose)):
        return None
    if not np.allclose(pose[3], [0.0, 0.0, 0.0, 1.0], atol=1e-6):
        return None
    rotation = pose[:3, :3]
    if not np.allclose(rotation.T @ rotation, np.eye(3), atol=1e-5):
        return None
    if not math.isclose(float(np.linalg.det(rotation)), 1.0, abs_tol=1e-5):
        return None
    return pose

def _phase1b_sample_indices(
    count: int,
    limit: int,
    *,
    scene_id: str,
    layer_id: str,
) -> np.ndarray:
    if count <= 0 or limit <= 0:
        return np.empty(0, dtype=np.int64)
    if count <= limit:
        return np.arange(count, dtype=np.int64)
    rng = np.random.default_rng(scene_seed(scene_id, layer_id))
    return np.sort(rng.choice(count, size=limit, replace=False))

def phase1b_pointcloud_layers(
    data: SceneData,
    corrected_centers: Sequence[Sequence[float]],
    *,
    full_max_points: int = PHASE1B_FULL_MAX_POINTS,
    local_max_points: int = PHASE1B_LOCAL_MAX_POINTS,
    local_radius_m: float = PHASE1B_LOCAL_RADIUS_M,
) -> dict[str, Any]:
    if full_max_points < 0 or local_max_points < 0 or local_radius_m <= 0:
        raise ValueError("Point-cloud limits must be non-negative and radius must be positive")
    points_organized = organized_pointcloud(data.depth, data.K)
    valid = (
        np.isfinite(data.depth)
        & (data.depth > 0)
        & (data.depth <= DEPTH_MAX_M)
        & np.all(np.isfinite(points_organized), axis=-1)
    )
    points = np.asarray(points_organized[valid], dtype=np.float32)
    colors = np.asarray(data.rgb[valid], dtype=np.uint8)

    full_indices = _phase1b_sample_indices(
        len(points),
        full_max_points,
        scene_id=data.spec.scene_id,
        layer_id="phase1b_full_pointcloud",
    )
    full_points = points[full_indices]
    full_colors = colors[full_indices]

    local_mask = np.zeros(len(points), dtype=bool)
    usable_centers: list[np.ndarray] = []
    radius_sq = float(local_radius_m) ** 2
    points64 = points.astype(np.float64, copy=False)
    for center_value in corrected_centers:
        try:
            center = np.asarray(center_value, dtype=np.float64)
        except (TypeError, ValueError):
            continue
        if center.shape != (3,) or not np.all(np.isfinite(center)):
            continue
        usable_centers.append(center)
        delta = points64 - center
        local_mask |= np.einsum("ij,ij->i", delta, delta) <= radius_sq

    local_source_points = points[local_mask]
    local_source_colors = colors[local_mask]
    local_indices = _phase1b_sample_indices(
        len(local_source_points),
        local_max_points,
        scene_id=data.spec.scene_id,
        layer_id="phase1b_local_pointcloud",
    )
    return {
        "full_points": full_points,
        "full_colors": full_colors,
        "local_points": local_source_points[local_indices],
        "local_colors": local_source_colors[local_indices],
        "valid_source_count": int(len(points)),
        "local_source_count": int(len(local_source_points)),
        "corrected_center_count": int(len(usable_centers)),
        "local_radius_m": float(local_radius_m),
    }

def _phase1b_latest_call_map(
    call_rows: Sequence[dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    for row in call_rows:
        logical_id = str(row.get("logical_id") or "")
        if logical_id:
            latest[logical_id] = row
    return latest

def _phase1b_exclusion_record(
    row: dict[str, Any],
    reasons: Sequence[str],
) -> dict[str, Any]:
    return {
        "logical_id": row.get("logical_id"),
        "cache_key": row.get("cache_key"),
        "stage": row.get("stage"),
        "representation_id": row.get("representation_id"),
        "interface_id": row.get("interface_id"),
        "rank": row.get("rank"),
        "candidate_index": row.get("candidate_index"),
        "reasons": list(reasons),
    }

def _phase1b_candidate_reasons(
    row: dict[str, Any],
    latest_calls: dict[str, dict[str, Any]],
    *,
    include_partial: bool,
) -> list[str]:
    reasons: list[str] = []
    logical_id = str(row.get("logical_id") or "")
    latest_call = latest_calls.get(logical_id)
    if latest_call is None:
        reasons.append("missing_latest_call")
    elif str(row.get("cache_key") or "") != str(latest_call.get("cache_key") or ""):
        reasons.append("stale_cache_key")

    flags = row.get("validation_flags")
    if not isinstance(flags, dict):
        reasons.append("missing_validation_flags")
        flags = {}
    partial = bool(row.get("partial_recovery_selected"))
    if partial and not include_partial:
        reasons.append("partial_recovery_excluded")
    required_flags = (
        "candidate_schema_valid",
        "pixel_in_bounds",
        "sensor_depth_valid",
        "orientation_valid",
        "corrected_pose_valid",
    )
    for flag in required_flags:
        if flags.get(flag) is not True:
            reasons.append("validation_failed:" + flag)
    if flags.get("request_parse_compliant") is not True and not (
        include_partial and partial
    ):
        reasons.append("validation_failed:request_parse_compliant")
    if _phase1b_valid_pose(row.get("sensor_corrected_pose_4x4")) is None:
        reasons.append("invalid_sensor_corrected_pose_4x4")
    return reasons

def select_phase1a_visualization_records(
    result: Phase1ARunResult,
    scene_id: str,
    task_id: str,
    *,
    repeat_index: int = 0,
    include_partial: bool = False,
) -> Phase1BSelection:
    latest_calls = _phase1b_latest_call_map(result.call_rows)
    call_statuses = [
        {
            "logical_id": row.get("logical_id"),
            "stage": row.get("stage"),
            "representation_id": row.get("representation_id"),
            "interface_id": row.get("interface_id"),
            "repeat_index": row.get("repeat_index"),
            "call_status": row.get("call_status"),
            "cache_key": row.get("cache_key"),
        }
        for row in latest_calls.values()
        if row.get("scene_id") == scene_id
        and row.get("task_id") == task_id
        and (
            row.get("repeat_index") in (None, repeat_index)
            or row.get("stage") == "R3_anchor"
        )
    ]
    call_statuses.sort(
        key=lambda row: (
            str(row.get("stage") or ""),
            str(row.get("interface_id") or ""),
            str(row.get("logical_id") or ""),
        )
    )

    excluded: list[dict[str, Any]] = []
    valid_d3: list[dict[str, Any]] = []
    relevant_d3 = [
        row for row in result.candidate_rows
        if row.get("scene_id") == scene_id
        and row.get("task_id") == task_id
        and int(row.get("repeat_index", -1)) == int(repeat_index)
        and row.get("stage") == "D3"
    ]
    for row in relevant_d3:
        reasons = _phase1b_candidate_reasons(
            row, latest_calls, include_partial=include_partial
        )
        if reasons:
            excluded.append(_phase1b_exclusion_record(row, reasons))
        else:
            valid_d3.append(row)

    limited_d3: list[dict[str, Any]] = []
    for interface_id in INTERFACE_IDS:
        interface_rows = sorted(
            (row for row in valid_d3 if row.get("interface_id") == interface_id),
            key=lambda row: (
                int(row.get("rank") or 999),
                int(row.get("candidate_index")) if row.get("candidate_index") is not None else 999,
                str(row.get("logical_id") or ""),
            ),
        )
        limited_d3.extend(interface_rows[:NUM_CANDIDATES])
        for extra in interface_rows[NUM_CANDIDATES:]:
            excluded.append(
                _phase1b_exclusion_record(extra, ["more_than_three_valid_candidates"])
            )
    if limited_d3:
        return Phase1BSelection(
            records=limited_d3,
            excluded=excluded,
            call_statuses=call_statuses,
            source_stage="D3",
            fallback_reason=None,
        )

    fallback_reason = "no_valid_formal_d3_corrected_pose"
    fallback_records: list[dict[str, Any]] = []
    relevant_d1 = [
        row for row in result.candidate_rows
        if row.get("scene_id") == scene_id
        and row.get("task_id") == task_id
        and int(row.get("repeat_index", -1)) == int(repeat_index)
        and row.get("stage") == "D1"
        and row.get("interface_id") == "direct_xyz"
        and row.get("representation_id") == result.selected_d1_for_d3
        and int(row.get("rank") or -1) == 1
    ]
    for row in relevant_d1:
        reasons = _phase1b_candidate_reasons(
            row, latest_calls, include_partial=False
        )
        if reasons:
            excluded.append(_phase1b_exclusion_record(row, reasons))
        else:
            fallback_records.append(row)
    fallback_records.sort(key=lambda row: str(row.get("logical_id") or ""))
    return Phase1BSelection(
        records=fallback_records[:1],
        excluded=excluded,
        call_statuses=call_statuses,
        source_stage="D1_fallback" if fallback_records else "none",
        fallback_reason=fallback_reason,
    )

def _phase1b_width(record: dict[str, Any]) -> tuple[float | None, float, bool]:
    raw = finite_float(record.get("estimated_object_width_along_close_m"))
    if raw is None or raw <= 0:
        return raw, PHASE1B_WIDTH_MAX_M, True
    display_width = float(np.clip(raw, PHASE1B_WIDTH_MIN_M, PHASE1B_WIDTH_MAX_M))
    return raw, display_width, not math.isclose(raw, display_width, abs_tol=1e-12)

def _phase1b_segment_coordinates(
    segments: Sequence[tuple[np.ndarray, np.ndarray]],
) -> tuple[list[float | None], list[float | None], list[float | None]]:
    x: list[float | None] = []
    y: list[float | None] = []
    z: list[float | None] = []
    for start, end in segments:
        x.extend((float(start[0]), float(end[0]), None))
        y.extend((float(start[1]), float(end[1]), None))
        z.extend((float(start[2]), float(end[2]), None))
    return x, y, z

def phase1b_gripper_wireframe(
    pose_4x4: Sequence[Sequence[float]],
    display_width_m: float,
) -> dict[str, Any]:
    pose = _phase1b_valid_pose(pose_4x4)
    if pose is None:
        raise ValueError("wireframe requires a valid corrected 4x4 pose")
    width = float(display_width_m)
    if not PHASE1B_WIDTH_MIN_M <= width <= PHASE1B_WIDTH_MAX_M:
        raise ValueError("wireframe width must already be clipped to the display range")
    center = pose[:3, 3]
    closing = pose[:3, 0]
    lateral = pose[:3, 1]
    approach = pose[:3, 2]
    half_width = width / 2.0
    half_finger_thickness = 0.006
    finger_back = center - approach * FINGER_LENGTH_METERS
    wrist = center - approach * GRIPPER_DEPTH_METERS
    segments: list[tuple[np.ndarray, np.ndarray]] = []
    for closing_sign in (-1.0, 1.0):
        for lateral_sign in (-1.0, 1.0):
            tip = (
                center
                + closing_sign * half_width * closing
                + lateral_sign * half_finger_thickness * lateral
            )
            finger_base = (
                finger_back
                + closing_sign * half_width * closing
                + lateral_sign * half_finger_thickness * lateral
            )
            segments.append((tip, finger_base))
        tip_low = center + closing_sign * half_width * closing - half_finger_thickness * lateral
        tip_high = center + closing_sign * half_width * closing + half_finger_thickness * lateral
        base_low = finger_back + closing_sign * half_width * closing - half_finger_thickness * lateral
        base_high = finger_back + closing_sign * half_width * closing + half_finger_thickness * lateral
        segments.extend(((tip_low, tip_high), (base_low, base_high)))
    for lateral_sign in (-1.0, 1.0):
        left = finger_back - half_width * closing + lateral_sign * half_finger_thickness * lateral
        right = finger_back + half_width * closing + lateral_sign * half_finger_thickness * lateral
        segments.append((left, right))
    segments.append((finger_back, wrist))
    x, y, z = _phase1b_segment_coordinates(segments)
    return {
        "x": x,
        "y": y,
        "z": z,
        "segments": segments,
        "center": center,
        "axis_endpoints": {
            "closing": center + PHASE1B_AXIS_LENGTH_M * closing,
            "lateral": center + PHASE1B_AXIS_LENGTH_M * lateral,
            "approach": center + PHASE1B_AXIS_LENGTH_M * approach,
        },
    }


In [ ]:
def _phase1b_color_strings(colors: np.ndarray) -> list[str]:
    return [
        "rgb({},{},{})".format(int(color[0]), int(color[1]), int(color[2]))
        for color in np.asarray(colors)
    ]

def _phase1b_format_number(value: Any, digits: int = 4) -> str:
    number = finite_float(value)
    return "n/a" if number is None else f"{number:.{digits}f}"

def _phase1b_hover_text(
    record: dict[str, Any],
    corrected_position: np.ndarray,
    raw_width: float | None,
    display_width: float,
    width_clipped: bool,
) -> str:
    pixel = record.get("pixel_uv")
    pixel_text = (
        f"({int(pixel[0])}, {int(pixel[1])})"
        if isinstance(pixel, (list, tuple)) and len(pixel) == 2
        else "n/a"
    )
    selected_label = (
        "server_selected"
        if record.get("server_selected")
        else (
            "partial_recovery_selected"
            if record.get("partial_recovery_selected")
            else "candidate"
        )
    )
    return "<br>".join(
        (
            f"<b>{record.get('interface_id')} rank {record.get('rank')}</b>",
            f"source: {record.get('stage')} / {selected_label}",
            f"confidence: {_phase1b_format_number(record.get('confidence'), 3)}",
            f"pixel (u,v): {pixel_text}",
            "corrected XYZ (m): "
            f"({corrected_position[0]:.4f}, {corrected_position[1]:.4f}, "
            f"{corrected_position[2]:.4f})",
            "5x5 depth median / MAD (m): "
            f"{_phase1b_format_number(record.get('sensor_depth_median_m'), 4)} / "
            f"{_phase1b_format_number(record.get('sensor_depth_mad_m'), 4)}",
            f"valid depth samples: {record.get('valid_depth_count', 'n/a')}",
            "width raw / display (m): "
            f"{_phase1b_format_number(raw_width, 4)} / {display_width:.4f}",
            f"width feasible [0.02,0.08] m: {bool(record.get('width_feasible'))}",
            f"display width clipped: {width_clipped}",
            "sensor consistency distance (m): "
            f"{_phase1b_format_number(record.get('sensor_position_difference_m'), 4)}",
        )
    )

def build_phase1a_grasp_figure(
    data: SceneData,
    records: Sequence[dict[str, Any]],
    *,
    source_stage: str = "D3",
) -> tuple[go.Figure, dict[str, Any]]:
    if not records:
        raise ValueError("At least one valid corrected-pose record is required")
    validated: list[tuple[dict[str, Any], np.ndarray]] = []
    for record in records:
        pose = _phase1b_valid_pose(record.get("sensor_corrected_pose_4x4"))
        if pose is None:
            raise ValueError("build_phase1a_grasp_figure received an invalid corrected pose")
        validated.append((record, pose))

    formal_multi_selected = any(
        record.get("interface_id") == "multi_pixel"
        and bool(record.get("server_selected"))
        and not bool(record.get("partial_recovery_selected"))
        for record, _ in validated
    )
    interfaces_present = [
        interface_id
        for interface_id in INTERFACE_IDS
        if any(record.get("interface_id") == interface_id for record, _ in validated)
    ]
    if formal_multi_selected:
        default_interface = "multi_pixel"
    elif "direct_xyz" in interfaces_present:
        default_interface = "direct_xyz"
    else:
        default_interface = interfaces_present[0]

    centers = [pose[:3, 3] for _, pose in validated]
    point_layers = phase1b_pointcloud_layers(data, centers)
    figure = go.Figure()
    trace_registry: list[dict[str, Any]] = []

    def add_trace(trace: go.Scatter3d, meta: dict[str, Any]) -> int:
        trace.meta = dict(meta)
        figure.add_trace(trace)
        index = len(figure.data) - 1
        trace_registry.append({"trace_index": index, **meta})
        return index

    full_index = add_trace(
        go.Scatter3d(
            x=point_layers["full_points"][:, 0],
            y=point_layers["full_points"][:, 1],
            z=point_layers["full_points"][:, 2],
            mode="markers",
            name="全场景点云",
            marker={
                "size": 1.7,
                "color": _phase1b_color_strings(point_layers["full_colors"]),
                "opacity": 0.18,
            },
            hoverinfo="skip",
            showlegend=True,
            visible=True,
        ),
        {"kind": "cloud_full"},
    )
    local_index = add_trace(
        go.Scatter3d(
            x=point_layers["local_points"][:, 0],
            y=point_layers["local_points"][:, 1],
            z=point_layers["local_points"][:, 2],
            mode="markers",
            name="抓取邻域点云",
            marker={
                "size": 2.7,
                "color": _phase1b_color_strings(point_layers["local_colors"]),
                "opacity": 0.82,
            },
            hoverinfo="skip",
            showlegend=True,
            visible=True,
        ),
        {"kind": "cloud_local"},
    )

    interface_colors = {
        "direct_xyz": "#D81B60",
        "pixel_depth": "#00ACC1",
        "depth_band": "#FB8C00",
        "multi_pixel": "#8E24AA",
    }
    axis_specs = (
        ("closing", "#E53935"),
        ("lateral", "#43A047"),
        ("approach", "#1E88E5"),
    )
    candidate_trace_indices: list[int] = []
    candidate_trace_interfaces: list[str] = []
    included_candidates: list[dict[str, Any]] = []

    for record, pose in sorted(
        validated,
        key=lambda item: (
            INTERFACE_IDS.index(str(item[0].get("interface_id"))),
            int(item[0].get("rank") or 999),
            (
                int(item[0].get("candidate_index"))
                if item[0].get("candidate_index") is not None
                else 999
            ),
        ),
    ):
        interface_id = str(record.get("interface_id"))
        rank = int(record.get("rank") or 0)
        candidate_index = int(record.get("candidate_index") or 0)
        logical_id = str(record.get("logical_id") or "")
        candidate_id = f"{interface_id}:rank{rank}:candidate{candidate_index}"
        legend_group = f"{interface_id}|{logical_id}|{rank}|{candidate_index}"
        raw_width, display_width, width_clipped = _phase1b_width(record)
        wireframe = phase1b_gripper_wireframe(pose, display_width)
        center = np.asarray(wireframe["center"], dtype=np.float64)
        is_formal_selected = (
            interface_id == "multi_pixel"
            and bool(record.get("server_selected"))
            and not bool(record.get("partial_recovery_selected"))
        )
        is_direct_fallback = (
            not formal_multi_selected
            and interface_id == "direct_xyz"
            and rank == 1
        )
        highlighted = is_formal_selected or is_direct_fallback
        candidate_color = "#D4AF37" if highlighted else interface_colors[interface_id]
        initial_visible = interface_id == default_interface
        hover_text = _phase1b_hover_text(
            record, center, raw_width, display_width, width_clipped
        )
        common_meta = {
            "interface_id": interface_id,
            "candidate_id": candidate_id,
            "logical_id": logical_id,
            "rank": rank,
        }

        def add_candidate_trace(trace: go.Scatter3d, kind: str) -> None:
            index = add_trace(trace, {"kind": kind, **common_meta})
            candidate_trace_indices.append(index)
            candidate_trace_interfaces.append(interface_id)

        add_candidate_trace(
            go.Scatter3d(
                x=wireframe["x"],
                y=wireframe["y"],
                z=wireframe["z"],
                mode="lines",
                name=f"{interface_id} r{rank} gripper",
                line={
                    "color": candidate_color,
                    "width": 8 if highlighted else 5,
                    "dash": "dot" if record.get("partial_recovery_selected") else "solid",
                },
                legendgroup=legend_group,
                showlegend=False,
                visible=initial_visible,
                hoverinfo="skip",
            ),
            "gripper_wireframe",
        )
        for axis_name, axis_color in axis_specs:
            endpoint = np.asarray(wireframe["axis_endpoints"][axis_name])
            add_candidate_trace(
                go.Scatter3d(
                    x=[center[0], endpoint[0]],
                    y=[center[1], endpoint[1]],
                    z=[center[2], endpoint[2]],
                    mode="lines",
                    name=f"{axis_name} axis",
                    line={"color": axis_color, "width": 7},
                    legendgroup=legend_group,
                    showlegend=False,
                    visible=initial_visible,
                    hoverinfo="skip",
                ),
                "axis_" + axis_name,
            )
        add_candidate_trace(
            go.Scatter3d(
                x=[center[0]],
                y=[center[1]],
                z=[center[2]],
                mode="markers",
                name=(
                    f"{interface_id} rank {rank}"
                    + (" ★ selected" if is_formal_selected else "")
                    + (" ★ fallback" if is_direct_fallback else "")
                ),
                marker={
                    "size": 9 if highlighted else 7,
                    "color": candidate_color,
                    "line": {
                        "color": "#3E2723" if highlighted else "white",
                        "width": 2,
                    },
                },
                legendgroup=legend_group,
                legendgrouptitle={"text": interface_id},
                showlegend=True,
                visible=initial_visible,
                hovertext=[hover_text],
                hovertemplate="%{hovertext}<extra></extra>",
            ),
            "corrected_center",
        )

        raw_position_value = record.get("predicted_position_3d")
        raw_position: np.ndarray | None = None
        if interface_id in {"direct_xyz", "pixel_depth"} and raw_position_value is not None:
            try:
                candidate_raw = np.asarray(raw_position_value, dtype=np.float64)
            except (TypeError, ValueError):
                candidate_raw = np.empty(0)
            if candidate_raw.shape == (3,) and np.all(np.isfinite(candidate_raw)):
                raw_position = candidate_raw
        if raw_position is not None:
            add_candidate_trace(
                go.Scatter3d(
                    x=[raw_position[0]],
                    y=[raw_position[1]],
                    z=[raw_position[2]],
                    mode="markers",
                    name="VLM raw position",
                    marker={
                        "size": 8,
                        "symbol": "diamond-open",
                        "color": candidate_color,
                        "line": {"color": candidate_color, "width": 3},
                    },
                    legendgroup=legend_group,
                    showlegend=False,
                    visible=initial_visible,
                    hovertext=[hover_text + "<br>marker: VLM raw position"],
                    hovertemplate="%{hovertext}<extra></extra>",
                ),
                "raw_position",
            )
            add_candidate_trace(
                go.Scatter3d(
                    x=[raw_position[0], center[0]],
                    y=[raw_position[1], center[1]],
                    z=[raw_position[2], center[2]],
                    mode="lines",
                    name="raw-to-corrected connector",
                    line={"color": candidate_color, "width": 3, "dash": "dash"},
                    legendgroup=legend_group,
                    showlegend=False,
                    visible=initial_visible,
                    hoverinfo="skip",
                ),
                "raw_corrected_connector",
            )

        included_candidates.append(
            {
                "candidate_id": candidate_id,
                "source_stage": record.get("stage"),
                "representation_id": record.get("representation_id"),
                "interface_id": interface_id,
                "rank": rank,
                "candidate_index": candidate_index,
                "logical_id": logical_id,
                "cache_key": record.get("cache_key"),
                "server_selected": bool(record.get("server_selected")),
                "partial_recovery_selected": bool(
                    record.get("partial_recovery_selected")
                ),
                "confidence": record.get("confidence"),
                "corrected_position_3d": center.tolist(),
                "raw_position_3d": (
                    raw_position.tolist() if raw_position is not None else None
                ),
                "raw_width_m": raw_width,
                "display_width_m": display_width,
                "display_width_clipped": width_clipped,
                "width_feasible": bool(record.get("width_feasible")),
            }
        )

    point_buttons = [
        {
            "label": "局部点云",
            "method": "restyle",
            "args": [{"visible": [False, True]}, [full_index, local_index]],
        },
        {
            "label": "全场景",
            "method": "restyle",
            "args": [{"visible": [True, False]}, [full_index, local_index]],
        },
        {
            "label": "两者",
            "method": "restyle",
            "args": [{"visible": [True, True]}, [full_index, local_index]],
        },
    ]
    interface_buttons = []
    for interface_id in INTERFACE_IDS:
        interface_buttons.append(
            {
                "label": interface_id,
                "method": "restyle",
                "args": [
                    {
                        "visible": [
                            trace_interface == interface_id
                            for trace_interface in candidate_trace_interfaces
                        ]
                    },
                    candidate_trace_indices,
                ],
            }
        )
    interface_buttons.append(
        {
            "label": "全部接口",
            "method": "restyle",
            "args": [
                {"visible": [True] * len(candidate_trace_indices)},
                candidate_trace_indices,
            ],
        }
    )
    interface_active = INTERFACE_IDS.index(default_interface)
    figure.update_layout(
        title={
            "text": (
                f"Phase 1B 6-DOF grasp — {data.spec.scene_id} / "
                f"{data.spec.task_id} ({source_stage})"
            ),
            "x": 0.5,
        },
        template="plotly_white",
        height=820,
        margin={"l": 0, "r": 0, "t": 115, "b": 0},
        scene={
            "xaxis_title": "camera X (m)",
            "yaxis_title": "camera Y (m)",
            "zaxis_title": "camera Z (m)",
            "aspectmode": "data",
            "camera": {
                "eye": {"x": 1.45, "y": -1.45, "z": -1.1},
                "up": {"x": 0.0, "y": -1.0, "z": 0.0},
            },
        },
        legend={
            "groupclick": "togglegroup",
            "itemsizing": "constant",
            "x": 0.01,
            "y": 0.99,
            "bgcolor": "rgba(255,255,255,0.72)",
        },
        hoverlabel={"namelength": -1},
        uirevision=f"{data.spec.scene_id}:{data.spec.task_id}:phase1b",
        updatemenus=[
            {
                "type": "buttons",
                "direction": "right",
                "active": 2,
                "x": 0.01,
                "y": 1.12,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": point_buttons,
            },
            {
                "type": "dropdown",
                "direction": "down",
                "active": interface_active,
                "x": 0.72,
                "y": 1.12,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": interface_buttons,
            },
        ],
        annotations=[
            {
                "text": "点云层",
                "xref": "paper",
                "yref": "paper",
                "x": 0.01,
                "y": 1.17,
                "showarrow": False,
            },
            {
                "text": "候选接口",
                "xref": "paper",
                "yref": "paper",
                "x": 0.72,
                "y": 1.17,
                "showarrow": False,
            },
        ],
    )
    metadata = {
        "plotly_version": plotly.__version__,
        "default_interface": default_interface,
        "formal_multi_server_selected_present": formal_multi_selected,
        "point_counts": {
            "valid_source": point_layers["valid_source_count"],
            "full_displayed": int(len(point_layers["full_points"])),
            "local_source": point_layers["local_source_count"],
            "local_displayed": int(len(point_layers["local_points"])),
        },
        "local_radius_m": point_layers["local_radius_m"],
        "included_candidates": included_candidates,
        "trace_registry": trace_registry,
        "point_trace_indices": [full_index, local_index],
        "candidate_trace_indices": candidate_trace_indices,
    }
    return figure, metadata

def _phase1b_update_manifest(
    manifest_path: Path,
    entry: dict[str, Any],
) -> None:
    if manifest_path.exists():
        manifest = json.loads(
            manifest_path.read_text(encoding="utf-8"),
            parse_constant=reject_nonstandard_constant,
        )
    else:
        manifest = {
            "visualization_schema_version": PHASE1B_VIZ_SCHEMA_VERSION,
            "entries": [],
        }
    if manifest.get("visualization_schema_version") != PHASE1B_VIZ_SCHEMA_VERSION:
        raise ValueError("Visualization manifest schema mismatch")
    entry_key = (
        entry.get("scene_id"),
        entry.get("task_id"),
        entry.get("repeat_index"),
        entry.get("include_partial_recovery"),
    )
    entries = [
        existing
        for existing in manifest.get("entries", [])
        if (
            existing.get("scene_id"),
            existing.get("task_id"),
            existing.get("repeat_index"),
            existing.get("include_partial_recovery"),
        )
        != entry_key
    ]
    entries.append(entry)
    entries.sort(
        key=lambda item: (
            str(item.get("scene_id") or ""),
            str(item.get("task_id") or ""),
            int(item.get("repeat_index") or 0),
            bool(item.get("include_partial_recovery")),
        )
    )
    manifest["entries"] = entries
    dump_json(manifest_path, manifest)

def save_phase1a_grasp_visualization(
    result: Phase1ARunResult,
    scene_id: str,
    task_id: str,
    *,
    repeat_index: int = 0,
    include_partial: bool = False,
    data: SceneData | None = None,
) -> Phase1AVisualizationResult:
    visualization_dir = result.run_dir / "visualizations"
    visualization_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = visualization_dir / "manifest.json"
    repeat_suffix = "" if repeat_index == 0 else f"__repeat{repeat_index}"
    diagnostic_suffix = "__diagnostic_partial" if include_partial else ""
    html_path = (
        visualization_dir
        / f"{scene_id}__{task_id}__d3{repeat_suffix}{diagnostic_suffix}.html"
    )
    selection = select_phase1a_visualization_records(
        result,
        scene_id,
        task_id,
        repeat_index=repeat_index,
        include_partial=include_partial,
    )
    base_entry = {
        "scene_id": scene_id,
        "task_id": task_id,
        "repeat_index": int(repeat_index),
        "source_stage": selection.source_stage,
        "fallback_reason": selection.fallback_reason,
        "include_partial_recovery": bool(include_partial),
        "call_statuses": selection.call_statuses,
        "excluded_candidates": selection.excluded,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    if not selection.records:
        if html_path.exists():
            html_path.unlink()
        entry = {
            **base_entry,
            "status": "no_valid_corrected_pose",
            "html_path": None,
            "included_candidates": [],
            "filter_summary": {
                "included_count": 0,
                "excluded_count": len(selection.excluded),
            },
        }
        _phase1b_update_manifest(manifest_path, entry)
        return Phase1AVisualizationResult(
            None, None, manifest_path, entry["status"], entry
        )

    if data is None:
        matching_specs = [
            spec for spec in MANIFEST
            if spec.scene_id == scene_id and spec.task_id == task_id
        ]
        if len(matching_specs) != 1:
            raise ValueError("Visualization scene/task must resolve to exactly one manifest row")
        data = load_scene(matching_specs[0])
    if data.spec.scene_id != scene_id or data.spec.task_id != task_id:
        raise ValueError("Provided SceneData does not match requested scene/task")

    figure, metadata = build_phase1a_grasp_figure(
        data, selection.records, source_stage=selection.source_stage
    )
    figure.write_html(
        str(html_path),
        include_plotlyjs=True,
        full_html=True,
        auto_open=False,
        config={
            "responsive": True,
            "scrollZoom": True,
            "displaylogo": False,
        },
    )
    entry = {
        **base_entry,
        "status": "ok",
        "html_path": html_path.relative_to(result.run_dir).as_posix(),
        "self_contained_html": True,
        "default_interface": metadata["default_interface"],
        "formal_multi_server_selected_present": metadata[
            "formal_multi_server_selected_present"
        ],
        "point_counts": metadata["point_counts"],
        "local_radius_m": metadata["local_radius_m"],
        "included_candidates": metadata["included_candidates"],
        "filter_summary": {
            "included_count": len(metadata["included_candidates"]),
            "excluded_count": len(selection.excluded),
        },
    }
    _phase1b_update_manifest(manifest_path, entry)
    return Phase1AVisualizationResult(
        figure, html_path, manifest_path, entry["status"], entry
    )

print(
    "Phase 1B visualization helpers ready: corrected-pose grippers, "
    "dual-layer point clouds, interface menus, and self-contained HTML."
)


## Offline verification (default execution path)

These checks use all 13 local captures and synthetic fixtures. The cache runner is tested with an injected in-memory transport; no socket or API call is made. The tests cover color order, D1 range/determinism, R3 failure behavior, all four strict parsers, D2/pose geometry, multi-pixel server selection, retry/budget/cache behavior, and profile call-count algebra.

In [ ]:
# Manifest and source-data invariants.
expected_scene_ids = {
    "20260417_115700", "20260417_115805", "20260417_115822",
    "20260417_115838", "20260417_115854", "20260417_115919",
    "20260417_120002", "20260417_120019", "20260417_120040",
    "20260417_120131", "20260417_120153", "20260417_120218",
    "20260417_120244",
}
assert {spec.scene_id for spec in MANIFEST} == expected_scene_ids
assert sum(spec.clutter for spec in MANIFEST) == 3
assert {spec.category for spec in MANIFEST} == {
    "rubiks_cube", "mustard_bottle", "coffee_can", "thermos"
}
assert all(row["valid_depth_fraction"] > 0.85 for row in AUDIT_ROWS)
assert all(row["bgr_to_rgb_preview_mae"] < row["unflipped_preview_mae"] for row in AUDIT_ROWS)

# R1 and R2 calibration/range behavior.
synthetic_depth = np.asarray([[np.nan, 0.0, 0.01, 1.25, 2.5, 3.5]], dtype=np.float32)
jet = np.asarray(render_calibrated_jet(synthetic_depth))
gray = np.asarray(render_linear_gray(synthetic_depth))
assert np.all(jet[0, 0] == 0) and np.all(jet[0, 1] == 0)
assert jet[0, 2, 2] > jet[0, 2, 0]  # near = blue
assert jet[0, 4, 0] > jet[0, 4, 2]  # far = red
assert np.array_equal(jet[0, 4], jet[0, 5])  # >2.5 m clips
assert np.all(gray[0, :2] == 0)
assert 120 <= int(gray[0, 3, 0]) <= 135
assert np.all(gray[0, 4] == 255) and np.all(gray[0, 5] == 255)
assert render_depth_scale().size == (512, 96)

# Exact R5 grid and D2 window conventions.
grid = coordinate_grid_points(1280, 720)
assert len(grid) == 91
assert grid[0] == (50, 50) and grid[-1] == (1250, 650)
depth_fixture = np.full((7, 7), np.nan, dtype=np.float32)
depth_fixture[:3, :3] = np.arange(1, 10, dtype=np.float32).reshape(3, 3)
edge_stats = sample_depth_stats(depth_fixture, (0, 0))
assert edge_stats.window_area == 9 and edge_stats.valid_count == 9
assert edge_stats.median_m == 5.0  # values above the display range remain valid for D2

# R6 plane normal points toward the camera and invalid areas remain black.
plane = np.ones((20, 20), dtype=np.float32)
plane_K = np.asarray([[100.0, 0.0, 9.5], [0.0, 100.0, 9.5], [0.0, 0.0, 1.0]])
plane_normals = np.asarray(render_normal_map(plane, plane_K))
assert plane_normals.shape == (20, 20, 3)
assert plane_normals[10, 10, 2] <= 1  # approximately [0,0,-1]
assert 126 <= plane_normals[10, 10, 0] <= 129
invalid_plane = plane.copy()
invalid_plane[8:13, 8:13] = np.nan
assert np.all(np.asarray(render_normal_map(invalid_plane, plane_K))[10, 10] == 0)

# Build all six formal representations without calling the VLM.
sample_scene = load_scene(MANIFEST[0])
sample_points = organized_pointcloud(sample_scene.depth, sample_scene.K)
sample_valid_points = (
    np.isfinite(sample_points).all(axis=-1)
    & (sample_scene.depth > 0)
    & (sample_scene.depth <= DEPTH_MAX_M)
)
assert np.all((-1.8 <= sample_points[..., 0][sample_valid_points]) & (sample_points[..., 0][sample_valid_points] <= 1.8))
assert np.all((-1.1 <= sample_points[..., 1][sample_valid_points]) & (sample_points[..., 1][sample_valid_points] <= 0.5))
manual_anchors = AnchorSet(
    object_bbox_xyxy=(400, 220, 850, 650),
    object_uv=(620, 410),
    grasp_uvs=((600, 390), (650, 430)),
    table_uv=(620, 680),
    source="offline_fixture",
)
bundles = {
    representation_id: build_representation(
        sample_scene,
        representation_id,
        anchors=manual_anchors if representation_id == "r3_local_depth_text" else None,
    )
    for representation_id in REPRESENTATION_IDS
}
assert bundles["r1_calibrated_jet"].images[1][1].size == (1280, 720)
assert bundles["r2_linear_gray"].images[1][1].size == (1280, 720)
assert bundles["r3_local_depth_text"].images[0][1].size == (1280, 720)
assert [image.size for _, image in bundles["r4_pointcloud_views"].images[1:]] == [(960, 720)] * 3
assert bundles["r5_coordinate_grid"].images[0][1].size == (1280, 720)
normal_array = np.asarray(bundles["r6_surface_normals"].images[1][1])
assert normal_array.shape == (720, 1280, 3)
assert np.all(normal_array[~(np.isfinite(sample_scene.depth) & (sample_scene.depth > 0))] == 0)

# Exercise every representation on each of the remaining 12 captures as well.
for remaining_spec in MANIFEST[1:]:
    remaining_scene = load_scene(remaining_spec)
    h, w = remaining_scene.depth.shape
    remaining_points = organized_pointcloud(remaining_scene.depth, remaining_scene.K)
    remaining_valid = (
        np.isfinite(remaining_points).all(axis=-1)
        & (remaining_scene.depth > 0)
        & (remaining_scene.depth <= DEPTH_MAX_M)
    )
    assert np.all((-1.8 <= remaining_points[..., 0][remaining_valid]) & (remaining_points[..., 0][remaining_valid] <= 1.8))
    assert np.all((-1.1 <= remaining_points[..., 1][remaining_valid]) & (remaining_points[..., 1][remaining_valid] <= 0.5))
    remaining_anchors = AnchorSet(
        object_bbox_xyxy=(w // 3, h // 4, 2 * w // 3, 3 * h // 4),
        object_uv=(w // 2, h // 2),
        grasp_uvs=((w // 2 - 20, h // 2), (w // 2 + 20, h // 2)),
        table_uv=(w // 2, h - 40),
        source="offline_fixture",
    )
    remaining_bundles = {
        representation_id: build_representation(
            remaining_scene,
            representation_id,
            anchors=remaining_anchors if representation_id == "r3_local_depth_text" else None,
        )
        for representation_id in REPRESENTATION_IDS
    }
    assert remaining_bundles["r1_calibrated_jet"].images[1][1].size == (w, h)
    assert remaining_bundles["r2_linear_gray"].images[1][1].size == (w, h)
    assert remaining_bundles["r3_local_depth_text"].images[0][1].size == (w, h)
    assert [image.size for _, image in remaining_bundles["r4_pointcloud_views"].images[1:]] == [(960, 720)] * 3
    assert remaining_bundles["r5_coordinate_grid"].images[0][1].size == (w, h)
    remaining_normals = np.asarray(remaining_bundles["r6_surface_normals"].images[1][1])
    remaining_invalid = ~(np.isfinite(remaining_scene.depth) & (remaining_scene.depth > 0))
    assert remaining_normals.shape == (h, w, 3)
    assert np.all(remaining_normals[remaining_invalid] == 0)

# Fixed scene-local sampling makes R4 byte deterministic.
r4_again = build_representation(sample_scene, "r4_pointcloud_views")
assert [
    hashlib.sha256(pil_png_bytes(image)).hexdigest()
    for _, image in bundles["r4_pointcloud_views"].images
] == [
    hashlib.sha256(pil_png_bytes(image)).hexdigest()
    for _, image in r4_again.images
]

# R3 prelocalization failure is explicit; no heuristic anchor is substituted.
try:
    derive_anchor_set(sample_scene, '{"candidates": []}')
except Exception:
    pass
else:
    raise AssertionError("R3 localization failure must raise")
wrapped_anchor_payload = {
    "object_box": [100, 100, 900, 900],
    "object_point": [1500, 500],
    "candidates": [
        {
            "grasp_region_box": [300, 300, 700, 700],
            "grasp_point": [500, 500],
        }
    ],
}
try:
    derive_anchor_set(sample_scene, json.dumps(wrapped_anchor_payload))
except ValueError:
    pass
else:
    raise AssertionError("R3 must reject normalized coordinates outside [0,1000]")
try:
    build_representation(sample_scene, "r3_local_depth_text", anchors=None)
except RepresentationUnavailable:
    pass
else:
    raise AssertionError("R3 without anchors must be unavailable")

assert legacy_diagnostic(sample_scene) is None or RUN_LEGACY_DIAGNOSTIC
print("Offline representation checks passed for all 13 captures and R1–R6.")


In [ ]:
def fixture_candidate(
    interface_id: str,
    rank: int,
    pixel_uv: tuple[int, int],
) -> dict[str, Any]:
    candidate: dict[str, Any] = {
        "rank": rank,
        "pixel_uv": list(pixel_uv),
        "closing_direction_3d": [2.0, 0.0, 0.0],
        "approach_direction_3d": [0.2, 0.0, 3.0],
        "estimated_object_width_along_close_m": 0.05,
        "confidence": 0.8,
        "reasoning_summary": f"fixture rank {rank}",
    }
    if interface_id == "direct_xyz":
        candidate["position_3d"] = [-0.3, -0.3, 1.0]
    elif interface_id == "pixel_depth":
        candidate["estimated_depth_m"] = 1.0
    elif interface_id == "depth_band":
        candidate["depth_band_index"] = 4
    return candidate

def fixture_response(
    interface_id: str,
    pixels: Sequence[tuple[int, int]] = ((10, 10), (30, 30), (50, 50)),
) -> str:
    return json.dumps(
        {
            "candidates": [
                fixture_candidate(interface_id, rank, pixel)
                for rank, pixel in zip((1, 2, 3), pixels, strict=True)
            ]
        },
        allow_nan=False,
    )

# All four strict schemas accept their own valid fixture.
for interface_id in INTERFACE_IDS:
    parsed_fixture = parse_response_strict(fixture_response(interface_id), interface_id)
    assert parsed_fixture.strict_json_valid and parsed_fixture.top_level_valid
    assert len(parsed_fixture.candidates) == 3
    assert all(candidate.schema_valid for candidate in parsed_fixture.candidates)

# Missing/extra fields, wrappers, non-standard constants, and loose numeric coercions fail.
missing_payload = json.loads(fixture_response("direct_xyz"))
del missing_payload["candidates"][0]["confidence"]
missing_parsed = parse_response_strict(json.dumps(missing_payload), "direct_xyz")
assert not missing_parsed.candidates[0].schema_valid

extra_payload = json.loads(fixture_response("multi_pixel"))
extra_payload["candidates"][0]["estimated_depth_m"] = 1.0
assert not parse_response_strict(json.dumps(extra_payload), "multi_pixel").candidates[0].schema_valid

fence = chr(96) * 3
assert not parse_response_strict(fence + "json\n" + fixture_response("direct_xyz") + "\n" + fence, "direct_xyz").strict_json_valid
assert not parse_response_strict('{"candidates":[{"rank":NaN}]}', "direct_xyz").strict_json_valid
loose_payload = json.loads(fixture_response("direct_xyz"))
loose_payload["candidates"][0]["pixel_uv"] = [10.0, "10"]
loose_payload["candidates"][0]["rank"] = True
loose_candidate = parse_response_strict(json.dumps(loose_payload), "direct_xyz").candidates[0]
assert not loose_candidate.schema_valid
out_of_range_rank_payload = json.loads(fixture_response("multi_pixel"))
out_of_range_rank_payload["candidates"][0]["rank"] = 4
out_of_range_rank = parse_response_strict(json.dumps(out_of_range_rank_payload), "multi_pixel")
assert not out_of_range_rank.candidates[0].schema_valid

wrong_count = json.loads(fixture_response("direct_xyz"))
wrong_count["candidates"].pop()
assert not parse_response_strict(json.dumps(wrong_count), "direct_xyz").top_level_valid
negative_z_payload = json.loads(fixture_response("direct_xyz"))
negative_z_payload["candidates"][0]["position_3d"][2] = -1.0
assert not parse_response_strict(json.dumps(negative_z_payload), "direct_xyz").candidates[0].schema_valid

# Gram–Schmidt uses closing/lateral/approach columns and rejects degeneracy.
orientation = build_orientation([2, 0, 0], [0.2, 0, 3])
assert orientation.valid and orientation.rotation_3x3 is not None
assert np.allclose(orientation.rotation_3x3[:, 0], [1, 0, 0])
assert np.allclose(orientation.rotation_3x3[:, 1], [0, 1, 0])
assert np.allclose(orientation.rotation_3x3[:, 2], [0, 0, 1])
assert np.isclose(np.linalg.det(orientation.rotation_3x3), 1.0)
assert not build_orientation([1, 0, 0], [2, 0, 0]).valid
assert not build_orientation([0, 0, 0], [0, 0, 1]).valid

# D2 correction and multi-pixel selection on controlled 5x5 windows.
dummy_depth = np.full((80, 80), np.nan, dtype=np.float32)
dummy_depth[8:13, 8:13] = np.linspace(0.8, 1.2, 25, dtype=np.float32).reshape(5, 5)
dummy_depth[28:33, 28:33] = 1.1
dummy_depth[28, 28] = np.nan
dummy_depth[48:53, 48:53] = 1.2
dummy_rgb = np.zeros((80, 80, 3), dtype=np.uint8)
dummy_K = np.asarray([[100.0, 0.0, 40.0], [0.0, 100.0, 40.0], [0.0, 0.0, 1.0]])
direct_schema_fixture = json.loads(interface_schema_text("direct_xyz", dummy_K))
for schema_candidate in direct_schema_fixture["candidates"]:
    schema_u, schema_v = schema_candidate["pixel_uv"]
    assert np.allclose(
        schema_candidate["position_3d"],
        deproject_pixel(schema_u, schema_v, 0.8, dummy_K),
        atol=1e-8,
    )
dummy_spec = SceneSpec("fixture_scene", "fixture_task", "grasp fixture", "fixture", False)
dummy_scene = SceneData(
    dummy_spec,
    ROOT,
    dummy_rgb[..., ::-1],
    dummy_rgb,
    dummy_depth,
    dummy_K,
    Image.fromarray(dummy_rgb, "RGB"),
    None,
)
dummy_call = CallResult("ok", "fixture-key", None, None, False, 0.25, 0.01, 1)

# Every interface is covered by missing-field, OOB, NaN, non-orthogonal, and degenerate fixtures.
for interface_id in INTERFACE_IDS:
    interface_missing = json.loads(fixture_response(interface_id))
    del interface_missing["candidates"][0]["confidence"]
    assert not parse_response_strict(json.dumps(interface_missing), interface_id).candidates[0].schema_valid

    interface_nan = fixture_response(interface_id).replace("\"confidence\": 0.8", "\"confidence\": NaN", 1)
    assert not parse_response_strict(interface_nan, interface_id).strict_json_valid

    interface_oob = parse_response_strict(
        fixture_response(interface_id, pixels=((100, 100), (30, 30), (50, 50))),
        interface_id,
    )
    interface_oob_records = evaluate_parsed_response(
        data=dummy_scene,
        parsed=interface_oob,
        stage="D3",
        representation_id="r1_calibrated_jet",
        interface_id=interface_id,
        repeat_index=0,
        call_result=dummy_call,
    )
    assert not next(row for row in interface_oob_records if row["rank"] == 1)["validation_flags"]["pixel_in_bounds"]

    interface_parallel = json.loads(fixture_response(interface_id))
    interface_parallel["candidates"][0]["approach_direction_3d"] = [4.0, 0.0, 0.0]
    parallel_candidate = parse_response_strict(json.dumps(interface_parallel), interface_id)
    parallel_rows = evaluate_parsed_response(
        data=dummy_scene, parsed=parallel_candidate, stage="D3",
        representation_id="r1_calibrated_jet", interface_id=interface_id,
        repeat_index=0, call_result=dummy_call,
    )
    assert not next(row for row in parallel_rows if row["rank"] == 1)["validation_flags"]["orientation_valid"]

    interface_zero = json.loads(fixture_response(interface_id))
    interface_zero["candidates"][0]["closing_direction_3d"] = [0.0, 0.0, 0.0]
    zero_candidate = parse_response_strict(json.dumps(interface_zero), interface_id)
    zero_rows = evaluate_parsed_response(
        data=dummy_scene, parsed=zero_candidate, stage="D3",
        representation_id="r1_calibrated_jet", interface_id=interface_id,
        repeat_index=0, call_result=dummy_call,
    )
    assert not next(row for row in zero_rows if row["rank"] == 1)["validation_flags"]["orientation_valid"]

direct_parsed = parse_response_strict(fixture_response("direct_xyz"), "direct_xyz")
direct_records = evaluate_parsed_response(
    data=dummy_scene,
    parsed=direct_parsed,
    stage="D1",
    representation_id="r1_calibrated_jet",
    interface_id="direct_xyz",
    repeat_index=0,
    call_result=dummy_call,
)
rank1 = next(record for record in direct_records if record["rank"] == 1)
expected_corrected = deproject_pixel(10, 10, 1.0, dummy_K)
assert np.allclose(rank1["sensor_corrected_position_3d"], expected_corrected)
assert np.allclose(np.asarray(rank1["sensor_corrected_pose_4x4"])[:3, :3], np.eye(3))
assert rank1["sensor_depth_median_m"] == 1.0
assert rank1["sensor_z_abs_difference_m"] == 0.0
assert rank1["vlm_backprojection_arithmetic_error_m"] is not None
assert rank1["projection_consistency_error_px"] is not None
reference_pose = np.eye(4, dtype=np.float64)
reference_pose[:3, 3] = [0.0, 0.0, 1.0]
reference_record = dict(rank1)
reference_record["predicted_position_3d"] = [0.0, 0.0, 1.0]
reference_record["sensor_corrected_position_3d"] = [-0.3, -0.3, 1.0]
reference_record["good_pixel_hit"] = None
reference_record["good_pixel_distance_px"] = None
reference_entry = ReferenceEntry(
    "fixture_scene", "fixture_task", (reference_pose,), (np.eye(3),), None
)
apply_reference_metrics([reference_record], {("fixture_scene", "fixture_task"): reference_entry})
assert reference_record["raw_epsilon_pos_m"] == 0.0
assert reference_record["corrected_epsilon_pos_m"] > 0.4
assert reference_record["epsilon_pos_m"] == reference_record["raw_epsilon_pos_m"]
assert reference_record["orientation_error_rad"] == 0.0
assert reference_record["good_pixel_hit"] is None
same_bytes_row = {
    "scene_id": "fixture_scene",
    "task_id": "fixture_task",
    "grasps": [{"pose_4x4": reference_pose.tolist()}],
}
same_bytes_payload = (json.dumps(same_bytes_row, allow_nan=False) + "\n").encode("utf-8")
same_bytes_references = load_reference_grasps(
    Path("fixture-reference.jsonl"), contents=same_bytes_payload
)
assert ("fixture_scene", "fixture_task") in same_bytes_references
nearest_3d_pose = np.eye(4, dtype=np.float64)
nearest_3d_pose[:3, 3] = [1.0, 0.0, 0.0]
nearest_xy_only_pose = np.eye(4, dtype=np.float64)
nearest_xy_only_pose[:3, 3] = [0.0, 0.0, 10.0]
same_reference_record = dict(reference_record)
same_reference_record["predicted_position_3d"] = [0.0, 0.0, 0.0]
same_reference_record["sensor_corrected_position_3d"] = [0.0, 0.0, 0.0]
same_reference_entry = replace(
    reference_entry, poses_4x4=(nearest_3d_pose, nearest_xy_only_pose)
)
apply_reference_metrics(
    [same_reference_record], {("fixture_scene", "fixture_task"): same_reference_entry}
)
assert math.isclose(same_reference_record["raw_epsilon_pos_m"], 1.0)
assert math.isclose(same_reference_record["raw_xy_error_m"], 1.0)
assert math.isclose(same_reference_record["corrected_epsilon_pos_m"], 1.0)
assert math.isclose(same_reference_record["corrected_xy_error_m"], 1.0)
good_pixel_record = dict(reference_record)
good_pixel_record["pixel_uv"] = [12, 10]
good_pixel_entry = replace(reference_entry, good_pixels_uv=frozenset({(10, 10), (20, 20)}))
apply_reference_metrics([good_pixel_record], {("fixture_scene", "fixture_task"): good_pixel_entry})
assert good_pixel_record["good_pixel_hit"] is False
assert good_pixel_record["good_pixel_distance_px"] == 2.0

multi_parsed = parse_response_strict(fixture_response("multi_pixel"), "multi_pixel")
multi_records = evaluate_parsed_response(
    data=dummy_scene,
    parsed=multi_parsed,
    stage="D3",
    representation_id="r1_calibrated_jet",
    interface_id="multi_pixel",
    repeat_index=0,
    call_result=dummy_call,
)
selected = [record for record in multi_records if record["server_selected"]]
assert len(selected) == 1 and selected[0]["rank"] == 3
assert next(record for record in multi_records if record["rank"] == 2)["valid_depth_count"] == 24
partially_bad_multi_payload = json.loads(fixture_response("multi_pixel"))
del partially_bad_multi_payload["candidates"][0]["confidence"]
partially_bad_multi = parse_response_strict(json.dumps(partially_bad_multi_payload), "multi_pixel")
partially_bad_records = evaluate_parsed_response(
    data=dummy_scene, parsed=partially_bad_multi, stage="D3",
    representation_id="r1_calibrated_jet", interface_id="multi_pixel",
    repeat_index=0, call_result=dummy_call,
)
assert not any(record["server_selected"] for record in partially_bad_records)
assert next(
    record for record in partially_bad_records if record["partial_recovery_selected"]
)["rank"] == 3

out_of_bounds_text = fixture_response(
    "direct_xyz",
    pixels=((100, 100), (30, 30), (50, 50)),
)
out_of_bounds_parsed = parse_response_strict(out_of_bounds_text, "direct_xyz")
out_of_bounds_records = evaluate_parsed_response(
    data=dummy_scene,
    parsed=out_of_bounds_parsed,
    stage="D1",
    representation_id="r1_calibrated_jet",
    interface_id="direct_xyz",
    repeat_index=0,
    call_result=dummy_call,
)
assert not next(record for record in out_of_bounds_records if record["rank"] == 1)["validation_flags"]["pixel_in_bounds"]

parallel_payload = json.loads(fixture_response("direct_xyz"))
parallel_payload["candidates"][0]["approach_direction_3d"] = [4.0, 0.0, 0.0]
parallel_parsed = parse_response_strict(json.dumps(parallel_payload), "direct_xyz")
parallel_records = evaluate_parsed_response(
    data=dummy_scene,
    parsed=parallel_parsed,
    stage="D1",
    representation_id="r1_calibrated_jet",
    interface_id="direct_xyz",
    repeat_index=0,
    call_result=dummy_call,
)
parallel_rank1 = next(record for record in parallel_records if record["rank"] == 1)
assert not parallel_rank1["validation_flags"]["orientation_valid"]
assert parallel_rank1["sensor_corrected_pose_4x4"] is None

assert depth_band_metrics(4, 1.1) == (True, 0.0)
assert depth_band_metrics(10, 3.0) == (True, 0.0)
overflow_hit, overflow_distance = depth_band_metrics(10, 2.4)
assert not overflow_hit and math.isclose(overflow_distance, 0.1)
print("Strict parser, D2, orientation, and multi-pixel fixture checks passed.")


In [ ]:
class FakeResponse:
    def __init__(self, payload: bytes):
        self.payload = payload

    def __enter__(self) -> "FakeResponse":
        return self

    def __exit__(self, exc_type: Any, exc: Any, traceback: Any) -> bool:
        return False

    def read(self) -> bytes:
        return self.payload

fixture_model_text = fixture_response("direct_xyz")
fixture_envelope = json.dumps(
    {
        "candidates": [
            {"content": {"parts": [{"text": fixture_model_text}]}}
        ]
    },
    allow_nan=False,
).encode("utf-8")
tiny_image = EncodedImage(
    "fixture RGB",
    "image/png",
    pil_png_bytes(Image.new("RGB", (8, 8), (10, 20, 30))),
)

original_cache_root = CACHE_ROOT
old_gemini_key = os.environ.get("GEMINI_API_KEY")
old_google_key = os.environ.get("GOOGLE_API_KEY")
transport_urls: list[str] = []
try:
    with tempfile.TemporaryDirectory(prefix="phase1a-cache-test-") as temp_dir:
        CACHE_ROOT = Path(temp_dir)

        def forbidden_transport(request: urllib.request.Request, *, timeout: float) -> Any:
            raise AssertionError("offline cache miss attempted transport")

        offline_result = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="offline-miss",
            images=[tiny_image],
            repeat_index=0,
            online=False,
            budget=CallBudget(0),
            transport=forbidden_transport,
        )
        assert offline_result.status == "offline_miss"

        os.environ["GEMINI_API_KEY"] = "unit-test-key-not-real"
        os.environ.pop("GOOGLE_API_KEY", None)

        def success_transport(request: urllib.request.Request, *, timeout: float) -> FakeResponse:
            transport_urls.append(request.full_url)
            assert "unit-test-key-not-real" not in request.full_url
            assert timeout > 0
            return FakeResponse(fixture_envelope)

        budget = CallBudget(4)
        first = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="cache-fixture",
            images=[tiny_image],
            repeat_index=0,
            online=True,
            budget=budget,
            transport=success_transport,
            sleeper=lambda _: None,
        )
        assert first.status == "ok" and not first.from_cache
        assert first.response_text == fixture_model_text
        assert budget.new_http_calls == 1 and len(transport_urls) == 1
        assert cache_path_for(first.cache_key).exists()

        second = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="cache-fixture",
            images=[tiny_image],
            repeat_index=0,
            online=False,
            budget=CallBudget(0),
            transport=forbidden_transport,
        )
        assert second.status == "ok" and second.from_cache
        assert second.cache_key == first.cache_key
        assert second.original_network_latency_s == first.original_network_latency_s
        assert len(transport_urls) == 1

        different_repeat = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="cache-fixture",
            images=[tiny_image],
            repeat_index=1,
            online=True,
            budget=budget,
            transport=success_transport,
            sleeper=lambda _: None,
        )
        assert different_repeat.cache_key != first.cache_key
        assert budget.new_http_calls == 2 and len(transport_urls) == 2

        # A hard budget is checked before transport and cannot be exceeded.
        hard_transport_calls: list[str] = []
        def counted_transport(request: urllib.request.Request, *, timeout: float) -> FakeResponse:
            hard_transport_calls.append(request.full_url)
            return FakeResponse(fixture_envelope)

        hard_budget = CallBudget(1)
        hard_first = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="hard-budget-a",
            images=[tiny_image],
            repeat_index=0,
            online=True,
            budget=hard_budget,
            transport=counted_transport,
            sleeper=lambda _: None,
        )
        hard_second = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="hard-budget-b",
            images=[tiny_image],
            repeat_index=0,
            online=True,
            budget=hard_budget,
            transport=counted_transport,
            sleeper=lambda _: None,
        )
        assert hard_first.status == "ok"
        assert hard_second.status == "budget_exhausted"
        assert hard_budget.new_http_calls == 1
        assert len(hard_transport_calls) == 1

        # Attempt ledger is cumulative across a reconstructed budget object.
        ledger_path = Path(temp_dir) / "attempt-ledger.json"
        persistent_budget = CallBudget(2, 0, ledger_path)
        persistent_budget.reserve_transport_attempt()
        persisted = json.loads(ledger_path.read_text(encoding="utf-8"))
        assert persisted["used_http_transport_attempts"] == 1
        resumed_budget = CallBudget(
            2, persisted["used_http_transport_attempts"], ledger_path
        )
        resumed_budget.reserve_transport_attempt()
        try:
            resumed_budget.reserve_transport_attempt()
        except BudgetExceeded:
            pass
        else:
            raise AssertionError("resumed budget exceeded its cumulative cap")
        assert json.loads(ledger_path.read_text())["used_http_transport_attempts"] == 2

        # 429 retries consume the budget; a single API failure returns a recordable result.
        retry_count = 0
        retry_delays: list[float] = []
        def retry_transport(request: urllib.request.Request, *, timeout: float) -> FakeResponse:
            global retry_count
            retry_count += 1
            if retry_count == 1:
                raise urllib.error.HTTPError(
                    request.full_url,
                    429,
                    "rate limited",
                    {},
                    BytesIO(b'{"error":"rate limited"}'),
                )
            return FakeResponse(fixture_envelope)

        retry_budget = CallBudget(2)
        retry_result = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="retry-fixture",
            images=[tiny_image],
            repeat_index=0,
            online=True,
            budget=retry_budget,
            transport=retry_transport,
            sleeper=retry_delays.append,
        )
        assert retry_result.status == "ok"
        assert retry_result.transport_attempts == 2
        assert retry_budget.new_http_calls == 2
        assert retry_delays == [1.0]

        server_error_count = 0
        def server_error_transport(
            request: urllib.request.Request, *, timeout: float
        ) -> FakeResponse:
            global server_error_count
            server_error_count += 1
            if server_error_count == 1:
                raise urllib.error.HTTPError(
                    request.full_url, 599, "server error", {}, BytesIO(b"{}")
                )
            return FakeResponse(fixture_envelope)

        all_5xx_result = run_cached_gemini_request(
            semantic_interface="direct_xyz", prompt="all-5xx-fixture",
            images=[tiny_image], repeat_index=0, online=True, budget=CallBudget(2),
            transport=server_error_transport, sleeper=lambda _: None,
        )
        assert all_5xx_result.status == "ok" and all_5xx_result.transport_attempts == 2

        failure_count = 0
        def failing_transport(request: urllib.request.Request, *, timeout: float) -> Any:
            global failure_count
            failure_count += 1
            raise urllib.error.URLError("fixture failure")

        failure_budget = CallBudget(4)
        failure_result = run_cached_gemini_request(
            semantic_interface="direct_xyz",
            prompt="failure-fixture",
            images=[tiny_image],
            repeat_index=0,
            online=True,
            budget=failure_budget,
            transport=failing_transport,
            sleeper=lambda _: None,
        )
        assert failure_result.status == "api_error"
        assert failure_result.transport_attempts == 4
        assert failure_budget.new_http_calls == 4

        def unexpected_transport(request: urllib.request.Request, *, timeout: float) -> Any:
            raise TypeError("fixture unexpected transport failure")

        unexpected_budget = CallBudget(1)
        unexpected_result = run_cached_gemini_request(
            semantic_interface="direct_xyz", prompt="unexpected-failure-fixture",
            images=[tiny_image], repeat_index=0, online=True, budget=unexpected_budget,
            transport=unexpected_transport, sleeper=lambda _: None,
        )
        assert unexpected_result.status == "api_error"
        assert unexpected_result.transport_attempts == 1
        assert unexpected_budget.new_http_calls == 1

        # Simulate a complete 10-request smoke first pass and a zero-transport cache rerun.
        smoke_first_budget = CallBudget(10)
        smoke_first = [
            run_cached_gemini_request(
                semantic_interface="direct_xyz",
                prompt=f"smoke-cache-{index}",
                images=[tiny_image],
                repeat_index=0,
                online=True,
                budget=smoke_first_budget,
                transport=success_transport,
                sleeper=lambda _: None,
            )
            for index in range(10)
        ]
        assert all(result.status == "ok" and not result.from_cache for result in smoke_first)
        assert smoke_first_budget.new_http_calls == 10
        smoke_rerun = [
            run_cached_gemini_request(
                semantic_interface="direct_xyz",
                prompt=f"smoke-cache-{index}",
                images=[tiny_image],
                repeat_index=0,
                online=False,
                budget=CallBudget(0),
                transport=forbidden_transport,
            )
            for index in range(10)
        ]
        assert all(result.from_cache and result.transport_attempts == 0 for result in smoke_rerun)

        failure_log = Path(temp_dir) / "failure-log.jsonl"
        append_jsonl(failure_log, {"logical_id": "failure", "status": failure_result.status})
        assert read_jsonl(failure_log)[0]["status"] == "api_error"
        assert latest_call_attempts(
            [
                {"logical_id": "resume", "call_status": "offline_miss"},
                {"logical_id": "resume", "call_status": "ok"},
            ]
        )[0]["call_status"] == "ok"
        latest_resume_call = [{"logical_id": "resume", "cache_key": "new-key"}]
        stale_candidate = {
            "logical_id": "resume", "cache_key": "old-key",
            "candidate_index": 0, "rank": 1,
        }
        fresh_candidate = {
            "logical_id": "resume", "cache_key": "new-key",
            "candidate_index": 0, "rank": 1,
        }
        assert candidates_for_latest_calls([stale_candidate], latest_resume_call) == []
        matching_candidates = candidates_for_latest_calls(
            [stale_candidate, fresh_candidate], latest_resume_call
        )
        assert candidate_for_call(
            matching_candidates, "resume", "new-key", rank=1
        ) == fresh_candidate

        assert json_safe(float("nan")) is None
        assert cache_key(
            semantic_interface="direct_xyz",
            prompt="same",
            images=[tiny_image],
            repeat_index=0,
        ) == cache_key(
            semantic_interface="direct_xyz",
            prompt="same",
            images=[tiny_image],
            repeat_index=0,
        )
finally:
    CACHE_ROOT = original_cache_root
    if old_gemini_key is None:
        os.environ.pop("GEMINI_API_KEY", None)
    else:
        os.environ["GEMINI_API_KEY"] = old_gemini_key
    if old_google_key is None:
        os.environ.pop("GOOGLE_API_KEY", None)
    else:
        os.environ["GOOGLE_API_KEY"] = old_google_key

# Nominal HTTP caps: one R3 anchor per scene, 6 D1 calls per repeat,
# and 3 new D3 calls because direct_xyz is reused.
for profile_name, config in PROFILE_CONFIGS.items():
    scene_count = len(config["scene_ids"])
    repeats = int(config["repeats"])
    calculated = scene_count + 6 * scene_count * repeats + 3 * scene_count * repeats
    assert calculated == int(config["expected_http_cap"])
assert [PROFILE_CONFIGS[name]["expected_http_cap"] for name in ("smoke", "pilot", "full")] == [10, 40, 364]

print("Offline cache, retry, hard-budget, resume-key, and call-count checks passed; real network calls: 0.")


In [ ]:
# Phase 1B synthetic tests: candidate provenance, point layers, pose geometry, and HTML.
viz_call_rows: list[dict[str, Any]] = []
viz_candidate_rows: list[dict[str, Any]] = []
for interface_id in INTERFACE_IDS:
    interface_key = f"viz-cache-{interface_id}"
    interface_call = CallResult(
        "ok", interface_key, fixture_response(interface_id), None, False, 0.2, 0.01, 1
    )
    interface_parsed = parse_response_strict(
        fixture_response(interface_id), interface_id
    )
    interface_records = evaluate_parsed_response(
        data=dummy_scene,
        parsed=interface_parsed,
        stage="D3",
        representation_id="r1_calibrated_jet",
        interface_id=interface_id,
        repeat_index=0,
        call_result=interface_call,
    )
    logical_id = f"phase1b-fixture-{interface_id}"
    viz_call_rows.append(
        {
            "logical_id": logical_id,
            "scene_id": dummy_spec.scene_id,
            "task_id": dummy_spec.task_id,
            "stage": "D3",
            "representation_id": "r1_calibrated_jet",
            "interface_id": interface_id,
            "repeat_index": 0,
            "call_status": "ok",
            "cache_key": interface_key,
        }
    )
    for record in interface_records:
        record["logical_id"] = logical_id
        record["cache_key"] = interface_key
        if interface_id == "direct_xyz" and record["rank"] == 1:
            record["estimated_object_width_along_close_m"] = 0.12
            record["width_feasible"] = False
            record["validation_flags"]["width_feasible_0p02_0p08_m"] = False
        viz_candidate_rows.append(record)

# A stale row and a partially recovered row must be preserved in exclusions, not drawn.
stale_record = json.loads(json.dumps(viz_candidate_rows[0], allow_nan=False))
stale_record["cache_key"] = "stale-cache-key"
viz_candidate_rows.append(stale_record)
partial_record = next(
    record for record in partially_bad_records
    if record["partial_recovery_selected"]
)
partial_record = json.loads(json.dumps(partial_record, allow_nan=False))
partial_record.update(
    {
        "logical_id": "phase1b-fixture-partial",
        "cache_key": "viz-cache-partial",
        "stage": "D3",
    }
)
viz_candidate_rows.append(partial_record)
partial_call_row = {
    "logical_id": "phase1b-fixture-partial",
    "scene_id": dummy_spec.scene_id,
    "task_id": dummy_spec.task_id,
    "stage": "D3",
    "representation_id": "r1_calibrated_jet",
    "interface_id": "multi_pixel",
    "repeat_index": 0,
    "call_status": "response_error",
    "cache_key": "viz-cache-partial",
}
viz_call_rows.append(partial_call_row)

with tempfile.TemporaryDirectory(prefix="phase1b-selection-") as selection_temp:
    selection_result = Phase1ARunResult(
        Path(selection_temp),
        "r1_calibrated_jet",
        CallBudget(0),
        0,
        viz_call_rows,
        viz_candidate_rows,
        [],
        [],
    )
    formal_selection = select_phase1a_visualization_records(
        selection_result, dummy_spec.scene_id, dummy_spec.task_id
    )
    assert formal_selection.source_stage == "D3"
    assert len(formal_selection.records) == 12
    assert all(
        record["cache_key"]
        == next(
            call["cache_key"]
            for call in viz_call_rows
            if call["logical_id"] == record["logical_id"]
        )
        for record in formal_selection.records
    )
    assert len([r for r in formal_selection.records if r["server_selected"]]) == 1
    excluded_reasons = {
        reason
        for item in formal_selection.excluded
        for reason in item["reasons"]
    }
    assert "stale_cache_key" in excluded_reasons
    assert "partial_recovery_excluded" in excluded_reasons

    partial_only_result = Phase1ARunResult(
        Path(selection_temp) / "partial-only",
        "r1_calibrated_jet",
        CallBudget(0),
        0,
        [partial_call_row],
        [partial_record],
        [],
        [],
    )
    assert not select_phase1a_visualization_records(
        partial_only_result, dummy_spec.scene_id, dummy_spec.task_id
    ).records
    diagnostic_selection = select_phase1a_visualization_records(
        partial_only_result,
        dummy_spec.scene_id,
        dummy_spec.task_id,
        include_partial=True,
    )
    assert len(diagnostic_selection.records) == 1
    assert diagnostic_selection.records[0]["partial_recovery_selected"]

# D3 absence falls back only to the selected D1 direct_xyz rank-1 pose.
fallback_record = json.loads(json.dumps(direct_records[0], allow_nan=False))
fallback_record.update(
    {
        "logical_id": "phase1b-fixture-d1-fallback",
        "cache_key": "viz-cache-d1-fallback",
        "stage": "D1",
    }
)
fallback_call = {
    "logical_id": fallback_record["logical_id"],
    "scene_id": dummy_spec.scene_id,
    "task_id": dummy_spec.task_id,
    "stage": "D1",
    "representation_id": "r1_calibrated_jet",
    "interface_id": "direct_xyz",
    "repeat_index": 0,
    "call_status": "ok",
    "cache_key": fallback_record["cache_key"],
}
fallback_result = Phase1ARunResult(
    Path(tempfile.gettempdir()) / "phase1b-fallback-fixture",
    "r1_calibrated_jet",
    CallBudget(0),
    0,
    [fallback_call],
    [fallback_record],
    [],
    [],
)
fallback_selection = select_phase1a_visualization_records(
    fallback_result, dummy_spec.scene_id, dummy_spec.task_id
)
assert fallback_selection.source_stage == "D1_fallback"
assert len(fallback_selection.records) == 1
assert fallback_selection.records[0]["rank"] == 1

# Point-cloud filtering occurs before independent deterministic full/local sampling.
viz_height, viz_width = 160, 160
viz_depth = np.ones((viz_height, viz_width), dtype=np.float32)
viz_depth[0, 0] = np.nan
viz_depth[0, 1] = 0.0
viz_depth[0, 2] = 3.0
viz_rgb = np.zeros((viz_height, viz_width, 3), dtype=np.uint8)
viz_rgb[..., 0] = np.arange(viz_width, dtype=np.uint8)[None, :]
viz_rgb[..., 1] = np.arange(viz_height, dtype=np.uint8)[:, None]
viz_rgb[..., 2] = 180
viz_K = np.asarray(
    [[300.0, 0.0, 79.5], [0.0, 300.0, 79.5], [0.0, 0.0, 1.0]]
)
viz_spec = SceneSpec(
    "phase1b_point_fixture", "point_fixture", "grasp fixture", "fixture", False
)
viz_scene = SceneData(
    viz_spec,
    ROOT,
    viz_rgb[..., ::-1],
    viz_rgb,
    viz_depth,
    viz_K,
    Image.fromarray(viz_rgb, "RGB"),
    None,
)
layer_kwargs = {
    "full_max_points": 100,
    "local_max_points": 120,
    "local_radius_m": 0.12,
}
layers_a = phase1b_pointcloud_layers(viz_scene, [[0.0, 0.0, 1.0]], **layer_kwargs)
layers_b = phase1b_pointcloud_layers(viz_scene, [[0.0, 0.0, 1.0]], **layer_kwargs)
assert layers_a["valid_source_count"] == viz_height * viz_width - 3
assert len(layers_a["full_points"]) == 100
assert layers_a["local_source_count"] > 120
assert len(layers_a["local_points"]) == 120
assert np.array_equal(layers_a["full_points"], layers_b["full_points"])
assert np.array_equal(layers_a["local_points"], layers_b["local_points"])
assert np.all((layers_a["full_points"][:, 2] > 0) & (layers_a["full_points"][:, 2] <= 2.5))
assert np.all(
    np.linalg.norm(layers_a["local_points"] - [0.0, 0.0, 1.0], axis=1)
    <= 0.120001
)

# The corrected pose supplies the center and all three axes; only width is display-clipped.
identity_pose = np.eye(4, dtype=np.float64)
raw_width, clipped_width, was_clipped = _phase1b_width(
    {"estimated_object_width_along_close_m": 0.20}
)
assert raw_width == 0.20 and clipped_width == 0.08 and was_clipped
wireframe_fixture = phase1b_gripper_wireframe(identity_pose, clipped_width)
wire_x = np.asarray([x for x in wireframe_fixture["x"] if x is not None])
assert np.isclose(wire_x.min(), -0.04) and np.isclose(wire_x.max(), 0.04)
assert np.allclose(wireframe_fixture["center"], [0.0, 0.0, 0.0])
assert np.allclose(
    wireframe_fixture["axis_endpoints"]["closing"],
    [PHASE1B_AXIS_LENGTH_M, 0.0, 0.0],
)
assert np.allclose(
    wireframe_fixture["axis_endpoints"]["lateral"],
    [0.0, PHASE1B_AXIS_LENGTH_M, 0.0],
)
assert np.allclose(
    wireframe_fixture["axis_endpoints"]["approach"],
    [0.0, 0.0, PHASE1B_AXIS_LENGTH_M],
)
assert min(z for z in wireframe_fixture["z"] if z is not None) < 0.0

# Figure controls alter disjoint trace subsets; hover keeps raw width and provenance.
fixture_figure, fixture_metadata = build_phase1a_grasp_figure(
    dummy_scene, formal_selection.records
)
assert fixture_metadata["default_interface"] == "multi_pixel"
assert fixture_metadata["formal_multi_server_selected_present"]
assert len(fixture_metadata["included_candidates"]) == 12
point_labels = [
    button.label for button in fixture_figure.layout.updatemenus[0].buttons
]
interface_labels = [
    button.label for button in fixture_figure.layout.updatemenus[1].buttons
]
assert point_labels == ["局部点云", "全场景", "两者"]
assert interface_labels == [*INTERFACE_IDS, "全部接口"]
assert all(
    button.method == "restyle"
    for menu in fixture_figure.layout.updatemenus
    for button in menu.buttons
)
assert set(fixture_metadata["point_trace_indices"]).isdisjoint(
    fixture_metadata["candidate_trace_indices"]
)
trace_kinds = [entry["kind"] for entry in fixture_metadata["trace_registry"]]
assert "raw_corrected_connector" in trace_kinds
assert {"axis_closing", "axis_lateral", "axis_approach"} <= set(trace_kinds)
assert fixture_figure.layout.legend.groupclick == "togglegroup"
clipped_candidate = next(
    item for item in fixture_metadata["included_candidates"]
    if item["interface_id"] == "direct_xyz" and item["rank"] == 1
)
assert clipped_candidate["raw_width_m"] == 0.12
assert clipped_candidate["display_width_m"] == 0.08
assert clipped_candidate["display_width_clipped"]
clipped_center_index = next(
    item["trace_index"]
    for item in fixture_metadata["trace_registry"]
    if item["kind"] == "corrected_center"
    and item["interface_id"] == "direct_xyz"
    and item["rank"] == 1
)
assert "0.1200" in fixture_figure.data[clipped_center_index].hovertext[0]
assert "width feasible [0.02,0.08] m: False" in fixture_figure.data[
    clipped_center_index
].hovertext[0]

multi_selected_record = next(
    record for record in formal_selection.records if record["server_selected"]
)
multi_pose = np.asarray(multi_selected_record["sensor_corrected_pose_4x4"])
closing_axis_index = next(
    item["trace_index"]
    for item in fixture_metadata["trace_registry"]
    if item["kind"] == "axis_closing"
    and item["interface_id"] == "multi_pixel"
    and item["rank"] == multi_selected_record["rank"]
)
closing_trace = fixture_figure.data[closing_axis_index]
closing_start = np.asarray(
    [closing_trace.x[0], closing_trace.y[0], closing_trace.z[0]], dtype=float
)
closing_end = np.asarray(
    [closing_trace.x[1], closing_trace.y[1], closing_trace.z[1]], dtype=float
)
assert np.allclose(closing_start, multi_pose[:3, 3])
assert np.allclose(
    closing_end,
    multi_pose[:3, 3] + PHASE1B_AXIS_LENGTH_M * multi_pose[:3, 0],
)

# Saving creates an offline self-contained HTML and a deduplicated provenance manifest.
with tempfile.TemporaryDirectory(prefix="phase1b-html-") as html_temp:
    html_result = Phase1ARunResult(
        Path(html_temp),
        "r1_calibrated_jet",
        CallBudget(0),
        0,
        viz_call_rows,
        viz_candidate_rows,
        [],
        [],
    )
    saved = save_phase1a_grasp_visualization(
        html_result,
        dummy_spec.scene_id,
        dummy_spec.task_id,
        data=dummy_scene,
    )
    assert saved.status == "ok"
    assert saved.html_path is not None and saved.html_path.exists()
    assert saved.html_path.name == "fixture_scene__fixture_task__d3.html"
    html_text = saved.html_path.read_text(encoding="utf-8")
    assert len(html_text) > 1_000_000
    assert "Plotly.newPlot" in html_text
    assert "<script src=" not in html_text
    assert all(interface_id in html_text for interface_id in INTERFACE_IDS)
    saved_again = save_phase1a_grasp_visualization(
        html_result,
        dummy_spec.scene_id,
        dummy_spec.task_id,
        data=dummy_scene,
    )
    saved_manifest = json.loads(saved_again.manifest_path.read_text(encoding="utf-8"))
    assert saved_manifest["visualization_schema_version"] == PHASE1B_VIZ_SCHEMA_VERSION
    assert len(saved_manifest["entries"]) == 1
    manifest_entry = saved_manifest["entries"][0]
    assert manifest_entry["status"] == "ok"
    assert manifest_entry["default_interface"] == "multi_pixel"
    assert manifest_entry["filter_summary"]["included_count"] == 12
    assert any(
        "stale_cache_key" in item["reasons"]
        for item in manifest_entry["excluded_candidates"]
    )

    diagnostic_html_result = Phase1ARunResult(
        Path(html_temp),
        "r1_calibrated_jet",
        CallBudget(0),
        0,
        [partial_call_row],
        [partial_record],
        [],
        [],
    )
    diagnostic_saved = save_phase1a_grasp_visualization(
        diagnostic_html_result,
        dummy_spec.scene_id,
        dummy_spec.task_id,
        include_partial=True,
        data=dummy_scene,
    )
    assert diagnostic_saved.status == "ok"
    assert diagnostic_saved.html_path is not None
    assert diagnostic_saved.html_path.name == (
        "fixture_scene__fixture_task__d3__diagnostic_partial.html"
    )
    assert diagnostic_saved.html_path != saved.html_path
    assert saved.html_path.exists() and diagnostic_saved.html_path.exists()
    diagnostic_manifest = json.loads(
        diagnostic_saved.manifest_path.read_text(encoding="utf-8")
    )
    assert len(diagnostic_manifest["entries"]) == 2
    assert {
        entry["include_partial_recovery"]
        for entry in diagnostic_manifest["entries"]
    } == {False, True}

    no_pose_record = json.loads(json.dumps(fallback_record, allow_nan=False))
    no_pose_record["sensor_corrected_pose_4x4"] = None
    no_pose_record["validation_flags"]["corrected_pose_valid"] = False
    no_pose_result = Phase1ARunResult(
        Path(html_temp) / "no-pose",
        "r1_calibrated_jet",
        CallBudget(0),
        0,
        [fallback_call],
        [no_pose_record],
        [],
        [],
    )
    stale_formal_html = (
        no_pose_result.run_dir
        / "visualizations"
        / "fixture_scene__fixture_task__d3.html"
    )
    stale_formal_html.parent.mkdir(parents=True, exist_ok=True)
    stale_formal_html.write_text("stale visualization", encoding="utf-8")
    assert stale_formal_html.exists()
    failed_visualization = save_phase1a_grasp_visualization(
        no_pose_result,
        dummy_spec.scene_id,
        dummy_spec.task_id,
        data=dummy_scene,
    )
    assert failed_visualization.status == "no_valid_corrected_pose"
    assert failed_visualization.html_path is None
    assert failed_visualization.manifest_path.exists()
    assert not stale_formal_html.exists()

print(
    "Phase 1B synthetic visualization checks passed: deterministic dual-layer "
    "clouds, corrected poses, menus, provenance filtering, and self-contained HTML."
)


## Optional batch execution

Nothing below runs unless `PHASE1A_EXECUTE_BATCH=1`. Cache-only execution uses `PHASE1A_EXECUTE_BATCH=1` with online mode unset. A real smoke run additionally requires `PHASE1A_RUN_ONLINE=1`, an API key in `GEMINI_API_KEY` or `GOOGLE_API_KEY`, and `PHASE1A_MAX_NEW_CALLS=10`. Set `PHASE1A_RENDER_3D=1` to write the self-contained HTML and show the first successful interactive figure in the executed notebook. Pilot and full use caps 40 and 364. Re-running the same profile/repeats reuses the global content cache; set `PHASE1A_RUN_ID` to resume the same artifact directory. Outputs are written beneath `output_vg/vlm_6dof_phase1a/`, and no API key is serialized.

In [ ]:
EXECUTE_BATCH = os.environ.get("PHASE1A_EXECUTE_BATCH", "0") == "1"
PHASE1A_VISUALIZATIONS: list[Phase1AVisualizationResult] = []
if EXECUTE_BATCH:
    PHASE1A_RESULT = run_phase1a(
        profile=PROFILE,
        online=RUN_ONLINE,
        max_new_calls=MAX_NEW_CALLS,
        run_id=RUN_ID_OVERRIDE,
        d1_for_d3=D1_FOR_D3,
    )
    print(f"Phase 1A artifacts: {PHASE1A_RESULT.run_dir}")
    print(f"Selected D1 for D3: {PHASE1A_RESULT.selected_d1_for_d3}")
    print(
        "New HTTP transport attempts: "
        f"{PHASE1A_RESULT.budget.new_http_calls - PHASE1A_RESULT.attempts_before_run}; "
        "cumulative: "
        f"{PHASE1A_RESULT.budget.new_http_calls}/"
        f"{PHASE1A_RESULT.budget.max_new_http_calls}"
    )
    if RENDER_3D:
        from IPython.display import HTML, display

        for visualization_spec in profile_specs(PROFILE):
            visualization_data = load_scene(visualization_spec)
            for visualization_repeat in range(int(PROFILE_CONFIGS[PROFILE]["repeats"])):
                visualization_result = save_phase1a_grasp_visualization(
                    PHASE1A_RESULT,
                    visualization_spec.scene_id,
                    visualization_spec.task_id,
                    repeat_index=visualization_repeat,
                    include_partial=False,
                    data=visualization_data,
                )
                PHASE1A_VISUALIZATIONS.append(visualization_result)
                print(
                    f"3D visualization {visualization_spec.scene_id}/"
                    f"{visualization_spec.task_id}/repeat-{visualization_repeat}: "
                    f"{visualization_result.status}; "
                    f"{visualization_result.html_path}"
                )
        first_visualization = next(
            (item for item in PHASE1A_VISUALIZATIONS if item.figure is not None),
            None,
        )
        if first_visualization is not None:
            display(
                HTML(
                    first_visualization.figure.to_html(
                        include_plotlyjs=True,
                        full_html=False,
                        config={
                            "responsive": True,
                            "scrollZoom": True,
                            "displaylogo": False,
                        },
                    )
                )
            )
else:
    PHASE1A_RESULT = None
    print(
        "Batch execution disabled (default). Offline verification completed with zero real network calls. "
        "Set PHASE1A_EXECUTE_BATCH=1 explicitly to run cache-only or online profiles."
    )
